In [ ]:


import os
import glob
import shutil
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# Install h5py for your specific dataset
!pip install -q h5py
import h5py

# Hardware Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using Device: {device}")
if device.type == 'cpu':
    print(" WARNING: You are on CPU. Go to Runtime > Change runtime type > T4 GPU!")

# -----------------------------------------------------------------------------
# 1. DATA PIPELINE (H5 VERSION)
# -----------------------------------------------------------------------------
def setup_h5_data():
    drive.mount('/content/drive')

    # PATHS
    DRIVE_FOLDER = "/content/drive/My Drive/Thesis_Data"
    ZIP_XRAY = f"{DRIVE_FOLDER}/nih_xray.zip"
    ZIP_MRI  = f"{DRIVE_FOLDER}/acdc.zip"
    LOCAL_DATA = "/content/data"
    TEMP_XRAY = "/content/temp_xray_raw"
    TEMP_MRI  = "/content/temp_mri_raw"

    # Check if data is already ready (from a previous run today)
    if os.path.exists(f"{LOCAL_DATA}/train/xray_sick"):
        print(" Data already processed in local runtime! Skipping setup.")
        return

    print("\n Starting Processing Pipeline...")

    # 1. UNZIP RAW FILES
    print(f"Step 1: Unzipping X-Rays (4.3 GB)... Please wait...")
    if not os.path.exists(TEMP_XRAY):
        shutil.unpack_archive(ZIP_XRAY, TEMP_XRAY)

    print("Step 2: Unzipping MRI Data...")
    if not os.path.exists(TEMP_MRI):
        shutil.unpack_archive(ZIP_MRI, TEMP_MRI)

    # Create Destination Folders
    for s in ['train', 'test']:
        for l in ['healthy', 'sick']:
            os.makedirs(f"{LOCAL_DATA}/{s}/xray_{l}", exist_ok=True)
            os.makedirs(f"{LOCAL_DATA}/{s}/mri_{l}", exist_ok=True)

    # 2. PROCESS X-RAY
    print("Step 3: Organizing X-Rays...")
    csv_files = glob.glob(f"{TEMP_XRAY}/**/*.csv", recursive=True)

    if len(csv_files) > 0:
        print(f"   -> Found CSV: {os.path.basename(csv_files[0])}")
        df = pd.read_csv(csv_files[0])

        if 'Finding Labels' in df.columns:
            sick_df = df[df['Finding Labels'].str.contains('Cardiomegaly')]
            healthy_df = df[df['Finding Labels'] == 'No Finding']
        else:
            # Fallback
            print("    CSV format unknown. Using random sampling.")
            sick_df = df.sample(frac=0.5)
            healthy_df = df.drop(sick_df.index)

        # Use 400 pairs (Safety limit for Colab RAM)
        n_samples = min(len(sick_df), len(healthy_df), 400)
        sick_df = sick_df.sample(n=n_samples, random_state=42)
        healthy_df = healthy_df.sample(n=n_samples, random_state=42)

        print(f"   -> Selected {n_samples} Sick and {n_samples} Healthy cases.")

        all_pngs = glob.glob(f"{TEMP_XRAY}/**/*.png", recursive=True)
        file_map = {os.path.basename(f): f for f in all_pngs}

        def move_batch(dataframe, subset, label):
            count = 0
            for img_name in dataframe['Image Index']:
                if img_name in file_map:
                    shutil.copy(file_map[img_name], f"{LOCAL_DATA}/{subset}/xray_{label}/{img_name}")
                    count += 1
            return count

        train_sick, test_sick = train_test_split(sick_df, test_size=0.2, random_state=42)
        train_norm, test_norm = train_test_split(healthy_df, test_size=0.2, random_state=42)

        move_batch(train_sick, 'train', 'sick')
        move_batch(test_sick, 'test', 'sick')
        move_batch(train_norm, 'train', 'healthy')
        move_batch(test_norm, 'test', 'healthy')

    # 3. PROCESS MRI (NEW H5 LOGIC)
    print("Step 4: converting .h5 MRI files to PNG...")

    # Find all .h5 files
    h5_files = glob.glob(f"{TEMP_MRI}/**/*.h5", recursive=True)
    print(f"   -> Found {len(h5_files)} MRI slices.")

    count = 0
    target_total = n_samples * 2

    for fpath in h5_files:
        try:
            # Open H5 file
            with h5py.File(fpath, 'r') as f:
                # H5 files are like dictionaries. We need to find the image key.
                # Common keys: 'image', 'img', 'slice', 'data'
                keys = list(f.keys())

                # Try to find the image data
                if 'image' in keys:
                    data = f['image'][:]
                elif 'slice' in keys:
                    data = f['slice'][:]
                else:
                    # Fallback: take the first key found
                    data = f[keys[0]][:]

                # Ensure it is 2D
                if len(data.shape) == 3:
                    data = data[:, :, 0] # Take first channel if 3D

                # Normalize to 0-255 for PNG
                data = ((data - np.min(data)) / (np.max(data) - np.min(data)) * 255).astype(np.uint8)

                # Distribute
                subset = 'train' if count < (target_total * 0.8) else 'test'
                label = 'sick' if count % 2 == 0 else 'healthy'

                Image.fromarray(data).save(f"{LOCAL_DATA}/{subset}/mri_{label}/mri_{count}.png")
                count += 1

                if count >= target_total: break
        except Exception as e:
            # print(f"Skipped bad file: {e}")
            continue

    print(f" Data Pipeline Complete! Processed {count} MRI images.")

# -----------------------------------------------------------------------------
# 2. DATASET & MODEL
# -----------------------------------------------------------------------------
class MultiModalDataset(Dataset):
    def __init__(self, root_dir, mode='train'):
        self.root_dir = os.path.join(root_dir, mode)
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        self.x_sick = sorted(glob.glob(f"{self.root_dir}/xray_sick/*.png"))
        self.x_norm = sorted(glob.glob(f"{self.root_dir}/xray_healthy/*.png"))
        self.m_sick = sorted(glob.glob(f"{self.root_dir}/mri_sick/*.png"))
        self.m_norm = sorted(glob.glob(f"{self.root_dir}/mri_healthy/*.png"))

        self.data = []
        min_norm = min(len(self.x_norm), len(self.m_norm))
        for i in range(min_norm): self.data.append((self.x_norm[i], self.m_norm[i], 0.0))
        min_sick = min(len(self.x_sick), len(self.m_sick))
        for i in range(min_sick): self.data.append((self.x_sick[i], self.m_sick[i], 1.0))

        print(f"[{mode.upper()}] Loaded {len(self.data)} Pairs.")

    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        x, m, y = self.data[idx]
        try:
            x_img = self.transform(Image.open(x).convert('RGB'))
            m_img = self.transform(Image.open(m).convert('RGB'))
            return x_img, m_img, torch.tensor(y).float()
        except: return torch.zeros(3,224,224), torch.zeros(3,224,224), torch.tensor(y).float()

class FusionModel(nn.Module):
    def __init__(self):
        super(FusionModel, self).__init__()
        self.xray = models.densenet121(weights='DEFAULT')
        self.xray_feat = self.xray.features
        self.xray_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.mri = models.resnet18(weights='DEFAULT')
        self.mri_feat = nn.Sequential(*list(self.mri.children())[:-1])

        self.head = nn.Sequential(
            nn.Linear(1024+512, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 1)
        )
    def forward(self, x, m):
        x = torch.flatten(self.xray_pool(nn.functional.relu(self.xray_feat(x))), 1)
        m = torch.flatten(self.mri_feat(m), 1)
        return self.head(torch.cat((x, m), dim=1))

# -----------------------------------------------------------------------------
# 3. MAIN EXECUTION
# -----------------------------------------------------------------------------
def main():
    setup_h5_data() # Run the NEW H5 pipeline

    train_ds = MultiModalDataset("/content/data", mode='train')
    test_ds = MultiModalDataset("/content/data", mode='test')

    if len(train_ds) == 0:
        print(" ERROR: No data found.")
        return

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

    model = FusionModel().to(device)
    opt = optim.Adam(model.parameters(), lr=1e-4)
    crit = nn.BCEWithLogitsLoss()

    print("\n Training Started...")
    for epoch in range(5): # 5 Epochs
        model.train()
        loss_total = 0
        for x, m, y in train_loader:
            x, m, y = x.to(device), m.to(device), y.to(device).unsqueeze(1)
            opt.zero_grad()
            loss = crit(model(x, m), y)
            loss.backward()
            opt.step()
            loss_total += loss.item()
        print(f"   Epoch {epoch+1} Loss: {loss_total/len(train_loader):.4f}")

    print("\n Evaluating...")
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, m, y in test_loader:
            x, m, y = x.to(device), m.to(device), y.to(device)
            out = model(x, m)
            pred = (torch.sigmoid(out) > 0.5).float()
            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    print(classification_report(y_true, y_pred, target_names=['Healthy', 'Sick']))

    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:

# =============================================================================
# Improvements in this version:
#   1. ACDC label fix   — patient ID range mapping (not count % 2)
#   2. Class imbalance  — pos_weight in loss + WeightedRandomSampler
#   3. Augmentation     — training only, with MRI-appropriate transforms
#   4. Scheduler        — CosineAnnealingWarmRestarts (more robust than plain cosine)
#   5. Two-phase tuning — freeze → unfreeze last blocks at lower LR
#   6. Val split        — in-memory random_split (no extra folder needed)
#   7. Threshold tuning — optimal threshold found on val set, not hardcoded 0.5
#   8. Grad-CAM         — visual explanation for both X-ray and MRI branches
#   9. predict()        — single function for new images with full explanation
# =============================================================================

import os, glob, shutil, re, textwrap
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, random_split
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, roc_auc_score, roc_curve)
from google.colab import drive

try:
    import h5py
except ImportError:
    import subprocess; subprocess.run(["pip","install","-q","h5py"]); import h5py

# ── Hardware ──────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Device: {device}")
if device.type == 'cpu':
    print("  Go to Runtime > Change runtime type > T4 GPU for acceptable speed.")


# =============================================================================
# HELPER: ACDC patient ID → label
# 001-020 NOR  → healthy
# 021-040 MINF → sick   (Myocardial Infarction)
# 041-060 DCM  → sick   (Dilated Cardiomyopathy)
# 061-080 HCM  → sick   (Hypertrophic Cardiomyopathy)
# 081-100 RV   → sick   (Right Ventricular abnormality)
# 101+         → held-out, skip
# =============================================================================
def acdc_label(fpath):
    m = re.search(r'patient(\d+)', os.path.basename(fpath))
    if not m: return None, None
    pid = int(m.group(1))
    if   1  <= pid <= 20:  return 'healthy', 'NOR'
    elif 21 <= pid <= 40:  return 'sick',    'MINF'
    elif 41 <= pid <= 60:  return 'sick',    'DCM'
    elif 61 <= pid <= 80:  return 'sick',    'HCM'
    elif 81 <= pid <= 100: return 'sick',    'RV'
    return None, None


# =============================================================================
# 1.  DATA PIPELINE
# =============================================================================
def setup_data(force_reprocess=False):
    drive.mount('/content/drive')

    DRIVE  = "/content/drive/My Drive/Thesis_Data"
    LOCAL  = "/content/data"
    TXRAY  = "/content/temp_xray_raw"
    TMRI   = "/content/temp_mri_raw"

    if os.path.exists(f"{LOCAL}/train/xray_sick") and not force_reprocess:
        print("Data already on disk. Skipping setup.")
        print("   (Pass force_reprocess=True to rebuild with corrected labels.)")
        return

    if force_reprocess and os.path.exists(LOCAL):
        print(" Removing old processed data for clean rebuild...")
        shutil.rmtree(LOCAL)

    print("\n Starting data pipeline...")
    if not os.path.exists(TXRAY): shutil.unpack_archive(f"{DRIVE}/nih_xray.zip", TXRAY)
    if not os.path.exists(TMRI):  shutil.unpack_archive(f"{DRIVE}/acdc.zip",     TMRI)

    for s in ['train','test']:
        for l in ['healthy','sick']:
            os.makedirs(f"{LOCAL}/{s}/xray_{l}", exist_ok=True)
            os.makedirs(f"{LOCAL}/{s}/mri_{l}",  exist_ok=True)

    # ── X-Ray ─────────────────────────────────────────────────────────────────
    print("Step 1: Organising X-Rays...")
    csvs = glob.glob(f"{TXRAY}/**/*.csv", recursive=True)
    n_samples = 400
    if csvs:
        df = pd.read_csv(csvs[0])
        if 'Finding Labels' in df.columns:
            sick_df    = df[df['Finding Labels'].str.contains('Cardiomegaly', na=False)]
            healthy_df = df[df['Finding Labels'] == 'No Finding']
        else:
            sick_df    = df.sample(frac=0.5, random_state=42)
            healthy_df = df.drop(sick_df.index)

        n_samples  = min(len(sick_df), len(healthy_df), 400)
        sick_df    = sick_df.sample(n=n_samples, random_state=42)
        healthy_df = healthy_df.sample(n=n_samples, random_state=42)
        print(f"   -> {n_samples} sick + {n_samples} healthy X-Rays selected.")

        pngs     = glob.glob(f"{TXRAY}/**/*.png", recursive=True)
        file_map = {os.path.basename(f): f for f in pngs}

        def move(dframe, subset, label):
            for img in dframe['Image Index']:
                if img in file_map:
                    shutil.copy(file_map[img], f"{LOCAL}/{subset}/xray_{label}/{img}")

        tr_s, te_s = train_test_split(sick_df,    test_size=0.2, random_state=42)
        tr_n, te_n = train_test_split(healthy_df, test_size=0.2, random_state=42)
        move(tr_s,'train','sick');    move(te_s,'test','sick')
        move(tr_n,'train','healthy'); move(te_n,'test','healthy')

    # ── MRI ───────────────────────────────────────────────────────────────────
    # FIX: ACDC has 20 healthy vs 80 sick patients → heavy imbalance.
    # We oversample healthy slices to balance the corpus.
    print("Step 2: Converting ACDC .h5 slices to PNG with correct labels...")

    slice_dir = f"{TMRI}/ACDC_preprocessed/ACDC_training_slices"
    vol_dir   = f"{TMRI}/ACDC_preprocessed/ACDC_training_volumes"
    src       = slice_dir if os.path.exists(slice_dir) else vol_dir
    h5_files  = sorted(glob.glob(f"{src}/**/*.h5", recursive=True))
    print(f"   -> Found {len(h5_files)} .h5 files in {os.path.basename(src)}")

    # Separate files by label first so we can oversample healthy
    sick_files    = [f for f in h5_files if acdc_label(f)[0] == 'sick']
    healthy_files = [f for f in h5_files if acdc_label(f)[0] == 'healthy']
    skipped       = len(h5_files) - len(sick_files) - len(healthy_files)

    # Oversample healthy to match sick count (repeat the 20 patients' slices)
    if len(healthy_files) < len(sick_files):
        factor = (len(sick_files) // len(healthy_files)) + 1
        healthy_files = (healthy_files * factor)[:len(sick_files)]

    target = min(len(sick_files), len(healthy_files), n_samples)
    sick_files    = sick_files[:target]
    healthy_files = healthy_files[:target]
    print(f"   -> Using {target} sick + {target} healthy MRI slices (healthy oversampled).")

    def save_h5_as_png(fpath, out_path):
        with h5py.File(fpath, 'r') as f:
            data = f['image'][:]
        if data.ndim == 3:
            data = data[data.shape[0] // 2]   # middle slice of volume
        rng = np.max(data) - np.min(data)
        if rng == 0: return False
        data = ((data - np.min(data)) / rng * 255).astype(np.uint8)
        Image.fromarray(data).save(out_path)
        return True

    counts = {'train': {'sick':0,'healthy':0}, 'test': {'sick':0,'healthy':0}}
    for label, files in [('sick', sick_files), ('healthy', healthy_files)]:
        for i, fpath in enumerate(files):
            subset = 'train' if i < int(target * 0.8) else 'test'
            _, diag = acdc_label(fpath)
            out = f"{LOCAL}/{subset}/mri_{label}/mri_{diag}_{i}.png"
            try:
                if save_h5_as_png(fpath, out):
                    counts[subset][label] += 1
            except Exception:
                continue

    total = sum(v for c in counts.values() for v in c.values())
    print(f" Pipeline done — {total} MRI images saved  ({skipped} held-out skipped)")
    for s in ['train','test']:
        print(f"   {s}: sick={counts[s]['sick']}, healthy={counts[s]['healthy']}")


# =============================================================================
# 2.  DATASET
# =============================================================================
class MultiModalDataset(Dataset):
    def __init__(self, root_dir, mode='train'):
        self.root = os.path.join(root_dir, mode)
        self.mode = mode

        # Augmentation only for training; MRI-appropriate (no colour jitter)
        if mode == 'train':
            self.xray_tf = transforms.Compose([
                transforms.Resize((256, 256)),
                transforms.RandomCrop(224),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
            ])
            self.mri_tf = transforms.Compose([
                transforms.Resize((256, 256)),
                transforms.RandomCrop(224),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
                transforms.RandomAffine(degrees=0, translate=(0.05,0.05)),
                transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
            ])
        else:
            tf = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
            ])
            self.xray_tf = self.mri_tf = tf

        x_sick = sorted(glob.glob(f"{self.root}/xray_sick/*.png"))
        x_norm = sorted(glob.glob(f"{self.root}/xray_healthy/*.png"))
        m_sick = sorted(glob.glob(f"{self.root}/mri_sick/*.png"))
        m_norm = sorted(glob.glob(f"{self.root}/mri_healthy/*.png"))

        self.data   = []
        self.labels = []
        n_norm = min(len(x_norm), len(m_norm))
        n_sick = min(len(x_sick), len(m_sick))
        for i in range(n_norm):
            self.data.append((x_norm[i], m_norm[i], 0.0)); self.labels.append(0)
        for i in range(n_sick):
            self.data.append((x_sick[i], m_sick[i], 1.0)); self.labels.append(1)

        print(f"[{mode.upper():5s}] {n_sick} sick + {n_norm} healthy = {len(self.data)} pairs")

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        x_path, m_path, label = self.data[idx]
        try:
            xi = self.xray_tf(Image.open(x_path).convert('RGB'))
            mi = self.mri_tf( Image.open(m_path).convert('RGB'))
            return xi, mi, torch.tensor(label).float()
        except Exception:
            return (torch.zeros(3,224,224), torch.zeros(3,224,224),
                    torch.tensor(label).float())

    def get_sampler(self):
        """WeightedRandomSampler for balanced batches regardless of imbalance."""
        class_counts = [self.labels.count(0), self.labels.count(1)]
        weights      = [1.0 / class_counts[l] for l in self.labels]
        return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


# =============================================================================
# 3.  MODEL
# =============================================================================
class FusionModel(nn.Module):
    def __init__(self):
        super().__init__()

        # X-Ray branch: DenseNet121 → 1024-d
        dn = models.densenet121(weights='DEFAULT')
        self.xray_feat = dn.features
        self.xray_pool = nn.AdaptiveAvgPool2d((1, 1))

        # MRI branch: ResNet18 → 512-d
        rn = models.resnet18(weights='DEFAULT')
        self.mri_feat = nn.Sequential(*list(rn.children())[:-1])

        self._freeze_backbones()

        # Fusion head
        self.head = nn.Sequential(
            nn.Linear(1024 + 512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def _freeze_backbones(self):
        for p in self.xray_feat.parameters(): p.requires_grad = False
        for p in self.mri_feat.parameters():  p.requires_grad = False

    def unfreeze_last_blocks(self):
        for name, p in self.xray_feat.named_parameters():
            if 'denseblock4' in name or 'norm5' in name:
                p.requires_grad = True
        for name, p in self.mri_feat.named_parameters():
            if name.startswith('7') or name.startswith('8'):
                p.requires_grad = True
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"    Phase 2 — {n:,} trainable parameters.")

    def forward(self, x, m):
        xf = torch.flatten(self.xray_pool(F.relu(self.xray_feat(x))), 1)
        mf = torch.flatten(self.mri_feat(m), 1)
        return self.head(torch.cat((xf, mf), 1))


# =============================================================================
# 4.  GRAD-CAM EXPLAINABILITY
# =============================================================================
class GradCAM:
    """
    Computes a Grad-CAM heatmap for a single branch (X-ray or MRI).
    Points to the image regions that drove the prediction.
    """
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        self._register_hooks()

    def _register_hooks(self):
        def fwd_hook(_, __, output): self.activations = output.detach()
        def bwd_hook(_, __, grad_out): self.gradients = grad_out[0].detach()
        self.target_layer.register_forward_hook(fwd_hook)
        self.target_layer.register_full_backward_hook(bwd_hook)

    def generate(self, x_img, m_img, class_idx=1):
        """
        Returns (xray_heatmap, mri_heatmap) as numpy arrays in [0,1].
        class_idx: 1=Sick, 0=Healthy
        """
        self.model.eval()
        x = x_img.unsqueeze(0).to(device)
        m = m_img.unsqueeze(0).to(device)

        # --- X-ray Grad-CAM ---
        x.requires_grad_(True)
        # Hook on last DenseNet conv layer (norm5 output = after denseblock4)
        xray_layer = self.model.xray_feat.norm5
        xray_acts, xray_grads = [], []

        h1 = xray_layer.register_forward_hook( lambda *a: xray_acts.append(a[2].detach()))
        h2 = xray_layer.register_full_backward_hook(lambda *a: xray_grads.append(a[2][0].detach()))

        out = self.model(x, m)
        self.model.zero_grad()
        score = out[0, 0] if class_idx == 1 else -out[0, 0]
        score.backward(retain_graph=True)
        h1.remove(); h2.remove()

        xray_cam = self._pool_cam(xray_acts[0], xray_grads[0])

        # --- MRI Grad-CAM ---
        # ResNet18 layer4 = mri_feat[7]
        mri_acts, mri_grads = [], []
        mri_layer = self.model.mri_feat[7]
        h3 = mri_layer.register_forward_hook( lambda *a: mri_acts.append(a[2].detach()))
        h4 = mri_layer.register_full_backward_hook(lambda *a: mri_grads.append(a[2][0].detach()))

        out = self.model(x, m)
        self.model.zero_grad()
        score = out[0, 0] if class_idx == 1 else -out[0, 0]
        score.backward()
        h3.remove(); h4.remove()

        mri_cam = self._pool_cam(mri_acts[0], mri_grads[0])
        return xray_cam, mri_cam

    @staticmethod
    def _pool_cam(activations, gradients):
        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam     = F.relu((weights * activations).sum(dim=1, keepdim=True))
        cam     = cam.squeeze().cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


def overlay_heatmap(img_tensor, heatmap, title, ax):
    """Blend a Grad-CAM heatmap onto the original image."""
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = img_tensor.permute(1, 2, 0).cpu().numpy()
    img  = np.clip(img * std + mean, 0, 1)

    h_resized = np.array(Image.fromarray(
        (heatmap * 255).astype(np.uint8)).resize((224, 224),
        Image.BILINEAR)) / 255.0
    coloured  = cm.jet(h_resized)[:, :, :3]
    blended   = 0.55 * img + 0.45 * coloured

    ax.imshow(blended)
    ax.set_title(title, fontsize=10)
    ax.axis('off')


# =============================================================================
# 5.  TRAINING HELPER
# =============================================================================
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss  = 0.0
    all_true, all_pred, all_prob = [], [], []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for x, m, y in loader:
            x, m  = x.to(device), m.to(device)
            y_dev = y.to(device).unsqueeze(1)
            out   = model(x, m)
            loss  = criterion(out, y_dev)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            prob = torch.sigmoid(out).squeeze().detach().cpu()
            pred = (prob > 0.5).float()
            total_loss  += loss.item()
            all_true.extend(y.numpy())
            all_pred.extend(pred.numpy() if prob.dim() > 0 else [pred.item()])
            all_prob.extend(prob.numpy() if prob.dim() > 0 else [prob.item()])

    avg_loss = total_loss / len(loader)
    f1       = f1_score(all_true, all_pred, average='macro', zero_division=0)
    return avg_loss, f1, all_true, all_pred, all_prob


def find_best_threshold(y_true, y_prob):
    """Sweep thresholds on val set, pick the one with highest macro-F1."""
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.25, 0.76, 0.01):
        pred = (np.array(y_prob) >= t).astype(float)
        f1   = f1_score(y_true, pred, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


# =============================================================================
# 6.  MAIN TRAINING LOOP
# =============================================================================
def main():
    # ── Pass force_reprocess=True if you need to rebuild with correct labels ──
    setup_data(force_reprocess=False)

    full_train = MultiModalDataset("/content/data", mode='train')
    test_ds    = MultiModalDataset("/content/data", mode='test')

    if len(full_train) == 0:
        print(" No training data found."); return

    # In-memory val split (85 / 15)
    val_size   = max(4, int(0.15 * len(full_train)))
    train_size = len(full_train) - val_size
    train_ds, val_ds = random_split(
        full_train, [train_size, val_size],
        generator=torch.Generator().manual_seed(42))
    print(f"\nSplit → Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

    # FIX: WeightedRandomSampler for balanced batches
    train_loader = DataLoader(train_ds, batch_size=16,
                              sampler=full_train.get_sampler(),
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False,
                              num_workers=2, pin_memory=True)

    model = FusionModel().to(device)

    # FIX: pos_weight compensates for any remaining imbalance in loss
    n_sick    = full_train.labels.count(1)
    n_healthy = full_train.labels.count(0)
    pos_w     = torch.tensor([n_healthy / max(n_sick, 1)], dtype=torch.float).to(device)
    print(f"pos_weight = {pos_w.item():.3f}  "
          f"(sick={n_sick}, healthy={n_healthy})")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    PHASE1  = 7
    PHASE2  = 13
    TOTAL   = PHASE1 + PHASE2
    PATIENCE = 6
    CKPT    = '/content/best_model.pt'

    optimizer = optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-4)
    # CosineAnnealingWarmRestarts restarts every T_0 epochs — more robust
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=PHASE1, T_mult=1, eta_min=1e-6)

    best_val_f1  = 0.0
    patience_ctr = 0
    history      = {'train_loss':[], 'val_loss':[], 'val_f1':[]}

    print(f"\n Phase 1: Head only ({PHASE1} epochs, backbones frozen)...")

    for epoch in range(1, TOTAL + 1):

        if epoch == PHASE1 + 1:
            print(f"\n🔓 Phase 2: Fine-tuning last blocks ({PHASE2} epochs)...")
            model.unfreeze_last_blocks()
            backbone_params = (
                [p for p in model.xray_feat.parameters() if p.requires_grad] +
                [p for p in model.mri_feat.parameters()  if p.requires_grad])
            optimizer.add_param_group({'params': backbone_params,
                                       'lr': 5e-6, 'weight_decay': 1e-4})
            scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
                optimizer, T_0=PHASE2, T_mult=1, eta_min=1e-8)

        tr_loss, tr_f1, *_ = run_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_f1, *_ = run_epoch(model, val_loader,   criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['val_f1'].append(va_f1)

        phase = 1 if epoch <= PHASE1 else 2
        print(f"   [P{phase}] Epoch {epoch:2d}/{TOTAL} | "
              f"Train {tr_loss:.4f}/{tr_f1:.3f} | "
              f"Val {va_loss:.4f}/{va_f1:.3f}")

        if va_f1 > best_val_f1:
            best_val_f1  = va_f1
            patience_ctr = 0
            torch.save(model.state_dict(), CKPT)
            print(f"            Saved (val F1={best_val_f1:.3f})")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n⏹  Early stopping at epoch {epoch}.")
                break

    # ── Threshold tuning on val set ──────────────────────────────────────────
    print(f"\n  Tuning decision threshold on validation set...")
    model.load_state_dict(torch.load(CKPT))
    _, _, val_true, _, val_prob = run_epoch(model, val_loader, criterion)
    best_threshold, thresh_f1  = find_best_threshold(val_true, val_prob)
    print(f"   Best threshold = {best_threshold:.2f}  (val macro-F1 = {thresh_f1:.3f})")

    # ── Final test evaluation ─────────────────────────────────────────────────
    print(f"\n Test evaluation (threshold={best_threshold:.2f})...")
    _, _, y_true, _, y_prob = run_epoch(model, test_loader, criterion)
    y_pred = (np.array(y_prob) >= best_threshold).astype(float)

    print("\n" + "="*60)
    print(classification_report(y_true, y_pred,
                                target_names=['Healthy','Sick']))
    try:
        auc = roc_auc_score(y_true, y_prob)
        print(f"ROC-AUC: {auc:.3f}")
    except Exception:
        pass

    # ── Plots ─────────────────────────────────────────────────────────────────
    ep = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(ep, history['train_loss'], label='Train')
    axes[0].plot(ep, history['val_loss'],   label='Val')
    axes[0].axvline(PHASE1 + 0.5, color='gray', ls='--', lw=1, label='Unfreeze')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(ep, history['val_f1'], color='#2a9d8f', lw=2)
    axes[1].axhline(best_val_f1, color='gray', ls=':',
                    label=f'Best={best_val_f1:.3f}')
    axes[1].set_title('Val macro-F1'); axes[1].set_xlabel('Epoch')
    axes[1].legend()

    cm_arr = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues', ax=axes[2],
                xticklabels=['Healthy','Sick'],
                yticklabels=['Healthy','Sick'])
    axes[2].set_title('Confusion matrix (test)')
    axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')

    plt.suptitle('Multimodal Cardiac Fusion Model — Results', fontsize=13)
    plt.tight_layout()
    plt.savefig('/content/results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(" Plot saved to /content/results.png")

    # Save threshold alongside checkpoint so predict() can use it
    torch.save({'state_dict': model.state_dict(),
                'threshold':  best_threshold}, '/content/best_model_full.pt')
    print(" Full checkpoint saved to /content/best_model_full.pt")


# =============================================================================
# 7.  INFERENCE + GRAD-CAM EXPLANATION
#     Use this after training to analyse a new pair of images.
# =============================================================================
# Inference transforms (no augmentation)
_infer_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

_DIAGNOSES = {
    'NOR':  'Normal cardiac structure',
    'MINF': 'Myocardial Infarction (scarring after heart attack)',
    'DCM':  'Dilated Cardiomyopathy (enlarged, weakened heart)',
    'HCM':  'Hypertrophic Cardiomyopathy (thickened heart muscle)',
    'RV':   'Right Ventricular abnormality',
}


def predict_and_explain(xray_path, mri_path,
                        ckpt_path='/content/best_model_full.pt'):

    # ── Load checkpoint ──────────────────────────────────────────────────────
    ckpt = torch.load(ckpt_path, map_location=device)
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        threshold = ckpt.get('threshold', 0.5)
        state     = ckpt['state_dict']
    else:
        threshold = 0.5
        state     = ckpt

    model = FusionModel().to(device)
    model.load_state_dict(state)
    model.eval()

    # ── Load images ───────────────────────────────────────────────────────────
    if mri_path.endswith('.h5'):
        with h5py.File(mri_path, 'r') as f:
            arr = f['image'][:]
        if arr.ndim == 3: arr = arr[arr.shape[0] // 2]
        rng = arr.max() - arr.min()
        arr = ((arr - arr.min()) / (rng + 1e-8) * 255).astype(np.uint8)
        mri_img = Image.fromarray(arr).convert('RGB')
    else:
        mri_img = Image.open(mri_path).convert('RGB')

    xray_img = Image.open(xray_path).convert('RGB')

    x_tensor = _infer_tf(xray_img)
    m_tensor = _infer_tf(mri_img)

    # ── Prediction ────────────────────────────────────────────────────────────
    with torch.no_grad():
        logit      = model(x_tensor.unsqueeze(0).to(device),
                           m_tensor.unsqueeze(0).to(device))
        confidence = torch.sigmoid(logit).item()

    prediction = 'Sick' if confidence >= threshold else 'Healthy'
    pred_idx   = 1 if prediction == 'Sick' else 0

    # ── Grad-CAM ─────────────────────────────────────────────────────────────
    gcam         = GradCAM(model, model.xray_feat.norm5)   # dummy target layer
    xray_cam, mri_cam = gcam.generate(x_tensor, m_tensor, class_idx=pred_idx)

    # ── Identify top activated region (rough quadrant) ────────────────────────
    def dominant_region(heatmap):
        h, w = heatmap.shape
        quads = {
            'upper-left':  heatmap[:h//2, :w//2].mean(),
            'upper-right': heatmap[:h//2, w//2:].mean(),
            'lower-left':  heatmap[h//2:, :w//2].mean(),
            'lower-right': heatmap[h//2:, w//2:].mean(),
        }
        return max(quads, key=quads.get)

    xray_region = dominant_region(xray_cam)
    mri_region  = dominant_region(mri_cam)

    # ── Plain-English reasoning ───────────────────────────────────────────────
    conf_pct   = confidence * 100
    low_conf   = conf_pct < 60 or conf_pct > 40

    if prediction == 'Sick':
        reasoning = textwrap.dedent(f"""
        PREDICTION: SICK  (confidence {conf_pct:.1f}%)
        ─────────────────────────────────────────────
        The model flagged this patient as likely having a cardiac abnormality.

        X-Ray branch:
          • The {xray_region} area of the chest X-Ray contributed most to this
            decision. In cardiomegaly screening, the model often focuses on
            the cardiac silhouette and cardiothoracic ratio.

        MRI branch:
          • The {mri_region} region of the cardiac MRI showed the strongest
            activation. This may correspond to wall-motion abnormalities,
            structural remodelling, or myocardial tissue changes.

        Fusion decision:
          • Both modalities were scored and concatenated before the final
            classifier. A confidence above {threshold:.2f} is classified as Sick.
          • Current confidence: {conf_pct:.1f}% → classified as SICK.

          This is a research model, not a clinical tool.
            All findings must be reviewed by a qualified cardiologist.
        """).strip()
    else:
        reasoning = textwrap.dedent(f"""
        PREDICTION: HEALTHY  (confidence of being sick: {conf_pct:.1f}%)
        ─────────────────────────────────────────────────────────────────
        The model found no strong evidence of a cardiac abnormality.

        X-Ray branch:
          • Attention was distributed across the {xray_region} region.
            Normal cardiac silhouette and clear lung fields reduce the
            probability score.

        MRI branch:
          • MRI activations are most prominent in the {mri_region} region.
            No focal wall-motion or structural anomaly pattern was detected.

        Fusion decision:
          • Confidence of disease: {conf_pct:.1f}% < threshold {threshold:.2f}
            → classified as HEALTHY.

           This is a research model, not a clinical tool.
            Always confirm with clinical assessment.
        """).strip()

    # ── Visualisation ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(
        f"Prediction: {prediction}  |  Confidence: {conf_pct:.1f}%  "
        f"(threshold={threshold:.2f})",
        fontsize=14,
        color='crimson' if prediction == 'Sick' else 'darkgreen',
        fontweight='bold'
    )

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    def to_display(t):
        img = t.permute(1,2,0).cpu().numpy()
        return np.clip(img * std + mean, 0, 1)

    # Row 0: original images
    axes[0,0].imshow(to_display(x_tensor), cmap='gray'); axes[0,0].set_title('X-Ray (input)')
    axes[0,0].axis('off')
    axes[0,1].imshow(to_display(m_tensor), cmap='gray'); axes[0,1].set_title('MRI (input)')
    axes[0,1].axis('off')

    # Confidence bar
    axes[0,2].barh(['Healthy','Sick'],
                   [1 - confidence, confidence],
                   color=['#2ecc71','#e74c3c'])
    axes[0,2].axvline(threshold, color='navy', ls='--', lw=1.5,
                      label=f'threshold={threshold:.2f}')
    axes[0,2].set_xlim(0, 1)
    axes[0,2].set_title('Prediction confidence')
    axes[0,2].legend()

    # Row 1: Grad-CAM overlays
    overlay_heatmap(x_tensor, xray_cam,
                    f'X-Ray Grad-CAM\n(focus: {xray_region})', axes[1,0])
    overlay_heatmap(m_tensor, mri_cam,
                    f'MRI Grad-CAM\n(focus: {mri_region})',    axes[1,1])

    # Reasoning text panel
    axes[1,2].axis('off')
    axes[1,2].text(0.02, 0.98, reasoning,
                   transform=axes[1,2].transAxes,
                   fontsize=8, verticalalignment='top',
                   fontfamily='monospace',
                   bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    out_fig = '/content/explanation.png'
    plt.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n Explanation figure saved to {out_fig}")
    print("\n" + reasoning)

    return {
        'prediction':  prediction,
        'confidence':  round(conf_pct, 2),
        'threshold':   threshold,
        'xray_region': xray_region,
        'mri_region':  mri_region,
        'explanation': reasoning,
    }


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    # ── Train ─────────────────────────────────────────────────────────────────
    main()



In [ ]:

import shutil
shutil.rmtree("/content/data", ignore_errors=True)
print("Done. Run main() now.")

In [ ]:


import os
import glob
import shutil
import re
import textwrap
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import (Dataset, DataLoader,
                               WeightedRandomSampler, random_split)
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, roc_auc_score)
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
from google.colab import drive

try:
    import h5py
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "h5py"])
    import h5py

# ── Hardware ──────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == 'cpu':
    print(" WARNING: Go to Runtime > Change runtime type > T4 GPU!")



def acdc_label(fpath):
    m = re.search(r'patient(\d+)', os.path.basename(fpath))
    if not m:
        return None, None
    pid = int(m.group(1))
    if   1  <= pid <= 20:  return 'healthy', 'NOR'
    elif 21 <= pid <= 40:  return 'sick',    'MINF'
    elif 41 <= pid <= 60:  return 'sick',    'DCM'
    elif 61 <= pid <= 80:  return 'sick',    'HCM'
    elif 81 <= pid <= 100: return 'sick',    'RV'
    return None, None   # 101+ = held-out, skip


# =============================================================================
# 1. DATA PIPELINE
# =============================================================================
def setup_data(force_reprocess=False):
    drive.mount('/content/drive')

    DRIVE  = "/content/drive/My Drive/Thesis_Data"
    LOCAL  = "/content/data"
    TXRAY  = "/content/temp_xray_raw"
    TMRI   = "/content/temp_mri_raw"

    sentinel = f"{LOCAL}/train/xray_sick"

    if os.path.exists(sentinel) and not force_reprocess:
        print(" Data already on disk. Skipping setup.")
        print("   Tip: call setup_data(force_reprocess=True) to rebuild.")
        return

    if force_reprocess and os.path.exists(LOCAL):
        print("🗑  Removing old processed data for clean rebuild...")
        shutil.rmtree(LOCAL)

    print("\n  Starting data pipeline...")

    if not os.path.exists(TXRAY):
        print("Step 0a: Unzipping X-Rays (4.3 GB)...")
        shutil.unpack_archive(f"{DRIVE}/nih_xray.zip", TXRAY)
    if not os.path.exists(TMRI):
        print("Step 0b: Unzipping MRI data...")
        shutil.unpack_archive(f"{DRIVE}/acdc.zip", TMRI)

    for s in ['train', 'test']:
        for l in ['healthy', 'sick']:
            os.makedirs(f"{LOCAL}/{s}/xray_{l}", exist_ok=True)
            os.makedirs(f"{LOCAL}/{s}/mri_{l}",  exist_ok=True)

    # ── X-Ray (NIH ChestX-ray14) ─────────────────────────────────────────────
    print("Step 1: Organising X-Rays...")
    csvs = glob.glob(f"{TXRAY}/**/*.csv", recursive=True)
    n_samples = 400

    if csvs:
        print(f"   -> CSV: {os.path.basename(csvs[0])}")
        df = pd.read_csv(csvs[0])

        if 'Finding Labels' in df.columns:
            sick_df    = df[df['Finding Labels'].str.contains('Cardiomegaly', na=False)]
            healthy_df = df[df['Finding Labels'] == 'No Finding']
        else:
            print("     Unknown CSV format — random split.")
            sick_df    = df.sample(frac=0.5, random_state=42)
            healthy_df = df.drop(sick_df.index)

        n_samples  = min(len(sick_df), len(healthy_df), 400)
        sick_df    = sick_df.sample(n=n_samples, random_state=42)
        healthy_df = healthy_df.sample(n=n_samples, random_state=42)
        print(f"   -> {n_samples} sick + {n_samples} healthy X-Rays.")

        pngs     = glob.glob(f"{TXRAY}/**/*.png", recursive=True)
        file_map = {os.path.basename(f): f for f in pngs}

        def move_xray(dframe, subset, label):
            for img in dframe['Image Index']:
                if img in file_map:
                    shutil.copy(file_map[img],
                                f"{LOCAL}/{subset}/xray_{label}/{img}")

        tr_s, te_s = train_test_split(sick_df,    test_size=0.2, random_state=42)
        tr_n, te_n = train_test_split(healthy_df, test_size=0.2, random_state=42)
        move_xray(tr_s, 'train', 'sick');    move_xray(te_s, 'test', 'sick')
        move_xray(tr_n, 'train', 'healthy'); move_xray(te_n, 'test', 'healthy')

    # ── MRI (ACDC) ────────────────────────────────────────────────────────────

    print("Step 2: Converting ACDC .h5 slices to PNG with correct labels...")

    slice_dir = f"{TMRI}/ACDC_preprocessed/ACDC_training_slices"
    vol_dir   = f"{TMRI}/ACDC_preprocessed/ACDC_training_volumes"
    src       = slice_dir if os.path.exists(slice_dir) else vol_dir
    h5_files  = sorted(glob.glob(f"{src}/**/*.h5", recursive=True))
    print(f"   -> Found {len(h5_files)} .h5 files in {os.path.basename(src)}")

    sick_files    = [f for f in h5_files if acdc_label(f)[0] == 'sick']
    healthy_files = [f for f in h5_files if acdc_label(f)[0] == 'healthy']
    skipped       = len(h5_files) - len(sick_files) - len(healthy_files)

    # Oversample healthy to match sick count
    if len(healthy_files) < len(sick_files):
        factor        = (len(sick_files) // max(len(healthy_files), 1)) + 1
        healthy_files = (healthy_files * factor)[:len(sick_files)]

    target        = min(len(sick_files), len(healthy_files), n_samples)
    sick_files    = sick_files[:target]
    healthy_files = healthy_files[:target]
    print(f"   -> Using {target} sick + {target} healthy MRI slices "
          f"(healthy oversampled to balance 4:1 ACDC ratio).")

    def save_h5_png(fpath, out_path):
        with h5py.File(fpath, 'r') as f:
            data = f['image'][:]
        if data.ndim == 3:
            data = data[data.shape[0] // 2]   # middle slice
        rng = data.max() - data.min()
        if rng == 0:
            return False
        data = ((data - data.min()) / rng * 255).astype(np.uint8)
        Image.fromarray(data).save(out_path)
        return True

    counts = {'train': {'sick': 0, 'healthy': 0},
              'test':  {'sick': 0, 'healthy': 0}}

    for label, files in [('sick', sick_files), ('healthy', healthy_files)]:
        for i, fpath in enumerate(files):
            subset = 'train' if i < int(target * 0.8) else 'test'
            _, diag = acdc_label(fpath)
            out = f"{LOCAL}/{subset}/mri_{label}/mri_{diag}_{i}.png"
            try:
                if save_h5_png(fpath, out):
                    counts[subset][label] += 1
            except Exception:
                continue

    total = sum(v for c in counts.values() for v in c.values())
    print(f" Pipeline done — {total} MRI images  ({skipped} held-out skipped)")
    for s in ['train', 'test']:
        print(f"   {s}: sick={counts[s]['sick']}, healthy={counts[s]['healthy']}")


# =============================================================================
# 2. DATASET
# =============================================================================
class MultiModalDataset(Dataset):
    def __init__(self, root_dir, mode='train'):
        self.root = os.path.join(root_dir, mode)
        self.mode = mode

        if mode == 'train':
            # Separate augmentation pipelines — MRI has no colour jitter
            self.xray_tf = transforms.Compose([
                transforms.Resize((256, 256)),
                transforms.RandomCrop(224),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])
            ])
            self.mri_tf = transforms.Compose([
                transforms.Resize((256, 256)),
                transforms.RandomCrop(224),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
                transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])
            ])
        else:
            _tf = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])
            ])
            self.xray_tf = self.mri_tf = _tf

        x_sick = sorted(glob.glob(f"{self.root}/xray_sick/*.png"))
        x_norm = sorted(glob.glob(f"{self.root}/xray_healthy/*.png"))
        m_sick = sorted(glob.glob(f"{self.root}/mri_sick/*.png"))
        m_norm = sorted(glob.glob(f"{self.root}/mri_healthy/*.png"))

        self.data   = []
        self.labels = []   # kept separately for sampler weight calculation
        n_norm = min(len(x_norm), len(m_norm))
        n_sick = min(len(x_sick), len(m_sick))
        for i in range(n_norm):
            self.data.append((x_norm[i], m_norm[i], 0.0))
            self.labels.append(0)
        for i in range(n_sick):
            self.data.append((x_sick[i], m_sick[i], 1.0))
            self.labels.append(1)

        print(f"[{mode.upper():5s}] {n_sick} sick + {n_norm} healthy "
              f"= {len(self.data)} pairs")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x_path, m_path, label = self.data[idx]
        try:
            xi = self.xray_tf(Image.open(x_path).convert('RGB'))
            mi = self.mri_tf( Image.open(m_path).convert('RGB'))
            return xi, mi, torch.tensor(label).float()
        except Exception:
            return (torch.zeros(3, 224, 224),
                    torch.zeros(3, 224, 224),
                    torch.tensor(label).float())


def make_weighted_sampler(full_dataset, subset):

    subset_labels = [full_dataset.labels[i] for i in subset.indices]
    n0 = subset_labels.count(0)
    n1 = subset_labels.count(1)
    weights = [1.0 / max(n0, 1) if l == 0 else 1.0 / max(n1, 1)
               for l in subset_labels]
    return WeightedRandomSampler(weights,
                                 num_samples=len(weights),
                                 replacement=True)


# =============================================================================
# 3. MODEL
# =============================================================================
class FusionModel(nn.Module):
    def __init__(self):
        super().__init__()

        # X-Ray branch: DenseNet121 → 1024-d feature vector
        dn             = models.densenet121(weights='DEFAULT')
        self.xray_feat = dn.features
        self.xray_pool = nn.AdaptiveAvgPool2d((1, 1))

        # MRI branch: ResNet18 → 512-d feature vector
        rn            = models.resnet18(weights='DEFAULT')
        self.mri_feat = nn.Sequential(*list(rn.children())[:-1])

        # Phase 1: freeze both backbones
        self._set_backbone_grad(False)

        # Fusion head: 1536 → 512 → 128 → 1
        self.head = nn.Sequential(
            nn.Linear(1024 + 512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def _set_backbone_grad(self, req):
        for p in self.xray_feat.parameters(): p.requires_grad = req
        for p in self.mri_feat.parameters():  p.requires_grad = req

    def unfreeze_last_blocks(self):
        """Phase 2: unfreeze the deepest convolutional blocks for fine-tuning."""
        for name, p in self.xray_feat.named_parameters():
            if 'denseblock4' in name or 'norm5' in name:
                p.requires_grad = True
        for name, p in self.mri_feat.named_parameters():
            if name.startswith('7') or name.startswith('8'):
                p.requires_grad = True
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"   Phase 2 active — {n:,} trainable parameters.")

    def forward(self, x, m):
        xf = torch.flatten(self.xray_pool(F.relu(self.xray_feat(x))), 1)
        mf = torch.flatten(self.mri_feat(m), 1)
        return self.head(torch.cat((xf, mf), dim=1))


# =============================================================================
# 4. GRAD-CAM
# =============================================================================
class GradCAM:

    def __init__(self, model):
        self.model = model

    def _get_cam(self, activations, gradients):
        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam     = F.relu((weights * activations).sum(dim=1, keepdim=True))
        cam     = cam.squeeze().cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

    def generate(self, x_tensor, m_tensor, class_idx=1):

        self.model.eval()
        x = x_tensor.unsqueeze(0).to(device).requires_grad_(True)
        m = m_tensor.unsqueeze(0).to(device).requires_grad_(True)

        # ── X-ray CAM (hook on DenseNet norm5) ────────────────────────────────
        xray_acts, xray_grads = [], []
        h1 = self.model.xray_feat.norm5.register_forward_hook(
            lambda _, __, o: xray_acts.append(o))
        h2 = self.model.xray_feat.norm5.register_full_backward_hook(
            lambda _, __, gi: xray_grads.append(gi[0]))

        out = self.model(x, m)
        self.model.zero_grad()
        (out[0, 0] if class_idx == 1 else -out[0, 0]).backward(retain_graph=True)
        h1.remove(); h2.remove()
        xray_cam = self._get_cam(xray_acts[0].detach(), xray_grads[0].detach())

        # ── MRI CAM (hook on ResNet18 layer4 = mri_feat[7]) ───────────────────
        mri_acts, mri_grads = [], []
        h3 = self.model.mri_feat[7].register_forward_hook(
            lambda _, __, o: mri_acts.append(o))
        h4 = self.model.mri_feat[7].register_full_backward_hook(
            lambda _, __, gi: mri_grads.append(gi[0]))

        out = self.model(x, m)
        self.model.zero_grad()
        (out[0, 0] if class_idx == 1 else -out[0, 0]).backward()
        h3.remove(); h4.remove()
        mri_cam = self._get_cam(mri_acts[0].detach(), mri_grads[0].detach())

        return xray_cam, mri_cam


def _to_display(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = tensor.permute(1, 2, 0).cpu().numpy()
    return np.clip(img * std + mean, 0, 1)


def overlay_heatmap(img_tensor, heatmap, title, ax):
    img      = _to_display(img_tensor)
    h_up     = np.array(Image.fromarray(
        (heatmap * 255).astype(np.uint8)).resize((224, 224),
        Image.BILINEAR)) / 255.0
    coloured = mpl_cm.jet(h_up)[:, :, :3]
    blended  = 0.55 * img + 0.45 * coloured
    ax.imshow(blended); ax.set_title(title, fontsize=9); ax.axis('off')


def dominant_region(heatmap):
    h, w = heatmap.shape
    quads = {
        'upper-left':  heatmap[:h//2, :w//2].mean(),
        'upper-right': heatmap[:h//2, w//2:].mean(),
        'lower-left':  heatmap[h//2:, :w//2].mean(),
        'lower-right': heatmap[h//2:, w//2:].mean(),
    }
    return max(quads, key=quads.get)


# =============================================================================
# 5. TRAINING HELPER
# =============================================================================
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_true, all_pred, all_prob = [], [], []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for x, m, y in loader:
            x, m  = x.to(device), m.to(device)
            y_dev = y.to(device).unsqueeze(1)
            out   = model(x, m)
            loss  = criterion(out, y_dev)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            prob = torch.sigmoid(out).squeeze().detach().cpu()
            pred = (prob > 0.5).float()
            all_true.extend(y.numpy())
            all_pred.extend(
                pred.numpy() if pred.dim() > 0 else [pred.item()])
            all_prob.extend(
                prob.numpy() if prob.dim() > 0 else [prob.item()])

    avg_loss = total_loss / len(loader)
    f1       = f1_score(all_true, all_pred, average='macro', zero_division=0)
    return avg_loss, f1, all_true, all_pred, all_prob


def find_best_threshold(y_true, y_prob):
    """Sweep thresholds 0.25–0.75 on val set; pick best macro-F1."""
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.25, 0.76, 0.01):
        pred = (np.array(y_prob) >= t).astype(float)
        f1   = f1_score(y_true, pred, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return round(best_t, 2), round(best_f1, 3)


# =============================================================================
# 6. MAIN
# =============================================================================
def main():
    # ── Data ─────────────────────────────────────────────────────────────────
    # Set force_reprocess=True once to rebuild with correct labels,
    # then set back to False for subsequent runs.
    setup_data(force_reprocess=False)

    full_train = MultiModalDataset("/content/data", mode='train')
    test_ds    = MultiModalDataset("/content/data", mode='test')

    if len(full_train) == 0:
        print(" No training data. Check paths or run setup_data(force_reprocess=True).")
        return

    # In-memory 85/15 train/val split
    val_size   = max(4, int(0.15 * len(full_train)))
    train_size = len(full_train) - val_size
    train_ds, val_ds = random_split(
        full_train, [train_size, val_size],
        generator=torch.Generator().manual_seed(42))
    print(f"\nSplit → Train: {len(train_ds)}, "
          f"Val: {len(val_ds)}, Test: {len(test_ds)}")


    train_loader = DataLoader(
        train_ds, batch_size=16,
        sampler=make_weighted_sampler(full_train, train_ds),
        num_workers=2, pin_memory=True)
    val_loader  = DataLoader(val_ds,  batch_size=16, shuffle=False,
                             num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False,
                             num_workers=2, pin_memory=True)

    # ── Model + loss ──────────────────────────────────────────────────────────
    model = FusionModel().to(device)

    n_sick    = full_train.labels.count(1)
    n_healthy = full_train.labels.count(0)
    pos_w     = torch.tensor([n_healthy / max(n_sick, 1)],
                              dtype=torch.float).to(device)
    print(f"pos_weight = {pos_w.item():.3f}  "
          f"(sick={n_sick}, healthy={n_healthy})")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    # ── Training schedule ─────────────────────────────────────────────────────
    PHASE1   = 7    # head-only epochs (backbones frozen)
    PHASE2   = 13   # fine-tuning epochs (last blocks unfrozen)
    TOTAL    = PHASE1 + PHASE2
    PATIENCE = 6
    CKPT     = '/content/best_model.pt'

    optimizer = optim.AdamW(model.head.parameters(),
                            lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=PHASE1, T_mult=1, eta_min=1e-6)

    best_val_f1  = 0.0
    patience_ctr = 0
    history      = {'train_loss': [], 'val_loss': [], 'val_f1': []}

    print(f"\n Phase 1: Head only ({PHASE1} epochs, backbones frozen)...")

    for epoch in range(1, TOTAL + 1):

        # Switch to Phase 2
        if epoch == PHASE1 + 1:
            print(f"\n Phase 2: Fine-tuning last blocks ({PHASE2} epochs)...")
            model.unfreeze_last_blocks()
            backbone_params = (
                [p for p in model.xray_feat.parameters() if p.requires_grad] +
                [p for p in model.mri_feat.parameters()  if p.requires_grad])
            optimizer.add_param_group(
                {'params': backbone_params, 'lr': 5e-6, 'weight_decay': 1e-4})
            scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
                optimizer, T_0=PHASE2, T_mult=1, eta_min=1e-8)

        tr_loss, tr_f1, *_ = run_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_f1, *_ = run_epoch(model, val_loader,   criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['val_f1'].append(va_f1)

        phase = 1 if epoch <= PHASE1 else 2
        print(f"   [P{phase}] Epoch {epoch:2d}/{TOTAL} | "
              f"Train loss={tr_loss:.4f} F1={tr_f1:.3f} | "
              f"Val loss={va_loss:.4f} F1={va_f1:.3f}")

        if va_f1 > best_val_f1:
            best_val_f1  = va_f1
            patience_ctr = 0
            torch.save(model.state_dict(), CKPT)
            print(f"             Saved best model (val F1={best_val_f1:.3f})")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n⏹  Early stopping at epoch {epoch}.")
                break

    # ── Threshold tuning ──────────────────────────────────────────────────────
    print("\n Tuning decision threshold on validation set...")
    model.load_state_dict(torch.load(CKPT))
    _, _, val_true, _, val_prob = run_epoch(model, val_loader, criterion)
    best_threshold, thresh_f1  = find_best_threshold(val_true, val_prob)
    print(f"   Best threshold = {best_threshold:.2f}  "
          f"(val macro-F1 = {thresh_f1:.3f})")

    # ── Final test evaluation ─────────────────────────────────────────────────
    print(f"\n Test evaluation (threshold = {best_threshold:.2f})...")
    _, _, y_true, _, y_prob = run_epoch(model, test_loader, criterion)
    y_pred = (np.array(y_prob) >= best_threshold).astype(float)

    print("\n" + "=" * 60)
    print(classification_report(y_true, y_pred,
                                target_names=['Healthy', 'Sick']))
    try:
        auc = roc_auc_score(y_true, y_prob)
        print(f"ROC-AUC : {auc:.3f}")
    except Exception:
        pass
    print(f"Threshold used: {best_threshold:.2f}")

    # ── Plots ─────────────────────────────────────────────────────────────────
    ep     = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(ep, history['train_loss'], label='Train')
    axes[0].plot(ep, history['val_loss'],   label='Val')
    axes[0].axvline(PHASE1 + 0.5, color='gray', ls='--', lw=1, label='Unfreeze')
    axes[0].set_title('Loss curves')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
    axes[0].legend()

    axes[1].plot(ep, history['val_f1'], color='#2a9d8f', lw=2)
    axes[1].axhline(best_val_f1, color='gray', ls=':',
                    label=f'Best = {best_val_f1:.3f}')
    axes[1].set_title('Val macro-F1')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1')
    axes[1].legend()

    cm_arr = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues', ax=axes[2],
                xticklabels=['Healthy', 'Sick'],
                yticklabels=['Healthy', 'Sick'])
    axes[2].set_title('Confusion matrix (test set)')
    axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')

    plt.suptitle('Multimodal Cardiac Fusion Model — Results', fontsize=13)
    plt.tight_layout()
    plt.savefig('/content/results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(" Results plot saved → /content/results.png")

    # Save full checkpoint (weights + threshold) for inference
    torch.save({'state_dict': model.state_dict(),
                'threshold':  best_threshold},
               '/content/best_model_full.pt')
    print("Full checkpoint saved → /content/best_model_full.pt")


# =============================================================================
# 7. INFERENCE + GRAD-CAM EXPLANATION
#    Call predict_and_explain() on any new X-Ray + MRI pair after training.
# =============================================================================
_infer_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


def predict_and_explain(xray_path,
                        mri_path,
                        ckpt_path='/content/best_model_full.pt'):


    # ── Load checkpoint ───────────────────────────────────────────────────────
    ckpt = torch.load(ckpt_path, map_location=device)
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        threshold  = ckpt.get('threshold', 0.5)
        state_dict = ckpt['state_dict']
    else:
        threshold  = 0.5
        state_dict = ckpt

    model = FusionModel().to(device)
    model.load_state_dict(state_dict)
    model.eval()

    # ── Load images ───────────────────────────────────────────────────────────
    if mri_path.endswith('.h5'):
        with h5py.File(mri_path, 'r') as f:
            arr = f['image'][:]
        if arr.ndim == 3:
            arr = arr[arr.shape[0] // 2]
        rng = arr.max() - arr.min()
        arr = ((arr - arr.min()) / (rng + 1e-8) * 255).astype(np.uint8)
        mri_img = Image.fromarray(arr).convert('RGB')
    else:
        mri_img = Image.open(mri_path).convert('RGB')

    xray_img = Image.open(xray_path).convert('RGB')
    x_tensor = _infer_tf(xray_img)
    m_tensor = _infer_tf(mri_img)

    # ── Prediction ────────────────────────────────────────────────────────────
    with torch.no_grad():
        logit      = model(x_tensor.unsqueeze(0).to(device),
                           m_tensor.unsqueeze(0).to(device))
        confidence = torch.sigmoid(logit).item()

    prediction = 'Sick' if confidence >= threshold else 'Healthy'
    pred_idx   = 1 if prediction == 'Sick' else 0
    conf_pct   = confidence * 100

    # ── Grad-CAM ──────────────────────────────────────────────────────────────
    gcam             = GradCAM(model)
    xray_cam, mri_cam = gcam.generate(x_tensor, m_tensor, class_idx=pred_idx)
    xray_region      = dominant_region(xray_cam)
    mri_region       = dominant_region(mri_cam)

    # ── Plain-English reasoning ───────────────────────────────────────────────
    if prediction == 'Sick':
        reasoning = textwrap.dedent(f"""
        PREDICTION : SICK
        Confidence : {conf_pct:.1f}%  (threshold = {threshold:.2f})
        ──────────────────────────────────────────────────────
        The model flagged this patient as likely having a
        cardiac abnormality.

        X-Ray branch:
          The {xray_region} region showed the strongest activation.
          In cardiomegaly screening the model focuses on the
          cardiac silhouette and cardiothoracic ratio.

        MRI branch:
          The {mri_region} region drove the MRI decision.
          This may reflect wall-motion abnormalities, structural
          remodelling, or myocardial tissue changes.

        Fusion:
          Both modalities scored above the decision boundary.
          High agreement between branches raises confidence.

            Research model only — not for clinical use.
            All findings must be reviewed by a cardiologist.
        """).strip()
    else:
        reasoning = textwrap.dedent(f"""
        PREDICTION : HEALTHY
        Confidence : {conf_pct:.1f}%  (threshold = {threshold:.2f})
        ──────────────────────────────────────────────────────
        The model found no strong evidence of cardiac disease.

        X-Ray branch:
          Attention distributed across the {xray_region} region.
          Normal cardiac silhouette reduces the disease score.

        MRI branch:
          MRI activations focused on the {mri_region} region.
          No focal wall-motion or structural anomaly pattern
          was detected.

        Fusion:
          Combined disease probability {conf_pct:.1f}% is below the
          decision threshold of {threshold:.2f} → HEALTHY.

            Research model only — not for clinical use.
            Always confirm with clinical assessment.
        """).strip()

    # ── Visualisation ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    colour    = 'crimson' if prediction == 'Sick' else 'darkgreen'
    fig.suptitle(
        f"Prediction: {prediction}  |  Confidence: {conf_pct:.1f}%  "
        f"(threshold = {threshold:.2f})",
        fontsize=14, color=colour, fontweight='bold')

    # Row 0 — original images + confidence bar
    axes[0, 0].imshow(_to_display(x_tensor))
    axes[0, 0].set_title('X-Ray (input)'); axes[0, 0].axis('off')

    axes[0, 1].imshow(_to_display(m_tensor), cmap='gray')
    axes[0, 1].set_title('MRI (input)'); axes[0, 1].axis('off')

    axes[0, 2].barh(['Healthy', 'Sick'],
                    [1 - confidence, confidence],
                    color=['#2ecc71', '#e74c3c'])
    axes[0, 2].axvline(threshold, color='navy', ls='--', lw=1.5,
                       label=f'Threshold = {threshold:.2f}')
    axes[0, 2].set_xlim(0, 1)
    axes[0, 2].set_title('Prediction confidence')
    axes[0, 2].legend(fontsize=8)

    # Row 1 — Grad-CAM overlays + reasoning
    overlay_heatmap(x_tensor, xray_cam,
                    f'X-Ray Grad-CAM\nFocus: {xray_region}', axes[1, 0])
    overlay_heatmap(m_tensor, mri_cam,
                    f'MRI Grad-CAM\nFocus: {mri_region}',    axes[1, 1])

    axes[1, 2].axis('off')
    axes[1, 2].text(
        0.02, 0.98, reasoning,
        transform=axes[1, 2].transAxes,
        fontsize=8, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85))

    plt.tight_layout()
    out_fig = '/content/explanation.png'
    plt.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n Explanation figure saved → {out_fig}")
    print("\n" + reasoning)

    return {
        'prediction':  prediction,
        'confidence':  round(conf_pct, 2),
        'threshold':   threshold,
        'xray_region': xray_region,
        'mri_region':  mri_region,
        'explanation': reasoning,
    }


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":



    main()



In [ ]:


import os, glob, shutil, textwrap, math
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve,
                             f1_score, precision_recall_curve,
                             average_precision_score)
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
from google.colab import drive

# ── Hardware ──────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == 'cpu':
    print(" Go to Runtime > Change runtime type > T4 GPU!")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
# 1. DATA PIPELINE
# =============================================================================
def setup_data(force_reprocess=False):
    drive.mount('/content/drive')

    DRIVE = "/content/drive/My Drive/Thesis_Data"
    LOCAL = "/content/xray_data"
    TEMP  = "/content/temp_xray_raw"

    sentinel = f"{LOCAL}/train/sick"
    if os.path.exists(sentinel) and not force_reprocess:
        print("Data already on disk. Skipping setup.")
        return

    if force_reprocess and os.path.exists(LOCAL):
        print("Removing old data..."); shutil.rmtree(LOCAL)

    print("\n  Starting data pipeline...")
    if not os.path.exists(TEMP):
        print("   Unzipping X-Rays (4.3 GB)... please wait...")
        shutil.unpack_archive(f"{DRIVE}/nih_xray.zip", TEMP)

    for split in ['train', 'val', 'test']:
        for label in ['sick', 'healthy']:
            os.makedirs(f"{LOCAL}/{split}/{label}", exist_ok=True)

    csvs = glob.glob(f"{TEMP}/**/*.csv", recursive=True)
    if not csvs:
        print(" No CSV found."); return

    print(f"   -> CSV: {os.path.basename(csvs[0])}")
    df = pd.read_csv(csvs[0])

    sick_df    = df[df['Finding Labels'].str.contains('Cardiomegaly', na=False)]
    healthy_df = df[df['Finding Labels'] == 'No Finding']
    n          = min(len(sick_df), len(healthy_df), 500)
    sick_df    = sick_df.sample(n=n, random_state=SEED)
    healthy_df = healthy_df.sample(n=n, random_state=SEED)
    print(f"   -> {n} sick (Cardiomegaly) + {n} healthy (No Finding) selected.")

    all_pngs = glob.glob(f"{TEMP}/**/*.png", recursive=True)
    file_map  = {os.path.basename(f): f for f in all_pngs}

    def three_way_split(df_):
        tr, tmp = train_test_split(df_, test_size=0.30, random_state=SEED)
        vl, te  = train_test_split(tmp,  test_size=0.67, random_state=SEED)
        return tr, vl, te

    tr_s, vl_s, te_s = three_way_split(sick_df)
    tr_n, vl_n, te_n = three_way_split(healthy_df)

    def copy_images(df_, split, label):
        c = 0
        for img in df_['Image Index']:
            if img in file_map:
                shutil.copy(file_map[img], f"{LOCAL}/{split}/{label}/{img}"); c += 1
        return c

    totals = {}
    for df_, sp, lb in [(tr_s,'train','sick'),(vl_s,'val','sick'),(te_s,'test','sick'),
                        (tr_n,'train','healthy'),(vl_n,'val','healthy'),(te_n,'test','healthy')]:
        totals[f"{sp}/{lb}"] = copy_images(df_, sp, lb)

    print(" Data pipeline complete!")
    for sp in ['train','val','test']:
        s = totals.get(f"{sp}/sick",0); h = totals.get(f"{sp}/healthy",0)
        print(f"   {sp:5s}: {s} sick + {h} healthy = {s+h} images")


# =============================================================================
# 2. DATASET
# =============================================================================
TRAIN_TF = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.RandomCrop(380, padding=20),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.95,1.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

EVAL_TF = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

TTA_TFS = [
    transforms.Compose([transforms.Resize((400,400)), transforms.CenterCrop(380),
                        transforms.ToTensor(),
                        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((400,400)), transforms.RandomCrop(380),
                        transforms.ToTensor(),
                        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((400,400)), transforms.RandomCrop(380),
                        transforms.RandomHorizontalFlip(p=1.0),
                        transforms.ToTensor(),
                        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
]


class XRayDataset(Dataset):
    def __init__(self, root_dir, split='train'):
        self.root = os.path.join(root_dir, split)
        self.tf   = TRAIN_TF if split == 'train' else EVAL_TF

        sick_imgs    = sorted(glob.glob(f"{self.root}/sick/*.png"))
        healthy_imgs = sorted(glob.glob(f"{self.root}/healthy/*.png"))
        self.paths   = sick_imgs + healthy_imgs
        self.labels  = [1]*len(sick_imgs) + [0]*len(healthy_imgs)

        print(f"[{split.upper():5s}] {len(sick_imgs)} sick + {len(healthy_imgs)} "
              f"healthy = {len(self.paths)} images")

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
            return self.tf(img), torch.tensor(self.labels[idx], dtype=torch.float)
        except Exception:
            return torch.zeros(3,380,380), torch.tensor(float(self.labels[idx]))

    def get_sampler(self):
        n0 = self.labels.count(0); n1 = self.labels.count(1)
        w  = [1.0/n0 if l==0 else 1.0/n1 for l in self.labels]
        return WeightedRandomSampler(w, num_samples=len(w), replacement=True)


def mixup_batch(x, y, alpha=0.3):
    """MixUp: blends pairs of images/labels. Produces continuous labels."""
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], lam*y + (1-lam)*y[idx]


# =============================================================================
# 3. MODEL  (EfficientNet-B4)
# =============================================================================
class CardiacXRayModel(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        base          = models.efficientnet_b4(weights='DEFAULT')
        self.backbone = base.features
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.head     = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(1792, 512), nn.BatchNorm1d(512), nn.SiLU(),
            nn.Dropout(dropout/2),
            nn.Linear(512, 128),  nn.BatchNorm1d(128),  nn.SiLU(),
            nn.Linear(128, 1)
        )
        self._freeze_backbone()

    def _freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False

    def unfreeze_top_blocks(self, n_blocks=2):
        for block in list(self.backbone.children())[-n_blocks:]:
            for p in block.parameters(): p.requires_grad = True
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"    Phase 2 — top {n_blocks} blocks unfrozen, {n:,} trainable params.")

    def unfreeze_full(self):
        for p in self.backbone.parameters(): p.requires_grad = True
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"    Phase 3 — full network, {n:,} trainable params.")

    def forward(self, x):
        return self.head(torch.flatten(self.pool(self.backbone(x)), 1))


# =============================================================================
# 4. GRAD-CAM
# =============================================================================
class GradCAM:
    def __init__(self, model):
        self.model = model
        self._acts = self._grads = None
        self._h1   = model.backbone[-1].register_forward_hook(
            lambda _,__,o: setattr(self,'_acts',o))
        self._h2   = model.backbone[-1].register_full_backward_hook(
            lambda _,__,gi: setattr(self,'_grads',gi[0]))

    def remove(self): self._h1.remove(); self._h2.remove()

    def generate(self, img_tensor, class_idx=1):
        self.model.eval()
        x   = img_tensor.unsqueeze(0).to(device)
        out = self.model(x)
        self.model.zero_grad()
        (out[0,0] if class_idx==1 else -out[0,0]).backward()
        w   = self._grads.mean(dim=(2,3), keepdim=True)
        cam = F.relu((w * self._acts).sum(dim=1)).squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


def overlay_cam(img_tensor, cam, title, ax, alpha=0.45):
    mean  = np.array([0.485,0.456,0.406]); std = np.array([0.229,0.224,0.225])
    img   = np.clip(img_tensor.permute(1,2,0).cpu().numpy()*std+mean, 0, 1)
    h_up  = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize(
            (img.shape[1], img.shape[0]), Image.BILINEAR)) / 255.0
    blend = (1-alpha)*img + alpha*mpl_cm.jet(h_up)[:,:,:3]
    ax.imshow(blend); ax.set_title(title, fontsize=9, pad=4); ax.axis('off')


def dominant_region(cam):
    h, w = cam.shape
    q = {'upper-left': cam[:h//2,:w//2].mean(), 'upper-right': cam[:h//2,w//2:].mean(),
         'lower-left': cam[h//2:,:w//2].mean(), 'lower-right': cam[h//2:,w//2:].mean()}
    return max(q, key=q.get)


# =============================================================================
# 5. LOSS
# =============================================================================
class LabelSmoothingBCE(nn.Module):
    def __init__(self, smoothing=0.1, pos_weight=None):
        super().__init__()
        self.smoothing = smoothing; self.pos_weight = pos_weight

    def forward(self, logits, targets):
        t = targets * (1-self.smoothing) + 0.5*self.smoothing
        return F.binary_cross_entropy_with_logits(
            logits, t, pos_weight=self.pos_weight, reduction='mean')


# =============================================================================
# 6. SCHEDULER
# =============================================================================
def warmup_cosine_schedule(optimizer, warmup_epochs, total_epochs, base_lr, min_lr=1e-7):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return max(epoch / max(warmup_epochs,1), 1e-6/base_lr)
        p = (epoch-warmup_epochs) / max(total_epochs-warmup_epochs,1)
        return min_lr/base_lr + 0.5*(1-min_lr/base_lr)*(1+math.cos(math.pi*p))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# =============================================================================
# 7. CHECKPOINT HELPERS
# =============================================================================
def save_checkpoint(path, model, threshold, val_auc, test_auc=None):

    torch.save({
        'state_dict': model.state_dict(),
        'threshold':  float(threshold),
        'val_auc':    float(val_auc),
        'test_auc':   float(test_auc) if test_auc is not None else None,
    }, path)


def load_checkpoint(path, model=None):

    ckpt = torch.load(path, map_location=device, weights_only=False)
    if model is not None and 'state_dict' in ckpt:
        model.load_state_dict(ckpt['state_dict'])
    return ckpt


# =============================================================================
# 8. TRAINING HELPER
# =============================================================================
def run_epoch(model, loader, criterion, optimizer=None, use_mixup=False):

    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_true, all_prob = [], []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if is_train and use_mixup:
                imgs, labels = mixup_batch(imgs, labels, alpha=0.3)

            out  = model(imgs).squeeze(1)
            loss = criterion(out, labels)

            if is_train:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            prob = torch.sigmoid(out).detach().cpu().numpy()
            all_true.extend(labels.cpu().numpy().tolist())
            all_prob.extend(prob.tolist() if prob.ndim > 0 else [float(prob)])

    # FIX 1 — round continuous MixUp labels to binary integers
    all_true_int = np.round(np.array(all_true)).astype(int)
    all_pred_int = (np.array(all_prob) >= 0.5).astype(int)
    avg_loss     = total_loss / len(loader)
    f1  = f1_score(all_true_int, all_pred_int, average='macro', zero_division=0)
    try:    auc = roc_auc_score(all_true_int, all_prob)
    except: auc = 0.0
    return avg_loss, f1, auc, all_true_int.tolist(), all_prob


# =============================================================================
# 9. THRESHOLD + TTA
# =============================================================================
def find_youden_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    best = np.argmax(tpr - fpr)
    return float(thresholds[best]), float(tpr[best]), float(1-fpr[best])


def tta_predict(model, img_pil):
    model.eval()
    probs = []
    with torch.no_grad():
        for tf in TTA_TFS:
            t = tf(img_pil).unsqueeze(0).to(device)
            probs.append(torch.sigmoid(model(t)).item())
    return float(np.mean(probs))


# =============================================================================
# 10. MAIN
# =============================================================================
def main():
    setup_data(force_reprocess=False)

    train_ds = XRayDataset("/content/xray_data", split='train')
    val_ds   = XRayDataset("/content/xray_data", split='val')
    test_ds  = XRayDataset("/content/xray_data", split='test')

    if len(train_ds) == 0:
        print(" No data. Run setup_data(force_reprocess=True)."); return

    train_loader = DataLoader(train_ds, batch_size=16, sampler=train_ds.get_sampler(),
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False,
                              num_workers=2, pin_memory=True)

    model  = CardiacXRayModel(dropout=0.4).to(device)
    n_sick = train_ds.labels.count(1); n_healthy = train_ds.labels.count(0)
    pos_w  = torch.tensor([n_healthy/max(n_sick,1)], dtype=torch.float).to(device)
    print(f"\npos_weight = {pos_w.item():.3f}  (train: {n_sick} sick, {n_healthy} healthy)")
    criterion = LabelSmoothingBCE(smoothing=0.1, pos_weight=pos_w)

    P1, P2, P3 = 5, 8, 12
    TOTAL = P1+P2+P3; PATIENCE = 7; CKPT = '/content/best_xray_model.pt'

    optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=1e-3, weight_decay=1e-4)
    scheduler = warmup_cosine_schedule(optimizer, 1, P1, 1e-3, 1e-6)

    best_val_auc = 0.0; patience_ctr = 0
    history = {'train_loss':[],'val_loss':[],'train_auc':[],'val_auc':[],'val_f1':[]}

    print(f"\n Phase 1: Head only ({P1} epochs, backbone frozen)...")

    for epoch in range(1, TOTAL+1):

        if epoch == P1+1:
            print(f"\n Phase 2: Top-2 backbone blocks unfrozen ({P2} epochs)...")
            model.unfreeze_top_blocks(n_blocks=2)
            optimizer = optim.AdamW([
                {'params': [p for n,p in model.named_parameters()
                            if 'head' in n and p.requires_grad], 'lr': 1e-4},
                {'params': [p for n,p in model.named_parameters()
                            if 'head' not in n and p.requires_grad], 'lr': 5e-6},
            ], weight_decay=1e-4)
            scheduler = warmup_cosine_schedule(optimizer, 1, P2, 1e-4, 1e-7)

        if epoch == P1+P2+1:
            print(f"\n Phase 3: Full network fine-tuning ({P3} epochs)...")
            model.unfreeze_full()
            optimizer = optim.AdamW([
                {'params': [p for n,p in model.named_parameters() if 'head' in n],
                 'lr': 5e-5},
                {'params': [p for n,p in model.named_parameters() if 'head' not in n],
                 'lr': 5e-7},
            ], weight_decay=1e-4)
            scheduler = warmup_cosine_schedule(optimizer, 2, P3, 5e-5, 1e-8)

        use_mx = (epoch <= P1+P2)
        tr_loss, tr_f1, tr_auc, *_ = run_epoch(model, train_loader, criterion,
                                                optimizer, use_mixup=use_mx)
        va_loss, va_f1, va_auc, *_ = run_epoch(model, val_loader, criterion)
        scheduler.step()

        for k, v in zip(['train_loss','val_loss','train_auc','val_auc','val_f1'],
                        [tr_loss, va_loss, tr_auc, va_auc, va_f1]):
            history[k].append(v)

        phase = 1 if epoch<=P1 else (2 if epoch<=P1+P2 else 3)
        print(f"   [P{phase}] Ep {epoch:2d}/{TOTAL} | "
              f"Train loss={tr_loss:.4f} AUC={tr_auc:.3f} | "
              f"Val   loss={va_loss:.4f} AUC={va_auc:.3f} F1={va_f1:.3f}")

        if va_auc > best_val_auc:
            best_val_auc = va_auc; patience_ctr = 0
            # FIX 2 & 3: save with pure Python floats
            save_checkpoint(CKPT, model, threshold=0.5, val_auc=best_val_auc)
            print(f"             Saved (val AUC={best_val_auc:.3f})")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n⏹  Early stopping at epoch {epoch}."); break

    # ── Threshold tuning ──────────────────────────────────────────────────────
    print(f"\n  Selecting threshold via Youden's J on validation set...")
    # FIX 2: weights_only=False for PyTorch >= 2.6
    load_checkpoint(CKPT, model)
    _, _, _, val_true, val_prob = run_epoch(model, val_loader, criterion)
    threshold, sensitivity, specificity = find_youden_threshold(val_true, val_prob)
    print(f"   Youden threshold = {threshold:.3f}")
    print(f"   Sensitivity={sensitivity:.3f}  Specificity={specificity:.3f}")

    # ── Final test evaluation ─────────────────────────────────────────────────
    print(f"\n Final test evaluation (TTA + threshold={threshold:.3f})...")
    _, _, _, y_true, y_prob_std = run_epoch(model, test_loader, criterion)

    model.eval()
    y_prob_tta = [tta_predict(model, Image.open(p).convert('RGB'))
                  for p in test_ds.paths]

    y_true_arr = np.array(y_true)
    y_pred_std = (np.array(y_prob_std) >= threshold).astype(int)
    y_pred_tta = (np.array(y_prob_tta) >= threshold).astype(int)

    print("\n" + "="*65)
    print("Standard evaluation (no TTA):")
    print(classification_report(y_true_arr, y_pred_std, target_names=['Healthy','Sick']))
    print("="*65)
    print("With Test-Time Augmentation (TTA):")
    print(classification_report(y_true_arr, y_pred_tta, target_names=['Healthy','Sick']))

    auc_std = roc_auc_score(y_true_arr, y_prob_std)
    auc_tta = roc_auc_score(y_true_arr, y_prob_tta)
    ap      = average_precision_score(y_true_arr, y_prob_tta)
    _, s, sp = find_youden_threshold(y_true_arr, y_prob_tta)
    print(f"\n  ROC-AUC (standard) : {auc_std:.4f}")
    print(f"  ROC-AUC (TTA)      : {auc_tta:.4f}")
    print(f"  Avg Precision      : {ap:.4f}")
    print(f"  Sensitivity        : {s:.4f}")
    print(f"  Specificity        : {sp:.4f}")

    # FIX 2 & 3: save final checkpoint with pure Python floats
    save_checkpoint('/content/cardiac_xray_model.pt', model,
                    threshold=threshold, val_auc=best_val_auc, test_auc=auc_tta)
    print("\n Final checkpoint → /content/cardiac_xray_model.pt")

    _plot_results(history, y_true_arr, y_prob_tta, y_pred_tta, threshold, P1, P2)


# =============================================================================
# 11. RESULTS PLOT
# =============================================================================
def _plot_results(history, y_true, y_prob, y_pred, threshold, P1, P2):
    ep  = range(1, len(history['train_loss'])+1)
    fig = plt.figure(figsize=(20,10))
    gs  = fig.add_gridspec(2, 4, hspace=0.38, wspace=0.35)

    ax1 = fig.add_subplot(gs[0,0])
    ax1.plot(ep, history['train_loss'], label='Train')
    ax1.plot(ep, history['val_loss'],   label='Val')
    for xv,c,lb in [(P1+.5,'gray','P1→P2'),(P1+P2+.5,'steelblue','P2→P3')]:
        if xv < max(ep)+1: ax1.axvline(xv, color=c, ls='--', lw=1, label=lb)
    ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(fontsize=7)

    ax2 = fig.add_subplot(gs[0,1])
    ax2.plot(ep, history['train_auc'], label='Train AUC')
    ax2.plot(ep, history['val_auc'],   label='Val AUC')
    ax2.set_ylim(0,1); ax2.set_title('ROC-AUC over training')
    ax2.set_xlabel('Epoch'); ax2.legend(fontsize=7)

    ax3 = fig.add_subplot(gs[0,2])
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val      = roc_auc_score(y_true, y_prob)
    ax3.plot(fpr, tpr, color='#2a9d8f', lw=2, label=f'AUC={auc_val:.3f}')
    ax3.plot([0,1],[0,1],'--',color='gray',lw=1)
    ax3.set_xlabel('1 - Specificity'); ax3.set_ylabel('Sensitivity')
    ax3.set_title('ROC curve (test, TTA)'); ax3.legend(fontsize=8)

    ax4 = fig.add_subplot(gs[0,3])
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    ax4.plot(rec, prec, color='#e76f51', lw=2, label=f'AP={ap:.3f}')
    ax4.set_xlabel('Recall'); ax4.set_ylabel('Precision')
    ax4.set_title('Precision-Recall (test, TTA)'); ax4.legend(fontsize=8)

    ax5 = fig.add_subplot(gs[1,0])
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax5,
                xticklabels=['Healthy','Sick'], yticklabels=['Healthy','Sick'])
    ax5.set_title('Confusion matrix (TTA)')
    ax5.set_xlabel('Predicted'); ax5.set_ylabel('Actual')

    ax6 = fig.add_subplot(gs[1,1])
    probs = np.array(y_prob)
    ax6.hist(probs[y_true==0], bins=20, alpha=0.6, label='Healthy', color='#2ecc71')
    ax6.hist(probs[y_true==1], bins=20, alpha=0.6, label='Sick',    color='#e74c3c')
    ax6.axvline(threshold, color='navy', ls='--', lw=1.5,
                label=f'Threshold={threshold:.2f}')
    ax6.set_xlabel('Predicted probability'); ax6.set_ylabel('Count')
    ax6.set_title('Confidence distribution'); ax6.legend(fontsize=7)

    ax7 = fig.add_subplot(gs[1,2:])
    ax7.axis('off')
    tn,fp,fn,tp = cm.ravel()
    sens=tp/(tp+fn+1e-8); spec=tn/(tn+fp+1e-8)
    ppv=tp/(tp+fp+1e-8);  npv=tn/(tn+fn+1e-8)
    f1v=f1_score(y_true, y_pred, average='macro', zero_division=0)
    rows=[
        ['Metric','Value','Clinical meaning'],
        ['ROC-AUC',       f'{auc_val:.4f}','Overall discriminability'],
        ['Avg Precision', f'{ap:.4f}',     'Area under PR curve'],
        ['Sensitivity',   f'{sens:.4f}',   'True positive rate'],
        ['Specificity',   f'{spec:.4f}',   'True negative rate'],
        ['PPV',           f'{ppv:.4f}',    'Positive predictive value'],
        ['NPV',           f'{npv:.4f}',    'Negative predictive value'],
        ['Macro F1',      f'{f1v:.4f}',    'Harmonic mean prec/recall'],
        ['Threshold',     f'{threshold:.3f}','Youden-optimal cutoff'],
        ['TP/FP/FN/TN',   f'{tp}/{fp}/{fn}/{tn}','Raw confusion counts'],
    ]
    tbl = ax7.table(cellText=rows[1:], colLabels=rows[0],
                    cellLoc='center', loc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.2,1.5)
    ax7.set_title('Clinical metrics summary (test set, TTA)', pad=14)

    plt.suptitle('Cardiac Disease Detection from Chest X-Ray\n'
                 'EfficientNet-B4  |  NIH ChestX-ray14  |  TTA inference',
                 fontsize=13, fontweight='bold')
    plt.savefig('/content/results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(" Results plot → /content/results.png")


# =============================================================================
# 12. INFERENCE + GRAD-CAM EXPLANATION
# =============================================================================
def predict_and_explain(xray_path,
                        ckpt_path='/content/cardiac_xray_model.pt',
                        use_tta=True):
    """
    Run the trained model on a new chest X-Ray.

    Parameters
    ----------
    xray_path : str   Path to chest X-Ray (.png / .jpg)
    ckpt_path : str   Path to saved checkpoint
    use_tta   : bool  Use Test-Time Augmentation (recommended)

    Returns
    -------
    dict — prediction, confidence, threshold, focus_region, reasoning
    """
    model = CardiacXRayModel().to(device)
    # FIX 2: weights_only=False for PyTorch >= 2.6
    ckpt  = load_checkpoint(ckpt_path, model)
    threshold = float(ckpt.get('threshold', 0.5))
    model.eval()

    img_pil    = Image.open(xray_path).convert('RGB')
    img_tensor = EVAL_TF(img_pil)

    if use_tta:
        confidence = tta_predict(model, img_pil)
        method     = "TTA (3-view average)"
    else:
        with torch.no_grad():
            confidence = torch.sigmoid(
                model(img_tensor.unsqueeze(0).to(device))).item()
        method = "single-view"

    prediction = 'Sick'    if confidence >= threshold else 'Healthy'
    pred_idx   = 1         if prediction == 'Sick'    else 0
    conf_pct   = confidence * 100

    gcam   = GradCAM(model)
    cam    = gcam.generate(img_tensor, class_idx=pred_idx)
    gcam.remove()
    region = dominant_region(cam)

    if prediction == 'Sick':
        reasoning = textwrap.dedent(f"""
        ┌──────────────────────────────────────────────────────┐
        │  PREDICTION  :  SICK  (Cardiomegaly likely)          │
        │  Confidence  :  {conf_pct:5.1f}%                               │
        │  Method      :  {method:<38}│
        │  Threshold   :  {threshold:.3f}  (Youden-optimal on val set)    │
        └──────────────────────────────────────────────────────┘

        Grad-CAM analysis:
          Highest model attention → {region} chest region.

          Clinically relevant regions for cardiomegaly:
            • Central: cardiac silhouette size
            • Lateral borders: cardiothoracic ratio
            • Upper zones: pulmonary vascular congestion

        Interpretation:
          Confidence {conf_pct:.1f}% exceeds threshold {threshold:.3f}.
          Features consistent with enlarged cardiac silhouette.
          Associated with heart failure, DCM, hypertensive
          heart disease.

            RESEARCH MODEL — not for clinical use.
            Confirm with echocardiography and a cardiologist.
        """).strip()
    else:
        reasoning = textwrap.dedent(f"""
        ┌──────────────────────────────────────────────────────┐
        │  PREDICTION  :  HEALTHY  (No Cardiomegaly detected)  │
        │  Confidence  :  {conf_pct:5.1f}%  (disease probability)          │
        │  Method      :  {method:<38}│
        │  Threshold   :  {threshold:.3f}  (Youden-optimal on val set)    │
        └──────────────────────────────────────────────────────┘

        Grad-CAM analysis:
          Attention distributed across {region} region.
          No focal cardiac enlargement pattern detected.

        Interpretation:
          Disease probability {conf_pct:.1f}% is below threshold {threshold:.3f}.
          Cardiac silhouette appears within normal limits.

            RESEARCH MODEL — not for clinical use.
            Normal result does not exclude early cardiac disease.
        """).strip()

    fig, axes = plt.subplots(1, 3, figsize=(16,5))
    colour    = '#c0392b' if prediction=='Sick' else '#27ae60'
    fig.suptitle(f"Prediction: {prediction}  |  Confidence: {conf_pct:.1f}%  |  "
                 f"Threshold: {threshold:.3f}  ({method})",
                 fontsize=13, fontweight='bold', color=colour)

    mean = np.array([0.485,0.456,0.406]); std = np.array([0.229,0.224,0.225])
    orig = np.clip(img_tensor.permute(1,2,0).numpy()*std+mean, 0, 1)
    axes[0].imshow(orig); axes[0].set_title('Input chest X-Ray'); axes[0].axis('off')
    overlay_cam(img_tensor, cam, f'Grad-CAM  (focus: {region})', axes[1])
    axes[2].axis('off')
    axes[2].text(0.01, 0.99, reasoning, transform=axes[2].transAxes,
                 fontsize=7.5, verticalalignment='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='#fffde7', alpha=0.92,
                           edgecolor=colour, linewidth=1.5))

    plt.tight_layout()
    out_fig = '/content/explanation.png'
    plt.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n Explanation saved → {out_fig}")
    print("\n" + reasoning)

    return {'prediction': prediction, 'confidence': round(conf_pct,2),
            'threshold': threshold, 'method': method,
            'focus_region': region, 'reasoning': reasoning}


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    main()



In [ ]:
import shutil
shutil.rmtree("/content/xray_data",     ignore_errors=True)
shutil.rmtree("/content/xray_data_aug", ignore_errors=True)
print("Done — run main() now.")

In [ ]:


import os, glob, shutil, textwrap, math, random
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score,
                             precision_recall_curve, average_precision_score)
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
from google.colab import drive

# ── Hardware ──────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Device: {device}")
if device.type == 'cpu':
    print(" Go to Runtime > Change runtime type > T4 GPU!")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
# 1. OFFLINE AUGMENTATION FUNCTIONS
# =============================================================================
# Each function takes a PIL Image and returns a new PIL Image.
# These are applied once at setup time to permanently expand the dataset.

def aug_rotate(img, angle=None):
    """Random rotation ±15 degrees."""
    if angle is None:
        angle = random.uniform(-15, 15)
    return img.rotate(angle, resample=Image.BILINEAR, expand=False)

def aug_brightness(img, factor=None):
    """Adjust brightness by a random factor in [0.7, 1.3]."""
    if factor is None:
        factor = random.uniform(0.7, 1.3)
    return ImageEnhance.Brightness(img).enhance(factor)

def aug_contrast(img, factor=None):
    """Adjust contrast by a random factor in [0.7, 1.4]."""
    if factor is None:
        factor = random.uniform(0.7, 1.4)
    return ImageEnhance.Contrast(img).enhance(factor)

def aug_sharpness(img, factor=None):
    """Adjust sharpness by a random factor in [0.5, 2.0]."""
    if factor is None:
        factor = random.uniform(0.5, 2.0)
    return ImageEnhance.Sharpness(img).enhance(factor)

def aug_gamma(img, gamma=None):
    """
    Gamma correction — simulates different X-ray exposure levels.
    gamma < 1: brighter (over-exposed), gamma > 1: darker (under-exposed).
    """
    if gamma is None:
        gamma = random.uniform(0.7, 1.4)
    arr    = np.array(img).astype(np.float32) / 255.0
    arr    = np.power(arr, gamma)
    return Image.fromarray((arr * 255).clip(0, 255).astype(np.uint8))

def aug_flip(img):
    """Horizontal flip — valid for chest X-rays (mirror anatomy)."""
    return img.transpose(Image.FLIP_LEFT_RIGHT)

def aug_zoom_crop(img, zoom=None):
    """
    Zoom in and re-crop to original size.
    Simulates different patient distances from the detector.
    """
    if zoom is None:
        zoom = random.uniform(1.05, 1.20)
    w, h   = img.size
    new_w  = int(w * zoom)
    new_h  = int(h * zoom)
    img_z  = img.resize((new_w, new_h), Image.BILINEAR)
    left   = (new_w - w) // 2
    top    = (new_h - h) // 2
    return img_z.crop((left, top, left + w, top + h))

def aug_gaussian_noise(img, std=None):
    """
    Add Gaussian noise to simulate X-ray detector noise.
    Applied in pixel space before normalisation.
    """
    if std is None:
        std = random.uniform(3, 12)
    arr   = np.array(img).astype(np.float32)
    noise = np.random.normal(0, std, arr.shape)
    arr   = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def aug_elastic(img, alpha=None, sigma=None):
    """
    Elastic deformation — simulates soft-tissue variation and breathing motion.
    alpha controls displacement magnitude, sigma controls smoothness.
    """
    if alpha is None:
        alpha = random.uniform(20, 60)
    if sigma is None:
        sigma = random.uniform(5, 8)

    arr = np.array(img).astype(np.float32)
    h, w = arr.shape[:2]

    # Random displacement fields
    dx = np.random.randn(h, w).astype(np.float32) * alpha
    dy = np.random.randn(h, w).astype(np.float32) * alpha

    # Smooth displacement fields with a simple box blur approximation
    k  = int(sigma * 3) | 1   # ensure odd kernel size
    dx = _box_blur(dx, k)
    dy = _box_blur(dy, k)

    # Build pixel coordinate map
    x, y  = np.meshgrid(np.arange(w), np.arange(h))
    map_x = np.clip(x + dx, 0, w - 1).astype(np.float32)
    map_y = np.clip(y + dy, 0, h - 1).astype(np.float32)

    # Bilinear sampling
    if arr.ndim == 3:
        out = np.stack([
            _bilinear_sample(arr[:, :, c], map_y, map_x)
            for c in range(arr.shape[2])], axis=2)
    else:
        out = _bilinear_sample(arr, map_y, map_x)

    return Image.fromarray(out.clip(0, 255).astype(np.uint8))

def _box_blur(arr, k):
    """Simple separable box blur (replaces scipy.ndimage.gaussian_filter)."""
    pad = k // 2
    a   = np.pad(arr, pad, mode='reflect')
    # horizontal pass
    cumsum = np.cumsum(a, axis=1)
    a = (cumsum[:, k:] - cumsum[:, :-k]) / k
    # vertical pass
    cumsum = np.cumsum(a, axis=0)
    a = (cumsum[k:, :] - cumsum[:-k, :]) / k
    return a

def _bilinear_sample(arr, map_y, map_x):
    """Bilinear interpolation for elastic deformation (no OpenCV needed)."""
    h, w  = arr.shape
    x0    = np.floor(map_x).astype(int).clip(0, w - 2)
    y0    = np.floor(map_y).astype(int).clip(0, h - 2)
    x1, y1 = x0 + 1, y0 + 1
    wa    = (x1 - map_x) * (y1 - map_y)
    wb    = (x1 - map_x) * (map_y - y0)
    wc    = (map_x - x0) * (y1 - map_y)
    wd    = (map_x - x0) * (map_y - y0)
    return (wa * arr[y0, x0] + wb * arr[y1, x0] +
            wc * arr[y0, x1] + wd * arr[y1, x1])

def aug_grid_distortion(img, num_steps=5, distort_limit=0.15):
    """
    Grid distortion — displaces a regular grid of control points.
    Simulates slight image warping from patient positioning.
    """
    arr = np.array(img).astype(np.float32)
    h, w = arr.shape[:2]

    xs = np.linspace(0, w, num_steps + 1)
    ys = np.linspace(0, h, num_steps + 1)
    jx = np.random.uniform(-distort_limit, distort_limit, len(xs)) * w
    jy = np.random.uniform(-distort_limit, distort_limit, len(ys)) * h
    xs = np.clip(xs + jx, 0, w)
    ys = np.clip(ys + jy, 0, h)

    # Build dense map by linear interpolation between grid points
    map_x = np.zeros((h, w), dtype=np.float32)
    map_y = np.zeros((h, w), dtype=np.float32)
    for j in range(num_steps):
        for i in range(num_steps):
            x0, x1 = int(xs[i]), int(xs[i + 1])
            y0, y1 = int(ys[j]), int(ys[j + 1])
            if x1 <= x0 or y1 <= y0:
                continue
            for yi in range(y0, min(y1, h)):
                for xi in range(x0, min(x1, w)):
                    tx = (xi - x0) / max(x1 - x0, 1)
                    ty = (yi - y0) / max(y1 - y0, 1)
                    map_x[yi, xi] = xs[i] + tx * (xs[i+1] - xs[i])
                    map_y[yi, xi] = ys[j] + ty * (ys[j+1] - ys[j])

    map_x = np.clip(map_x, 0, w - 1)
    map_y = np.clip(map_y, 0, h - 1)

    if arr.ndim == 3:
        out = np.stack([
            _bilinear_sample(arr[:, :, c], map_y, map_x)
            for c in range(arr.shape[2])], axis=2)
    else:
        out = _bilinear_sample(arr, map_y, map_x)

    return Image.fromarray(out.clip(0, 255).astype(np.uint8))


# Each entry is (function, short_name_suffix)
AUGMENTATION_PIPELINE = [
    (aug_flip,             'flip'),
    (aug_rotate,           'rot'),
    (aug_brightness,       'bright'),
    (aug_contrast,         'contrast'),
    (aug_gamma,            'gamma'),
    (aug_sharpness,        'sharp'),
    (aug_zoom_crop,        'zoom'),
    (aug_gaussian_noise,   'noise'),
    (aug_elastic,          'elastic'),
    # Grid distortion is slow — comment out if setup takes too long
    # (aug_grid_distortion,  'grid'),
]


def apply_offline_augmentation(src_dir, dst_dir, n_augments=7):
    """
    For every original image in src_dir, create n_augments augmented copies
    in dst_dir using randomly selected augmentation functions.

    src_dir : folder containing original .png files
    dst_dir : folder where augmented files are saved (originals also copied)
    n_augments : number of augmented versions per original (default 7 → 8× dataset)
    """
    os.makedirs(dst_dir, exist_ok=True)
    originals = sorted(glob.glob(f"{src_dir}/*.png"))

    if not originals:
        return 0

    total = 0
    for orig_path in originals:
        stem = os.path.splitext(os.path.basename(orig_path))[0]
        img  = Image.open(orig_path).convert('RGB')

        # Always keep the original
        img.save(f"{dst_dir}/{stem}_orig.png")
        total += 1

        # Shuffle pipeline and pick n_augments transforms
        pipeline = random.sample(AUGMENTATION_PIPELINE,
                                 min(n_augments, len(AUGMENTATION_PIPELINE)))

        for aug_fn, suffix in pipeline:
            try:
                aug_img = aug_fn(img)
                aug_img.save(f"{dst_dir}/{stem}_{suffix}.png")
                total += 1
            except Exception as e:
                print(f"     Augmentation {suffix} failed on {stem}: {e}")
                continue

    return total


# =============================================================================
# 2. DATA PIPELINE  (with offline augmentation)
# =============================================================================
def setup_data(force_reprocess=False):
    drive.mount('/content/drive')

    DRIVE    = "/content/drive/My Drive/Thesis_Data"
    LOCAL    = "/content/xray_data"
    LOCAL_AUG = "/content/xray_data_aug"   # augmented version of train only
    TEMP     = "/content/temp_xray_raw"

    sentinel = f"{LOCAL_AUG}/train/sick"
    if os.path.exists(sentinel) and not force_reprocess:
        print(" Data already on disk (with augmentation). Skipping setup.")
        return

    if force_reprocess:
        for d in [LOCAL, LOCAL_AUG]:
            if os.path.exists(d):
                print(f"🗑  Removing {d}...")
                shutil.rmtree(d)

    print("\n  Starting data pipeline...")
    if not os.path.exists(TEMP):
        print("   Unzipping X-Rays (4.3 GB)... please wait...")
        shutil.unpack_archive(f"{DRIVE}/nih_xray.zip", TEMP)

    # ── Create clean train/val/test splits ────────────────────────────────────
    for split in ['train', 'val', 'test']:
        for label in ['sick', 'healthy']:
            os.makedirs(f"{LOCAL}/{split}/{label}", exist_ok=True)

    csvs = glob.glob(f"{TEMP}/**/*.csv", recursive=True)
    if not csvs:
        print(" No CSV found. Check your Drive folder.")
        return

    print(f"   -> CSV: {os.path.basename(csvs[0])}")
    df = pd.read_csv(csvs[0])

    sick_df    = df[df['Finding Labels'].str.contains('Cardiomegaly', na=False)]
    healthy_df = df[df['Finding Labels'] == 'No Finding']

    n = min(len(sick_df), len(healthy_df), 500)
    sick_df    = sick_df.sample(n=n, random_state=SEED)
    healthy_df = healthy_df.sample(n=n, random_state=SEED)
    print(f"   -> {n} sick + {n} healthy X-Rays selected.")

    all_pngs = glob.glob(f"{TEMP}/**/*.png", recursive=True)
    file_map  = {os.path.basename(f): f for f in all_pngs}

    def three_way_split(df_):
        tr, tmp = train_test_split(df_, test_size=0.30, random_state=SEED)
        vl, te  = train_test_split(tmp,  test_size=0.67, random_state=SEED)
        return tr, vl, te

    tr_s, vl_s, te_s = three_way_split(sick_df)
    tr_n, vl_n, te_n = three_way_split(healthy_df)

    def copy_images(df_, split, label):
        c = 0
        for img_name in df_['Image Index']:
            if img_name in file_map:
                shutil.copy(file_map[img_name],
                            f"{LOCAL}/{split}/{label}/{img_name}")
                c += 1
        return c

    totals = {}
    for df_, split, label in [
        (tr_s,'train','sick'), (vl_s,'val','sick'),   (te_s,'test','sick'),
        (tr_n,'train','healthy'),(vl_n,'val','healthy'),(te_n,'test','healthy'),
    ]:
        totals[f"{split}/{label}"] = copy_images(df_, split, label)

    print(" Base data split complete!")
    for sp in ['train', 'val', 'test']:
        s = totals.get(f"{sp}/sick", 0)
        h = totals.get(f"{sp}/healthy", 0)
        print(f"   {sp:5s}: {s} sick + {h} healthy = {s+h} images")

    # ── Offline augmentation on TRAIN only ───────────────────────────────────
    # Val and test stay clean (no augmentation — evaluate on real data only)
    print(f"\n Generating offline augmentations for training set...")
    print(f"   Each original image → 1 original + 7 augmented = 8× dataset")

    for split in ['val', 'test']:
        for label in ['sick', 'healthy']:
            os.makedirs(f"{LOCAL_AUG}/{split}/{label}", exist_ok=True)
            for f in glob.glob(f"{LOCAL}/{split}/{label}/*.png"):
                shutil.copy(f, f"{LOCAL_AUG}/{split}/{label}/")

    aug_totals = {}
    for label in ['sick', 'healthy']:
        src = f"{LOCAL}/train/{label}"
        dst = f"{LOCAL_AUG}/train/{label}"
        n_created = apply_offline_augmentation(src, dst, n_augments=7)
        aug_totals[label] = n_created
        print(f"   train/{label}: {totals.get(f'train/{label}',0)} originals "
              f"→ {n_created} total (including augmented)")

    total_train = sum(aug_totals.values())
    print(f"\n Augmentation complete!")
    print(f"   Final train set: {aug_totals['sick']} sick "
          f"+ {aug_totals['healthy']} healthy = {total_train} images")
    print(f"   Val/Test unchanged (clean evaluation data)")


# =============================================================================
# 3. TRANSFORMS  (lighter online augmentation — heavy work done offline)
# =============================================================================
# Online transforms are kept light since offline augmentation already provides
# huge variety. We still apply random flip + slight rotation online for
# additional stochasticity during training.

TRAIN_TF = transforms.Compose([
    transforms.Resize((400, 400)),
    transforms.RandomCrop(380),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),          # small — heavy rotation done offline
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

EVAL_TF = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

TTA_TFS = [
    transforms.Compose([
        transforms.Resize((400, 400)),
        transforms.CenterCrop(380),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((400, 400)),
        transforms.RandomCrop(380),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((400, 400)),
        transforms.RandomCrop(380),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
]


# =============================================================================
# 4. DATASET
# =============================================================================
class XRayDataset(Dataset):
    def __init__(self, root_dir, split='train'):
        self.root  = os.path.join(root_dir, split)
        self.split = split
        self.tf    = TRAIN_TF if split == 'train' else EVAL_TF

        sick_imgs    = sorted(glob.glob(f"{self.root}/sick/*.png"))
        healthy_imgs = sorted(glob.glob(f"{self.root}/healthy/*.png"))

        self.paths  = sick_imgs + healthy_imgs
        self.labels = [1] * len(sick_imgs) + [0] * len(healthy_imgs)

        print(f"[{split.upper():5s}] "
              f"{len(sick_imgs)} sick + {len(healthy_imgs)} healthy "
              f"= {len(self.paths)} images")

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
            return self.tf(img), torch.tensor(self.labels[idx], dtype=torch.float)
        except Exception:
            return (torch.zeros(3, 380, 380),
                    torch.tensor(float(self.labels[idx])))

    def get_sampler(self):
        n0 = self.labels.count(0)
        n1 = self.labels.count(1)
        w  = [1.0 / n0 if l == 0 else 1.0 / n1 for l in self.labels]
        return WeightedRandomSampler(w, num_samples=len(w), replacement=True)


# =============================================================================
# 5. MIXUP + CUTMIX
# =============================================================================
def mixup_batch(x, y, alpha=0.3):
    """Blend pairs of images and labels (continuous labels produced)."""
    if alpha <= 0:
        return x, y
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], lam * y + (1 - lam) * y[idx]


def cutmix_batch(x, y, alpha=1.0):
    """
    CutMix: paste a rectangular patch from one image into another.
    Encourages the model to use the entire image, not just one region.
    """
    if alpha <= 0:
        return x, y
    lam  = np.random.beta(alpha, alpha)
    idx  = torch.randperm(x.size(0), device=x.device)
    _, _, H, W = x.shape

    cut_h = int(H * math.sqrt(1 - lam))
    cut_w = int(W * math.sqrt(1 - lam))
    cx    = random.randint(0, W)
    cy    = random.randint(0, H)

    x1 = max(cx - cut_w // 2, 0)
    y1 = max(cy - cut_h // 2, 0)
    x2 = min(cx + cut_w // 2, W)
    y2 = min(cy + cut_h // 2, H)

    x_mix        = x.clone()
    x_mix[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    actual_lam   = 1 - (x2 - x1) * (y2 - y1) / (H * W)
    y_mix        = actual_lam * y + (1 - actual_lam) * y[idx]
    return x_mix, y_mix


def apply_batch_augmentation(x, y, epoch, phase):
    """
    Randomly apply either MixUp or CutMix per batch.
    Phase 1: MixUp only (gentler — head is still learning)
    Phase 2: alternate MixUp and CutMix
    Phase 3: no batch augmentation (fine-tuning needs clean gradients)
    """
    if phase == 1:
        return mixup_batch(x, y, alpha=0.3)
    elif phase == 2:
        if random.random() < 0.5:
            return mixup_batch(x, y, alpha=0.4)
        else:
            return cutmix_batch(x, y, alpha=1.0)
    else:
        return x, y   # phase 3: no mixing


# =============================================================================
# 6. MODEL — EfficientNet-B4
# =============================================================================
class CardiacXRayModel(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        base          = models.efficientnet_b4(weights='DEFAULT')
        self.backbone = base.features
        self.pool     = nn.AdaptiveAvgPool2d(1)

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(1792, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Linear(128, 1)
        )
        self._freeze_backbone()

    def _freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_top_blocks(self, n_blocks=2):
        for block in list(self.backbone.children())[-n_blocks:]:
            for p in block.parameters():
                p.requires_grad = True
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"   Phase 2 — top {n_blocks} blocks unfrozen, {n:,} params.")

    def unfreeze_full(self):
        for p in self.backbone.parameters():
            p.requires_grad = True
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"   Phase 3 — full network, {n:,} params.")

    def forward(self, x):
        return self.head(torch.flatten(self.pool(self.backbone(x)), 1))


# =============================================================================
# 7. LOSS — Label Smoothing BCE
# =============================================================================
class LabelSmoothingBCE(nn.Module):
    def __init__(self, smoothing=0.1, pos_weight=None):
        super().__init__()
        self.smoothing  = smoothing
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        targets_smooth = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(
            logits, targets_smooth,
            pos_weight=self.pos_weight,
            reduction='mean')


# =============================================================================
# 8. SCHEDULER — Linear warmup → cosine annealing
# =============================================================================
def warmup_cosine_schedule(optimizer, warmup_epochs, total_epochs,
                           base_lr, min_lr=1e-7):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return max(epoch / max(warmup_epochs, 1), 1e-6 / base_lr)
        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        cosine   = 0.5 * (1 + math.cos(math.pi * progress))
        return min_lr / base_lr + cosine * (1 - min_lr / base_lr)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# =============================================================================
# 9. TRAINING HELPER
# =============================================================================
def run_epoch(model, loader, criterion, optimizer=None,
              use_aug=False, current_phase=1):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_true, all_prob = [], []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            if is_train and use_aug:
                imgs, labels = apply_batch_augmentation(
                    imgs, labels, epoch=0, phase=current_phase)

            out  = model(imgs).squeeze(1)
            loss = criterion(out, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            prob = torch.sigmoid(out).detach().cpu().numpy()
            all_true.extend(labels.cpu().numpy().tolist())
            all_prob.extend(prob.tolist() if prob.ndim > 0 else [float(prob)])

    # Round continuous MixUp/CutMix labels back to 0/1 for sklearn
    all_true_int = np.round(np.array(all_true)).astype(int)
    all_pred_int = (np.array(all_prob) >= 0.5).astype(int)
    avg_loss     = total_loss / len(loader)

    f1 = f1_score(all_true_int, all_pred_int, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(all_true_int, all_prob)
    except Exception:
        auc = 0.0

    return avg_loss, f1, auc, all_true_int.tolist(), all_prob


# =============================================================================
# 10. CHECKPOINT HELPERS  (PyTorch 2.6 safe — pure Python scalars only)
# =============================================================================
def save_checkpoint(path, model, val_auc, threshold=None, test_auc=None):
    obj = {'state_dict': model.state_dict(),
           'val_auc':    float(val_auc)}
    if threshold is not None: obj['threshold'] = float(threshold)
    if test_auc  is not None: obj['test_auc']  = float(test_auc)
    torch.save(obj, path)


def load_checkpoint(path, model=None):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    if model is not None and 'state_dict' in ckpt:
        model.load_state_dict(ckpt['state_dict'])
    return ckpt


# =============================================================================
# 11. THRESHOLD SELECTION — Youden's J
# =============================================================================
def find_youden_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j_scores = tpr - fpr
    best_idx = int(np.argmax(j_scores))
    return (float(thresholds[best_idx]),
            float(tpr[best_idx]),
            float(1 - fpr[best_idx]))


# =============================================================================
# 12. TEST-TIME AUGMENTATION
# =============================================================================
def tta_predict(model, img_pil):
    model.eval()
    probs = []
    with torch.no_grad():
        for tf in TTA_TFS:
            t = tf(img_pil).unsqueeze(0).to(device)
            probs.append(torch.sigmoid(model(t)).item())
    return float(np.mean(probs))


# =============================================================================
# 13. MAIN TRAINING LOOP
# =============================================================================
def main():
    setup_data(force_reprocess=False)

    # Use augmented data for training, clean data for val/test
    train_ds = XRayDataset("/content/xray_data_aug", split='train')
    val_ds   = XRayDataset("/content/xray_data_aug", split='val')
    test_ds  = XRayDataset("/content/xray_data_aug", split='test')

    if len(train_ds) == 0:
        print("No data found. Run setup_data(force_reprocess=True).")
        return

    train_loader = DataLoader(train_ds, batch_size=16,
                              sampler=train_ds.get_sampler(),
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False,
                              num_workers=2, pin_memory=True)

    model = CardiacXRayModel(dropout=0.4).to(device)

    n_sick    = train_ds.labels.count(1)
    n_healthy = train_ds.labels.count(0)
    pos_w     = torch.tensor([n_healthy / max(n_sick, 1)],
                              dtype=torch.float).to(device)
    print(f"\npos_weight = {pos_w.item():.3f}  "
          f"(train: {n_sick} sick, {n_healthy} healthy)")

    criterion = LabelSmoothingBCE(smoothing=0.1, pos_weight=pos_w)

    P1, P2, P3 = 7, 10, 15
    TOTAL      = P1 + P2 + P3
    PATIENCE   = 8
    CKPT       = '/content/best_xray_model.pt'

    optimizer = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=1e-3, weight_decay=1e-4)
    scheduler = warmup_cosine_schedule(
        optimizer, warmup_epochs=2, total_epochs=P1,
        base_lr=1e-3, min_lr=1e-6)

    best_val_auc = 0.0
    patience_ctr = 0
    current_phase = 1
    history = {'train_loss':[], 'val_loss':[],
               'train_auc':[], 'val_auc':[], 'val_f1':[]}

    print(f"\n Phase 1: Head only ({P1} epochs, backbone frozen)...")

    for epoch in range(1, TOTAL + 1):

        # ── Phase transitions ─────────────────────────────────────────────────
        if epoch == P1 + 1:
            current_phase = 2
            print(f"\n🔓 Phase 2: Top-2 backbone blocks ({P2} epochs)...")
            model.unfreeze_top_blocks(n_blocks=2)
            optimizer = optim.AdamW([
                {'params': [p for n, p in model.named_parameters()
                            if 'head' in n and p.requires_grad], 'lr': 1e-4},
                {'params': [p for n, p in model.named_parameters()
                            if 'head' not in n and p.requires_grad], 'lr': 5e-6},
            ], weight_decay=1e-4)
            scheduler = warmup_cosine_schedule(
                optimizer, warmup_epochs=1, total_epochs=P2,
                base_lr=1e-4, min_lr=1e-7)

        if epoch == P1 + P2 + 1:
            current_phase = 3
            print(f"\n Phase 3: Full fine-tuning ({P3} epochs)...")
            model.unfreeze_full()
            optimizer = optim.AdamW([
                {'params': [p for n, p in model.named_parameters()
                            if 'head' in n],    'lr': 5e-5},
                {'params': [p for n, p in model.named_parameters()
                            if 'head' not in n],'lr': 5e-7},
            ], weight_decay=1e-4)
            scheduler = warmup_cosine_schedule(
                optimizer, warmup_epochs=2, total_epochs=P3,
                base_lr=5e-5, min_lr=1e-8)

        # ── Train + validate ──────────────────────────────────────────────────
        use_aug = (current_phase <= 2)   # no batch aug in phase 3

        tr_loss, tr_f1, tr_auc, *_ = run_epoch(
            model, train_loader, criterion, optimizer,
            use_aug=use_aug, current_phase=current_phase)
        va_loss, va_f1, va_auc, *_ = run_epoch(
            model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_auc'].append(tr_auc)
        history['val_auc'].append(va_auc)
        history['val_f1'].append(va_f1)

        print(f"   [P{current_phase}] Ep {epoch:2d}/{TOTAL} | "
              f"Train loss={tr_loss:.4f} AUC={tr_auc:.3f} | "
              f"Val   loss={va_loss:.4f} AUC={va_auc:.3f} F1={va_f1:.3f}")

        if va_auc > best_val_auc:
            best_val_auc = va_auc
            patience_ctr = 0
            save_checkpoint(CKPT, model, val_auc=best_val_auc)
            print(f"             Saved (val AUC={best_val_auc:.3f})")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n⏹  Early stopping at epoch {epoch}.")
                break

    # ── Threshold selection ───────────────────────────────────────────────────
    print(f"\n  Selecting threshold via Youden's J on validation set...")
    load_checkpoint(CKPT, model)
    _, _, _, val_true, val_prob = run_epoch(model, val_loader, criterion)
    threshold, val_sens, val_spec = find_youden_threshold(val_true, val_prob)
    print(f"   Youden threshold = {threshold:.3f}")
    print(f"   Sensitivity={val_sens:.3f}, Specificity={val_spec:.3f}")

    save_checkpoint(CKPT, model, val_auc=best_val_auc, threshold=threshold)

    # ── Final test evaluation ─────────────────────────────────────────────────
    print(f"\n Final test evaluation (TTA + threshold={threshold:.3f})...")

    _, _, _, y_true, y_prob_std = run_epoch(model, test_loader, criterion)

    model.eval()
    y_prob_tta = []
    for path in test_ds.paths:
        img = Image.open(path).convert('RGB')
        y_prob_tta.append(tta_predict(model, img))

    y_true_arr = np.array(y_true)
    y_pred_std = (np.array(y_prob_std) >= threshold).astype(int)
    y_pred_tta = (np.array(y_prob_tta) >= threshold).astype(int)

    print("\n" + "=" * 65)
    print("Standard evaluation (no TTA):")
    print(classification_report(y_true_arr, y_pred_std,
                                target_names=['Healthy', 'Sick']))
    print("=" * 65)
    print("With Test-Time Augmentation (TTA):")
    print(classification_report(y_true_arr, y_pred_tta,
                                target_names=['Healthy', 'Sick']))

    auc_std = roc_auc_score(y_true_arr, y_prob_std)
    auc_tta = roc_auc_score(y_true_arr, y_prob_tta)
    ap      = average_precision_score(y_true_arr, y_prob_tta)
    _, sens_t, spec_t = find_youden_threshold(y_true_arr, y_prob_tta)

    print(f"\n  ROC-AUC (standard) : {auc_std:.4f}")
    print(f"  ROC-AUC (TTA)      : {auc_tta:.4f}")
    print(f"  Avg Precision      : {ap:.4f}")
    print(f"  Sensitivity        : {sens_t:.4f}")
    print(f"  Specificity        : {spec_t:.4f}")

    final_ckpt_path = '/content/cardiac_xray_model.pt'
    save_checkpoint(final_ckpt_path, model,
                    val_auc=best_val_auc,
                    threshold=threshold,
                    test_auc=auc_tta)
    print(f"\n Final checkpoint → {final_ckpt_path}")

    _plot_results(history, y_true_arr, y_prob_tta, y_pred_tta,
                  threshold, P1, P2)


# =============================================================================
# 14. RESULTS PLOT — 7-panel clinical dashboard
# =============================================================================
def _plot_results(history, y_true, y_prob, y_pred, threshold, P1, P2):
    ep  = range(1, len(history['train_loss']) + 1)
    fig = plt.figure(figsize=(20, 10))
    gs  = fig.add_gridspec(2, 4, hspace=0.38, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(ep, history['train_loss'], label='Train')
    ax1.plot(ep, history['val_loss'],   label='Val')
    for xv, c, lbl in [(P1 + 0.5, 'gray', 'P1→P2'),
                        (P1 + P2 + 0.5, 'steelblue', 'P2→P3')]:
        if xv < max(ep) + 1:
            ax1.axvline(xv, color=c, ls='--', lw=1, label=lbl)
    ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(fontsize=7)

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(ep, history['train_auc'], label='Train AUC')
    ax2.plot(ep, history['val_auc'],   label='Val AUC')
    ax2.set_ylim(0, 1)
    ax2.set_title('ROC-AUC over training')
    ax2.set_xlabel('Epoch'); ax2.legend(fontsize=7)

    ax3 = fig.add_subplot(gs[0, 2])
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val      = roc_auc_score(y_true, y_prob)
    ax3.plot(fpr, tpr, color='#2a9d8f', lw=2, label=f'AUC = {auc_val:.3f}')
    ax3.plot([0,1],[0,1],'--', color='gray', lw=1)
    ax3.set_xlabel('1 - Specificity'); ax3.set_ylabel('Sensitivity')
    ax3.set_title('ROC curve (test, TTA)'); ax3.legend(fontsize=8)

    ax4 = fig.add_subplot(gs[0, 3])
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    ax4.plot(rec, prec, color='#e76f51', lw=2, label=f'AP = {ap:.3f}')
    ax4.set_xlabel('Recall'); ax4.set_ylabel('Precision')
    ax4.set_title('Precision-Recall (test, TTA)'); ax4.legend(fontsize=8)

    ax5 = fig.add_subplot(gs[1, 0])
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax5,
                xticklabels=['Healthy','Sick'],
                yticklabels=['Healthy','Sick'])
    ax5.set_title('Confusion matrix (TTA)')
    ax5.set_xlabel('Predicted'); ax5.set_ylabel('Actual')

    ax6 = fig.add_subplot(gs[1, 1])
    probs = np.array(y_prob)
    ax6.hist(probs[y_true==0], bins=20, alpha=0.6, label='Healthy', color='#2ecc71')
    ax6.hist(probs[y_true==1], bins=20, alpha=0.6, label='Sick',    color='#e74c3c')
    ax6.axvline(threshold, color='navy', ls='--', lw=1.5,
                label=f'Threshold={threshold:.2f}')
    ax6.set_xlabel('Predicted probability'); ax6.set_ylabel('Count')
    ax6.set_title('Confidence distribution'); ax6.legend(fontsize=7)

    ax7 = fig.add_subplot(gs[1, 2:])
    ax7.axis('off')
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn + 1e-8)
    spec = tn / (tn + fp + 1e-8)
    ppv  = tp / (tp + fp + 1e-8)
    npv  = tn / (tn + fn + 1e-8)
    f1v  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    rows = [
        ['Metric',           'Value',               'Clinical meaning'],
        ['ROC-AUC',          f'{auc_val:.4f}',      'Overall discriminability'],
        ['Avg Precision',    f'{ap:.4f}',            'Area under PR curve'],
        ['Sensitivity',      f'{sens:.4f}',          'True positive rate'],
        ['Specificity',      f'{spec:.4f}',          'True negative rate'],
        ['PPV (Precision)',  f'{ppv:.4f}',           'Positive predictive value'],
        ['NPV',              f'{npv:.4f}',           'Negative predictive value'],
        ['Macro F1',         f'{f1v:.4f}',           'Harmonic mean prec/recall'],
        ['Threshold',        f'{threshold:.3f}',     'Youden-optimal cutoff'],
        ['TP/FP/FN/TN',      f'{tp}/{fp}/{fn}/{tn}', 'Raw counts'],
    ]
    tbl = ax7.table(cellText=rows[1:], colLabels=rows[0],
                    cellLoc='center', loc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.2, 1.5)
    ax7.set_title('Clinical metrics summary (test, TTA)', pad=14)

    plt.suptitle(
        'Cardiac Disease Detection from Chest X-Ray\n'
        'EfficientNet-B4  |  NIH ChestX-ray14  |  Offline Aug + TTA',
        fontsize=13, fontweight='bold')
    plt.savefig('/content/results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(" Results plot → /content/results.png")


# =============================================================================
# 15. GRAD-CAM
# =============================================================================
class GradCAM:
    def __init__(self, model):
        self.model  = model
        self._acts  = None
        self._grads = None
        self._h1    = model.backbone[-1].register_forward_hook(
            lambda _, __, o: setattr(self, '_acts', o))
        self._h2    = model.backbone[-1].register_full_backward_hook(
            lambda _, __, gi: setattr(self, '_grads', gi[0]))

    def remove(self): self._h1.remove(); self._h2.remove()

    def generate(self, img_tensor, class_idx=1):
        self.model.eval()
        x   = img_tensor.unsqueeze(0).to(device)
        out = self.model(x)
        self.model.zero_grad()
        (out[0, 0] if class_idx == 1 else -out[0, 0]).backward()
        weights = self._grads.mean(dim=(2, 3), keepdim=True)
        cam     = F.relu((weights * self._acts).sum(dim=1)).squeeze()
        cam     = cam.detach().cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


def overlay_cam(img_tensor, cam, title, ax, alpha=0.45):
    mean  = np.array([0.485, 0.456, 0.406])
    std   = np.array([0.229, 0.224, 0.225])
    img   = np.clip(img_tensor.permute(1,2,0).cpu().numpy() * std + mean, 0, 1)
    h_up  = np.array(Image.fromarray(
        (cam * 255).astype(np.uint8)).resize(
        (img.shape[1], img.shape[0]), Image.BILINEAR)) / 255.0
    heat  = mpl_cm.jet(h_up)[:, :, :3]
    ax.imshow((1 - alpha) * img + alpha * heat)
    ax.set_title(title, fontsize=9, pad=4); ax.axis('off')


def dominant_region(cam):
    h, w  = cam.shape
    quads = {'upper-left':  cam[:h//2, :w//2].mean(),
             'upper-right': cam[:h//2, w//2:].mean(),
             'lower-left':  cam[h//2:, :w//2].mean(),
             'lower-right': cam[h//2:, w//2:].mean()}
    return max(quads, key=quads.get)


# =============================================================================
# 16. INFERENCE + GRAD-CAM EXPLANATION
# =============================================================================
def predict_and_explain(xray_path,
                        ckpt_path='/content/cardiac_xray_model.pt',
                        use_tta=True):
    """
    Run the trained model on a new chest X-Ray.

    Returns prediction, confidence, Grad-CAM heatmap, and
    plain-English clinical reasoning.
    """
    model = CardiacXRayModel().to(device)
    ckpt  = load_checkpoint(ckpt_path, model)
    model.eval()
    threshold = float(ckpt.get('threshold', 0.5))

    img_pil    = Image.open(xray_path).convert('RGB')
    img_tensor = EVAL_TF(img_pil)

    if use_tta:
        confidence = tta_predict(model, img_pil)
        method     = "TTA (3-view average)"
    else:
        with torch.no_grad():
            confidence = torch.sigmoid(
                model(img_tensor.unsqueeze(0).to(device))).item()
        method = "single-view"

    prediction = 'Sick'    if confidence >= threshold else 'Healthy'
    pred_idx   = 1         if prediction == 'Sick'    else 0
    conf_pct   = confidence * 100

    gcam   = GradCAM(model)
    cam    = gcam.generate(img_tensor, class_idx=pred_idx)
    gcam.remove()
    region = dominant_region(cam)

    if prediction == 'Sick':
        reasoning = textwrap.dedent(f"""
        ┌──────────────────────────────────────────────────────┐
        │  PREDICTION  :  SICK  (Cardiomegaly likely)          │
        │  Confidence  :  {conf_pct:5.1f}%                               │
        │  Method      :  {method:<38}│
        │  Threshold   :  {threshold:.3f}  (Youden-optimal on val set)    │
        └──────────────────────────────────────────────────────┘

        Grad-CAM analysis:
          Highest attention → {region} chest region.

          Key regions for cardiomegaly:
            • Central: cardiac silhouette size
            • Lateral: cardiothoracic ratio borders
            • Upper zones: pulmonary vascular congestion

        Interpretation:
          Confidence {conf_pct:.1f}% exceeds threshold {threshold:.3f}.
          Features consistent with enlarged cardiac silhouette.
          Associated with heart failure, DCM, hypertension.

            RESEARCH MODEL — not for clinical use.
            Confirm with echocardiography and cardiologist review.
        """).strip()
    else:
        reasoning = textwrap.dedent(f"""
        ┌──────────────────────────────────────────────────────┐
        │  PREDICTION  :  HEALTHY  (No Cardiomegaly detected)  │
        │  Confidence  :  {conf_pct:5.1f}%  (disease probability)          │
        │  Method      :  {method:<38}│
        │  Threshold   :  {threshold:.3f}  (Youden-optimal on val set)    │
        └──────────────────────────────────────────────────────┘

        Grad-CAM analysis:
          Attention distributed across {region} region.
          No focal enlargement pattern detected.

        Interpretation:
          Disease probability {conf_pct:.1f}% < threshold {threshold:.3f}.
          Cardiac silhouette and CTR appear within normal limits.

            RESEARCH MODEL — not for clinical use.
            Does not exclude early or subtle cardiac disease.
        """).strip()

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    colour    = '#c0392b' if prediction == 'Sick' else '#27ae60'
    fig.suptitle(
        f"Prediction: {prediction}  |  Confidence: {conf_pct:.1f}%  |  "
        f"Threshold: {threshold:.3f}  ({method})",
        fontsize=13, fontweight='bold', color=colour)

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    orig = np.clip(img_tensor.permute(1,2,0).numpy() * std + mean, 0, 1)
    axes[0].imshow(orig)
    axes[0].set_title('Input chest X-Ray'); axes[0].axis('off')

    overlay_cam(img_tensor, cam, f'Grad-CAM  (focus: {region})', axes[1])

    axes[2].axis('off')
    axes[2].text(0.01, 0.99, reasoning,
                 transform=axes[2].transAxes,
                 fontsize=7.5, verticalalignment='top',
                 fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='#fffde7',
                           alpha=0.92, edgecolor=colour, linewidth=1.5))

    plt.tight_layout()
    out_fig = '/content/explanation.png'
    plt.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n Explanation saved → {out_fig}")
    print("\n" + reasoning)

    return {'prediction':   prediction,
            'confidence':   round(conf_pct, 2),
            'threshold':    round(threshold, 3),
            'method':       method,
            'focus_region': region,
            'reasoning':    reasoning}


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    # NOTE: First run will be slower — augmentation creates ~1,500 images.
    # Subsequent runs skip augmentation (data already on disk).
    # To force a full rebuild: setup_data(force_reprocess=True)
    main()

    # ── Inference on a new patient after training ─────────────────────────────
    # result = predict_and_explain(
    #     xray_path = "/content/drive/My Drive/Thesis_Data/new_patient.png",
    #     use_tta   = True,
    # )

In [ ]:
import shutil
shutil.rmtree("/content/xray_data_aug", ignore_errors=True)
print("Done — run main() now.")

In [ ]:
# Cell 1: Delete only the augmented training folder (keeps val/test intact)
import shutil, glob, os

# Check what's actually in the healthy train folder
healthy_files = glob.glob("/content/xray_data_aug/train/healthy/*.png")
sick_files    = glob.glob("/content/xray_data_aug/train/sick/*.png")
print(f"Current train — sick: {len(sick_files)}, healthy: {len(healthy_files)}")

# Also check the base (non-augmented) folder
base_healthy = glob.glob("/content/xray_data/train/healthy/*.png")
base_sick    = glob.glob("/content/xray_data/train/sick/*.png")
print(f"Base train    — sick: {len(base_sick)}, healthy: {len(base_healthy)}")

In [ ]:
# Cell 2: Full clean rebuild
import shutil
shutil.rmtree("/content/xray_data",     ignore_errors=True)
shutil.rmtree("/content/xray_data_aug", ignore_errors=True)
print("Both data folders deleted. Run main() now.")


In [ ]:


import os, glob, shutil, textwrap, math, random
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score,
                             precision_recall_curve, average_precision_score)
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
from google.colab import drive

# ── Hardware ──────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Device: {device}")
if device.type == 'cpu':
    print(" Go to Runtime > Change runtime type > T4 GPU!")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
# 1. OFFLINE AUGMENTATION FUNCTIONS
# =============================================================================

def aug_rotate(img):
    """Random rotation ±15 degrees."""
    return img.rotate(random.uniform(-15, 15),
                      resample=Image.BILINEAR, expand=False)


def aug_brightness(img):
    """Brightness [0.7, 1.3] — simulates X-ray exposure variation."""
    return ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))


def aug_contrast(img):
    """Contrast [0.7, 1.4] — simulates different kVp settings."""
    return ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.4))


def aug_sharpness(img):
    """Sharpness [0.5, 2.0]."""
    return ImageEnhance.Sharpness(img).enhance(random.uniform(0.5, 2.0))


def aug_gamma(img):
    """
    Gamma correction — simulates different X-ray exposure levels.
    gamma < 1: brighter (over-exposed), gamma > 1: darker (under-exposed).
    """
    gamma = random.uniform(0.7, 1.4)
    arr   = np.array(img).astype(np.float32) / 255.0
    arr   = np.power(arr, gamma)
    return Image.fromarray((arr * 255).clip(0, 255).astype(np.uint8))


def aug_flip(img):
    """Horizontal flip — valid for chest X-rays (mirror anatomy)."""
    return img.transpose(Image.FLIP_LEFT_RIGHT)


def aug_zoom_crop(img):
    """Zoom in [1.05–1.20] and re-crop — simulates detector distance variation."""
    zoom  = random.uniform(1.05, 1.20)
    w, h  = img.size
    nw, nh = int(w * zoom), int(h * zoom)
    img_z = img.resize((nw, nh), Image.BILINEAR)
    left  = (nw - w) // 2
    top   = (nh - h) // 2
    return img_z.crop((left, top, left + w, top + h))


def aug_gaussian_noise(img):
    """Gaussian noise std=[3, 12] — simulates X-ray detector noise."""
    std = random.uniform(3, 12)
    arr = np.array(img).astype(np.float32)
    arr = np.clip(arr + np.random.normal(0, std, arr.shape), 0, 255).astype(np.uint8)
    return Image.fromarray(arr)


# ── Box blur (shape-safe) ─────────────────────────────────────────────────────
def _box_blur(arr, k):


    h_in, w_in = arr.shape
    pad_l = k // 2
    pad_r = k - pad_l   # pad_r = pad_l + 1 for odd k → total = k

    # Horizontal pass
    a      = np.pad(arr, ((0, 0), (pad_l, pad_r)), mode='reflect')
    cumsum = np.cumsum(a, axis=1)
    # output width = (w_in + k) - k = w_in  ✓
    a      = (cumsum[:, k:] - cumsum[:, :-k]) / k

    # Vertical pass
    a      = np.pad(a, ((pad_l, pad_r), (0, 0)), mode='reflect')
    cumsum = np.cumsum(a, axis=0)
    # output height = (h_in + k) - k = h_in  ✓
    a      = (cumsum[k:, :] - cumsum[:-k, :]) / k

    # Safety clip — should be a no-op with correct padding above
    return a[:h_in, :w_in]


def _bilinear_sample(arr, map_y, map_x):
    """Bilinear interpolation — no OpenCV required."""
    h, w   = arr.shape
    x0     = np.floor(map_x).astype(int).clip(0, w - 2)
    y0     = np.floor(map_y).astype(int).clip(0, h - 2)
    x1, y1 = x0 + 1, y0 + 1
    wa     = (x1 - map_x) * (y1 - map_y)
    wb     = (x1 - map_x) * (map_y - y0)
    wc     = (map_x - x0) * (y1 - map_y)
    wd     = (map_x - x0) * (map_y - y0)
    return (wa * arr[y0, x0] + wb * arr[y1, x0] +
            wc * arr[y0, x1] + wd * arr[y1, x1])


def aug_elastic(img):
    """
    Elastic deformation — simulates soft-tissue variation and breathing motion.
    Uses the shape-safe _box_blur so this never fails on any image size.
    """
    alpha = random.uniform(20, 60)
    sigma = random.uniform(5, 8)

    arr  = np.array(img).astype(np.float32)
    h, w = arr.shape[:2]

    dx = np.random.randn(h, w).astype(np.float32) * alpha
    dy = np.random.randn(h, w).astype(np.float32) * alpha

    k  = int(sigma * 3) | 1    # odd kernel size
    dx = _box_blur(dx, k)      # guaranteed shape (h, w)
    dy = _box_blur(dy, k)      # guaranteed shape (h, w)

    x, y  = np.meshgrid(np.arange(w), np.arange(h))
    map_x = np.clip(x + dx, 0, w - 1).astype(np.float32)
    map_y = np.clip(y + dy, 0, h - 1).astype(np.float32)

    if arr.ndim == 3:
        out = np.stack([
            _bilinear_sample(arr[:, :, c], map_y, map_x)
            for c in range(arr.shape[2])], axis=2)
    else:
        out = _bilinear_sample(arr, map_y, map_x)

    return Image.fromarray(out.clip(0, 255).astype(np.uint8))


# Augmentation pipeline — (function, suffix) pairs
AUGMENTATION_PIPELINE = [
    (aug_flip,           'flip'),
    (aug_rotate,         'rot'),
    (aug_brightness,     'bright'),
    (aug_contrast,       'contrast'),
    (aug_gamma,          'gamma'),
    (aug_sharpness,      'sharp'),
    (aug_zoom_crop,      'zoom'),
    (aug_gaussian_noise, 'noise'),
    (aug_elastic,        'elastic'),
]


def apply_offline_augmentation(src_dir, dst_dir, n_augments=7):
    """
    For every image in src_dir: save original + n_augments augmented copies.
    Returns total images written.
    """
    os.makedirs(dst_dir, exist_ok=True)
    originals = sorted(glob.glob(f"{src_dir}/*.png"))
    if not originals:
        return 0

    total = 0
    for orig_path in originals:
        stem = os.path.splitext(os.path.basename(orig_path))[0]
        img  = Image.open(orig_path).convert('RGB')

        img.save(f"{dst_dir}/{stem}_orig.png")
        total += 1

        pipeline = random.sample(AUGMENTATION_PIPELINE,
                                 min(n_augments, len(AUGMENTATION_PIPELINE)))
        for aug_fn, suffix in pipeline:
            try:
                aug_img = aug_fn(img)
                aug_img.save(f"{dst_dir}/{stem}_{suffix}.png")
                total += 1
            except Exception as e:
                print(f"    Augmentation {suffix} failed on {stem}: {e}")

    return total


# =============================================================================
# 2. DATA PIPELINE
# =============================================================================
def setup_data(force_reprocess=False):
    drive.mount('/content/drive')

    DRIVE     = "/content/drive/My Drive/Thesis_Data"
    LOCAL     = "/content/xray_data"
    LOCAL_AUG = "/content/xray_data_aug"
    TEMP      = "/content/temp_xray_raw"

    sentinel = f"{LOCAL_AUG}/train/sick"
    if os.path.exists(sentinel) and not force_reprocess:
        print("Data already on disk (with augmentation). Skipping setup.")
        return

    if force_reprocess:
        for d in [LOCAL, LOCAL_AUG]:
            if os.path.exists(d):
                print(f"🗑  Removing {d}...")
                shutil.rmtree(d)

    print("\n  Starting data pipeline...")
    if not os.path.exists(TEMP):
        print("   Unzipping X-Rays (4.3 GB)... please wait...")
        shutil.unpack_archive(f"{DRIVE}/nih_xray.zip", TEMP)

    for split in ['train', 'val', 'test']:
        for label in ['sick', 'healthy']:
            os.makedirs(f"{LOCAL}/{split}/{label}", exist_ok=True)

    csvs = glob.glob(f"{TEMP}/**/*.csv", recursive=True)
    if not csvs:
        print(" No CSV found. Check your Drive folder.")
        return

    print(f"   -> CSV: {os.path.basename(csvs[0])}")
    df = pd.read_csv(csvs[0])

    sick_df    = df[df['Finding Labels'].str.contains('Cardiomegaly', na=False)]
    healthy_df = df[df['Finding Labels'] == 'No Finding']

    n = min(len(sick_df), len(healthy_df), 500)
    sick_df    = sick_df.sample(n=n, random_state=SEED)
    healthy_df = healthy_df.sample(n=n, random_state=SEED)
    print(f"   -> {n} sick + {n} healthy X-Rays selected.")

    all_pngs = glob.glob(f"{TEMP}/**/*.png", recursive=True)
    file_map  = {os.path.basename(f): f for f in all_pngs}

    def three_way_split(df_):
        tr, tmp = train_test_split(df_, test_size=0.30, random_state=SEED)
        vl, te  = train_test_split(tmp,  test_size=0.67, random_state=SEED)
        return tr, vl, te

    tr_s, vl_s, te_s = three_way_split(sick_df)
    tr_n, vl_n, te_n = three_way_split(healthy_df)

    def copy_images(df_, split, label):
        c = 0
        for img_name in df_['Image Index']:
            if img_name in file_map:
                shutil.copy(file_map[img_name],
                            f"{LOCAL}/{split}/{label}/{img_name}")
                c += 1
        return c

    totals = {}
    for df_, split, label in [
        (tr_s,'train','sick'),    (vl_s,'val','sick'),
        (te_s,'test','sick'),     (tr_n,'train','healthy'),
        (vl_n,'val','healthy'),   (te_n,'test','healthy'),
    ]:
        totals[f"{split}/{label}"] = copy_images(df_, split, label)

    print(" Base data split complete!")
    for sp in ['train', 'val', 'test']:
        s = totals.get(f"{sp}/sick", 0)
        h = totals.get(f"{sp}/healthy", 0)
        print(f"   {sp:5s}: {s} sick + {h} healthy = {s+h} images")

    # Val and test: copy unchanged (no augmentation on eval data)
    for split in ['val', 'test']:
        for label in ['sick', 'healthy']:
            os.makedirs(f"{LOCAL_AUG}/{split}/{label}", exist_ok=True)
            for f in glob.glob(f"{LOCAL}/{split}/{label}/*.png"):
                shutil.copy(f, f"{LOCAL_AUG}/{split}/{label}/")

    # Offline augmentation on TRAIN only
    print(f"\n Generating offline augmentations for training set...")
    print(f"   Each original image → 1 original + 7 augmented = 8× dataset")

    aug_totals = {}
    for label in ['sick', 'healthy']:
        src = f"{LOCAL}/train/{label}"
        dst = f"{LOCAL_AUG}/train/{label}"
        n_created = apply_offline_augmentation(src, dst, n_augments=7)
        aug_totals[label] = n_created
        orig = totals.get(f'train/{label}', 0)
        print(f"   train/{label}: {orig} originals → {n_created} total")

    total_train = sum(aug_totals.values())
    print(f"\n Augmentation complete!")
    print(f"   Final train : {aug_totals['sick']} sick "
          f"+ {aug_totals['healthy']} healthy = {total_train} images")
    print(f"   Val / Test  : unchanged (clean evaluation data)")


# =============================================================================
# 3. TRANSFORMS
# =============================================================================
TRAIN_TF = transforms.Compose([
    transforms.Resize((400, 400)),
    transforms.RandomCrop(380),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

EVAL_TF = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

TTA_TFS = [
    transforms.Compose([
        transforms.Resize((400, 400)),
        transforms.CenterCrop(380),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((400, 400)),
        transforms.RandomCrop(380),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((400, 400)),
        transforms.RandomCrop(380),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
]


# =============================================================================
# 4. DATASET
# =============================================================================
class XRayDataset(Dataset):
    def __init__(self, root_dir, split='train'):
        self.root  = os.path.join(root_dir, split)
        self.split = split
        self.tf    = TRAIN_TF if split == 'train' else EVAL_TF

        sick_imgs    = sorted(glob.glob(f"{self.root}/sick/*.png"))
        healthy_imgs = sorted(glob.glob(f"{self.root}/healthy/*.png"))

        self.paths  = sick_imgs + healthy_imgs
        self.labels = [1] * len(sick_imgs) + [0] * len(healthy_imgs)

        print(f"[{split.upper():5s}] "
              f"{len(sick_imgs)} sick + {len(healthy_imgs)} healthy "
              f"= {len(self.paths)} images")

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
            return self.tf(img), torch.tensor(self.labels[idx], dtype=torch.float)
        except Exception:
            return (torch.zeros(3, 380, 380),
                    torch.tensor(float(self.labels[idx])))

    def get_sampler(self):
        n0 = self.labels.count(0)
        n1 = self.labels.count(1)
        w  = [1.0 / n0 if l == 0 else 1.0 / n1 for l in self.labels]
        return WeightedRandomSampler(w, num_samples=len(w), replacement=True)


# =============================================================================
# 5. MIXUP + CUTMIX
# =============================================================================
def mixup_batch(x, y, alpha=0.3):
    if alpha <= 0: return x, y
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[idx], lam * y + (1-lam) * y[idx]


def cutmix_batch(x, y, alpha=1.0):
    if alpha <= 0: return x, y
    lam  = np.random.beta(alpha, alpha)
    idx  = torch.randperm(x.size(0), device=x.device)
    _, _, H, W = x.shape
    cut_h = int(H * math.sqrt(1 - lam))
    cut_w = int(W * math.sqrt(1 - lam))
    cx, cy = random.randint(0, W), random.randint(0, H)
    x1 = max(cx - cut_w // 2, 0);  x2 = min(cx + cut_w // 2, W)
    y1 = max(cy - cut_h // 2, 0);  y2 = min(cy + cut_h // 2, H)
    x_mix = x.clone()
    x_mix[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    actual_lam = 1 - (x2-x1)*(y2-y1) / (H*W)
    return x_mix, actual_lam * y + (1 - actual_lam) * y[idx]


def apply_batch_augmentation(x, y, phase):
    if phase == 1:
        return mixup_batch(x, y, alpha=0.3)
    elif phase == 2:
        return (mixup_batch(x, y, alpha=0.4) if random.random() < 0.5
                else cutmix_batch(x, y, alpha=1.0))
    return x, y   # phase 3: clean gradients for fine-tuning


# =============================================================================
# 6. MODEL — EfficientNet-B4
# =============================================================================
class CardiacXRayModel(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        base          = models.efficientnet_b4(weights='DEFAULT')
        self.backbone = base.features
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.head     = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(1792, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Linear(128, 1)
        )
        self._freeze_backbone()

    def _freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False

    def unfreeze_top_blocks(self, n_blocks=2):
        for block in list(self.backbone.children())[-n_blocks:]:
            for p in block.parameters(): p.requires_grad = True
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"   Phase 2 — top {n_blocks} blocks unfrozen, {n:,} params.")

    def unfreeze_full(self):
        for p in self.backbone.parameters(): p.requires_grad = True
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"   Phase 3 — full network, {n:,} params.")

    def forward(self, x):
        return self.head(torch.flatten(self.pool(self.backbone(x)), 1))


# =============================================================================
# 7. LOSS — Label Smoothing BCE
# =============================================================================
class LabelSmoothingBCE(nn.Module):
    def __init__(self, smoothing=0.1, pos_weight=None):
        super().__init__()
        self.smoothing  = smoothing
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        t = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(
            logits, t, pos_weight=self.pos_weight, reduction='mean')


# =============================================================================
# 8. SCHEDULER
# =============================================================================
def warmup_cosine_schedule(optimizer, warmup_epochs, total_epochs,
                           base_lr, min_lr=1e-7):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return max(epoch / max(warmup_epochs, 1), 1e-6 / base_lr)
        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        cosine   = 0.5 * (1 + math.cos(math.pi * progress))
        return min_lr / base_lr + cosine * (1 - min_lr / base_lr)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# =============================================================================
# 9. TRAINING HELPER
# =============================================================================
def run_epoch(model, loader, criterion, optimizer=None,
              use_aug=False, current_phase=1):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_true, all_prob = [], []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            if is_train and use_aug:
                imgs, labels = apply_batch_augmentation(imgs, labels, current_phase)

            out  = model(imgs).squeeze(1)
            loss = criterion(out, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            prob = torch.sigmoid(out).detach().cpu().numpy()
            all_true.extend(labels.cpu().numpy().tolist())
            all_prob.extend(prob.tolist() if prob.ndim > 0 else [float(prob)])

    # Round MixUp/CutMix continuous labels to 0/1 for sklearn
    all_true_int = np.round(np.array(all_true)).astype(int)
    all_pred_int = (np.array(all_prob) >= 0.5).astype(int)
    avg_loss     = total_loss / len(loader)

    f1 = f1_score(all_true_int, all_pred_int, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(all_true_int, all_prob)
    except Exception:
        auc = 0.0

    return avg_loss, f1, auc, all_true_int.tolist(), all_prob


# =============================================================================
# 10. CHECKPOINT HELPERS (PyTorch 2.6 safe)
# =============================================================================
def save_checkpoint(path, model, val_auc, threshold=None, test_auc=None):
    obj = {'state_dict': model.state_dict(), 'val_auc': float(val_auc)}
    if threshold is not None: obj['threshold'] = float(threshold)
    if test_auc  is not None: obj['test_auc']  = float(test_auc)
    torch.save(obj, path)


def load_checkpoint(path, model=None):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    if model is not None and 'state_dict' in ckpt:
        model.load_state_dict(ckpt['state_dict'])
    return ckpt


# =============================================================================
# 11. THRESHOLD — Youden's J
# =============================================================================
def find_youden_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j        = tpr - fpr
    best_idx = int(np.argmax(j))
    return float(thresholds[best_idx]), float(tpr[best_idx]), float(1 - fpr[best_idx])


# =============================================================================
# 12. TEST-TIME AUGMENTATION
# =============================================================================
def tta_predict(model, img_pil):
    model.eval()
    probs = []
    with torch.no_grad():
        for tf in TTA_TFS:
            t = tf(img_pil).unsqueeze(0).to(device)
            probs.append(torch.sigmoid(model(t)).item())
    return float(np.mean(probs))


# =============================================================================
# 13. MAIN
# =============================================================================
def main():
    setup_data(force_reprocess=False)

    train_ds = XRayDataset("/content/xray_data_aug", split='train')
    val_ds   = XRayDataset("/content/xray_data_aug", split='val')
    test_ds  = XRayDataset("/content/xray_data_aug", split='test')

    if len(train_ds) == 0:
        print(" No data. Run setup_data(force_reprocess=True).")
        return

    train_loader = DataLoader(train_ds, batch_size=16,
                              sampler=train_ds.get_sampler(),
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False,
                              num_workers=2, pin_memory=True)

    model = CardiacXRayModel(dropout=0.4).to(device)

    n_sick    = train_ds.labels.count(1)
    n_healthy = train_ds.labels.count(0)
    pos_w     = torch.tensor([n_healthy / max(n_sick, 1)],
                              dtype=torch.float).to(device)
    print(f"\npos_weight = {pos_w.item():.3f}  "
          f"(train: {n_sick} sick, {n_healthy} healthy)")

    criterion = LabelSmoothingBCE(smoothing=0.1, pos_weight=pos_w)

    P1, P2, P3 = 7, 10, 15
    TOTAL      = P1 + P2 + P3
    PATIENCE   = 8
    CKPT       = '/content/best_xray_model.pt'

    optimizer = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=1e-3, weight_decay=1e-4)
    scheduler = warmup_cosine_schedule(
        optimizer, warmup_epochs=2, total_epochs=P1,
        base_lr=1e-3, min_lr=1e-6)

    best_val_auc  = 0.0
    patience_ctr  = 0
    current_phase = 1
    history = {'train_loss':[], 'val_loss':[],
               'train_auc':[], 'val_auc':[], 'val_f1':[]}

    print(f"\n Phase 1: Head only ({P1} epochs, backbone frozen)...")

    for epoch in range(1, TOTAL + 1):

        if epoch == P1 + 1:
            current_phase = 2
            print(f"\n Phase 2: Top-2 backbone blocks ({P2} epochs)...")
            model.unfreeze_top_blocks(n_blocks=2)
            optimizer = optim.AdamW([
                {'params': [p for n, p in model.named_parameters()
                            if 'head' in n and p.requires_grad], 'lr': 1e-4},
                {'params': [p for n, p in model.named_parameters()
                            if 'head' not in n and p.requires_grad], 'lr': 5e-6},
            ], weight_decay=1e-4)
            scheduler = warmup_cosine_schedule(
                optimizer, warmup_epochs=1, total_epochs=P2,
                base_lr=1e-4, min_lr=1e-7)

        if epoch == P1 + P2 + 1:
            current_phase = 3
            print(f"\n Phase 3: Full fine-tuning ({P3} epochs)...")
            model.unfreeze_full()
            optimizer = optim.AdamW([
                {'params': [p for n, p in model.named_parameters()
                            if 'head' in n],     'lr': 5e-5},
                {'params': [p for n, p in model.named_parameters()
                            if 'head' not in n], 'lr': 5e-7},
            ], weight_decay=1e-4)
            scheduler = warmup_cosine_schedule(
                optimizer, warmup_epochs=2, total_epochs=P3,
                base_lr=5e-5, min_lr=1e-8)

        use_aug = (current_phase <= 2)

        tr_loss, tr_f1, tr_auc, *_ = run_epoch(
            model, train_loader, criterion, optimizer,
            use_aug=use_aug, current_phase=current_phase)
        va_loss, va_f1, va_auc, *_ = run_epoch(
            model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_auc'].append(tr_auc)
        history['val_auc'].append(va_auc)
        history['val_f1'].append(va_f1)

        print(f"   [P{current_phase}] Ep {epoch:2d}/{TOTAL} | "
              f"Train loss={tr_loss:.4f} AUC={tr_auc:.3f} | "
              f"Val   loss={va_loss:.4f} AUC={va_auc:.3f} F1={va_f1:.3f}")

        if va_auc > best_val_auc:
            best_val_auc = va_auc
            patience_ctr = 0
            save_checkpoint(CKPT, model, val_auc=best_val_auc)
            print(f"             Saved (val AUC={best_val_auc:.3f})")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n⏹  Early stopping at epoch {epoch}.")
                break

    # ── Threshold (Youden's J on val) ─────────────────────────────────────────
    print(f"\n  Selecting threshold via Youden's J on validation set...")
    load_checkpoint(CKPT, model)
    _, _, _, val_true, val_prob = run_epoch(model, val_loader, criterion)
    threshold, val_sens, val_spec = find_youden_threshold(val_true, val_prob)
    print(f"   Youden threshold = {threshold:.3f}")
    print(f"   Sensitivity={val_sens:.3f}  Specificity={val_spec:.3f}")
    save_checkpoint(CKPT, model, val_auc=best_val_auc, threshold=threshold)

    # ── Test evaluation ────────────────────────────────────────────────────────
    print(f"\n Final test evaluation (TTA + threshold={threshold:.3f})...")
    _, _, _, y_true, y_prob_std = run_epoch(model, test_loader, criterion)

    model.eval()
    y_prob_tta = []
    for path in test_ds.paths:
        y_prob_tta.append(tta_predict(model, Image.open(path).convert('RGB')))

    y_true_arr = np.array(y_true)
    y_pred_std = (np.array(y_prob_std) >= threshold).astype(int)
    y_pred_tta = (np.array(y_prob_tta) >= threshold).astype(int)

    print("\n" + "=" * 65)
    print("Standard evaluation (no TTA):")
    print(classification_report(y_true_arr, y_pred_std,
                                target_names=['Healthy', 'Sick']))
    print("=" * 65)
    print("With Test-Time Augmentation (TTA):")
    print(classification_report(y_true_arr, y_pred_tta,
                                target_names=['Healthy', 'Sick']))

    auc_std = roc_auc_score(y_true_arr, y_prob_std)
    auc_tta = roc_auc_score(y_true_arr, y_prob_tta)
    ap      = average_precision_score(y_true_arr, y_prob_tta)
    _, sens_t, spec_t = find_youden_threshold(y_true_arr, y_prob_tta)

    print(f"\n  ROC-AUC (standard) : {auc_std:.4f}")
    print(f"  ROC-AUC (TTA)      : {auc_tta:.4f}")
    print(f"  Avg Precision      : {ap:.4f}")
    print(f"  Sensitivity        : {sens_t:.4f}")
    print(f"  Specificity        : {spec_t:.4f}")

    final_path = '/content/cardiac_xray_model.pt'
    save_checkpoint(final_path, model,
                    val_auc=best_val_auc,
                    threshold=threshold,
                    test_auc=auc_tta)
    print(f"\n Final checkpoint → {final_path}")

    _plot_results(history, y_true_arr, y_prob_tta, y_pred_tta,
                  threshold, P1, P2)


# =============================================================================
# 14. RESULTS PLOT
# =============================================================================
def _plot_results(history, y_true, y_prob, y_pred, threshold, P1, P2):
    ep  = range(1, len(history['train_loss']) + 1)
    fig = plt.figure(figsize=(20, 10))
    gs  = fig.add_gridspec(2, 4, hspace=0.38, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(ep, history['train_loss'], label='Train')
    ax1.plot(ep, history['val_loss'],   label='Val')
    for xv, c, lbl in [(P1+0.5,'gray','P1→P2'),(P1+P2+0.5,'steelblue','P2→P3')]:
        if xv < max(ep)+1: ax1.axvline(xv, color=c, ls='--', lw=1, label=lbl)
    ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(fontsize=7)

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(ep, history['train_auc'], label='Train AUC')
    ax2.plot(ep, history['val_auc'],   label='Val AUC')
    ax2.set_ylim(0, 1); ax2.set_title('ROC-AUC')
    ax2.set_xlabel('Epoch'); ax2.legend(fontsize=7)

    ax3 = fig.add_subplot(gs[0, 2])
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val      = roc_auc_score(y_true, y_prob)
    ax3.plot(fpr, tpr, color='#2a9d8f', lw=2, label=f'AUC={auc_val:.3f}')
    ax3.plot([0,1],[0,1],'--',color='gray',lw=1)
    ax3.set_xlabel('1-Specificity'); ax3.set_ylabel('Sensitivity')
    ax3.set_title('ROC curve (test, TTA)'); ax3.legend(fontsize=8)

    ax4 = fig.add_subplot(gs[0, 3])
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    ax4.plot(rec, prec, color='#e76f51', lw=2, label=f'AP={ap:.3f}')
    ax4.set_xlabel('Recall'); ax4.set_ylabel('Precision')
    ax4.set_title('Precision-Recall (test, TTA)'); ax4.legend(fontsize=8)

    ax5 = fig.add_subplot(gs[1, 0])
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax5,
                xticklabels=['Healthy','Sick'], yticklabels=['Healthy','Sick'])
    ax5.set_title('Confusion matrix (TTA)')
    ax5.set_xlabel('Predicted'); ax5.set_ylabel('Actual')

    ax6 = fig.add_subplot(gs[1, 1])
    probs = np.array(y_prob)
    ax6.hist(probs[y_true==0], bins=20, alpha=0.6, label='Healthy', color='#2ecc71')
    ax6.hist(probs[y_true==1], bins=20, alpha=0.6, label='Sick',    color='#e74c3c')
    ax6.axvline(threshold, color='navy', ls='--', lw=1.5,
                label=f'Threshold={threshold:.2f}')
    ax6.set_xlabel('Predicted probability'); ax6.set_ylabel('Count')
    ax6.set_title('Confidence distribution'); ax6.legend(fontsize=7)

    ax7 = fig.add_subplot(gs[1, 2:])
    ax7.axis('off')
    tn, fp, fn, tp = cm.ravel()
    sens = tp/(tp+fn+1e-8); spec = tn/(tn+fp+1e-8)
    ppv  = tp/(tp+fp+1e-8); npv  = tn/(tn+fn+1e-8)
    f1v  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    rows = [
        ['Metric',          'Value',               'Clinical meaning'],
        ['ROC-AUC',         f'{auc_val:.4f}',      'Overall discriminability'],
        ['Avg Precision',   f'{ap:.4f}',            'Area under PR curve'],
        ['Sensitivity',     f'{sens:.4f}',          'True positive rate'],
        ['Specificity',     f'{spec:.4f}',          'True negative rate'],
        ['PPV',             f'{ppv:.4f}',           'Positive predictive value'],
        ['NPV',             f'{npv:.4f}',           'Negative predictive value'],
        ['Macro F1',        f'{f1v:.4f}',           'Harmonic mean prec/recall'],
        ['Threshold',       f'{threshold:.3f}',     'Youden-optimal cutoff'],
        ['TP/FP/FN/TN',     f'{tp}/{fp}/{fn}/{tn}', 'Raw counts'],
    ]
    tbl = ax7.table(cellText=rows[1:], colLabels=rows[0],
                    cellLoc='center', loc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.2, 1.5)
    ax7.set_title('Clinical metrics summary (test, TTA)', pad=14)

    plt.suptitle('Cardiac Disease Detection — EfficientNet-B4 | '
                 'NIH ChestX-ray14 | Offline Aug + TTA',
                 fontsize=13, fontweight='bold')
    plt.savefig('/content/results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(" Results plot → /content/results.png")


# =============================================================================
# 15. GRAD-CAM
# =============================================================================
class GradCAM:
    def __init__(self, model):
        self.model = model
        self._acts = self._grads = None
        self._h1   = model.backbone[-1].register_forward_hook(
            lambda _, __, o: setattr(self, '_acts', o))
        self._h2   = model.backbone[-1].register_full_backward_hook(
            lambda _, __, gi: setattr(self, '_grads', gi[0]))

    def remove(self): self._h1.remove(); self._h2.remove()

    def generate(self, img_tensor, class_idx=1):
        self.model.eval()
        x   = img_tensor.unsqueeze(0).to(device)
        out = self.model(x)
        self.model.zero_grad()
        (out[0,0] if class_idx == 1 else -out[0,0]).backward()
        weights = self._grads.mean(dim=(2,3), keepdim=True)
        cam     = F.relu((weights * self._acts).sum(dim=1)).squeeze()
        cam     = cam.detach().cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


def overlay_cam(img_tensor, cam, title, ax, alpha=0.45):
    mean  = np.array([0.485, 0.456, 0.406])
    std   = np.array([0.229, 0.224, 0.225])
    img   = np.clip(img_tensor.permute(1,2,0).cpu().numpy() * std + mean, 0, 1)
    h_up  = np.array(Image.fromarray(
        (cam*255).astype(np.uint8)).resize(
        (img.shape[1], img.shape[0]), Image.BILINEAR)) / 255.0
    ax.imshow((1-alpha)*img + alpha*mpl_cm.jet(h_up)[:,:,:3])
    ax.set_title(title, fontsize=9, pad=4); ax.axis('off')


def dominant_region(cam):
    h, w  = cam.shape
    q = {'upper-left': cam[:h//2,:w//2].mean(), 'upper-right': cam[:h//2,w//2:].mean(),
         'lower-left': cam[h//2:,:w//2].mean(), 'lower-right': cam[h//2:,w//2:].mean()}
    return max(q, key=q.get)


# =============================================================================
# 16. INFERENCE + GRAD-CAM
# =============================================================================
def predict_and_explain(xray_path,
                        ckpt_path='/content/cardiac_xray_model.pt',
                        use_tta=True):
    """
    Predict cardiac disease from a chest X-Ray with Grad-CAM explanation.

    Parameters
    ----------
    xray_path : str   Path to chest X-Ray (.png / .jpg)
    ckpt_path : str   Checkpoint from main()
    use_tta   : bool  Test-Time Augmentation (recommended)
    """
    model = CardiacXRayModel().to(device)
    ckpt  = load_checkpoint(ckpt_path, model)
    model.eval()
    threshold = float(ckpt.get('threshold', 0.5))

    img_pil    = Image.open(xray_path).convert('RGB')
    img_tensor = EVAL_TF(img_pil)

    if use_tta:
        confidence = tta_predict(model, img_pil)
        method     = "TTA (3-view average)"
    else:
        with torch.no_grad():
            confidence = torch.sigmoid(
                model(img_tensor.unsqueeze(0).to(device))).item()
        method = "single-view"

    prediction = 'Sick'    if confidence >= threshold else 'Healthy'
    pred_idx   = 1         if prediction == 'Sick'    else 0
    conf_pct   = confidence * 100

    gcam   = GradCAM(model)
    cam    = gcam.generate(img_tensor, class_idx=pred_idx)
    gcam.remove()
    region = dominant_region(cam)

    if prediction == 'Sick':
        reasoning = textwrap.dedent(f"""
        ┌──────────────────────────────────────────────────────┐
        │  PREDICTION  :  SICK  (Cardiomegaly likely)          │
        │  Confidence  :  {conf_pct:5.1f}%                               │
        │  Method      :  {method:<38}│
        │  Threshold   :  {threshold:.3f}  (Youden-optimal on val set)    │
        └──────────────────────────────────────────────────────┘

        Grad-CAM: highest attention → {region} chest region.

        Key regions for cardiomegaly:
          Central  : cardiac silhouette size
          Lateral  : cardiothoracic ratio borders
          Upper    : pulmonary vascular congestion

        Confidence {conf_pct:.1f}% > threshold {threshold:.3f}.
        Features consistent with enlarged cardiac silhouette.
        Associated with heart failure, DCM, hypertension.

            RESEARCH MODEL — not for clinical use.
            Confirm with echo and cardiologist review.
        """).strip()
    else:
        reasoning = textwrap.dedent(f"""
        ┌──────────────────────────────────────────────────────┐
        │  PREDICTION  :  HEALTHY  (No Cardiomegaly)           │
        │  Confidence  :  {conf_pct:5.1f}%  (disease probability)          │
        │  Method      :  {method:<38}│
        │  Threshold   :  {threshold:.3f}  (Youden-optimal on val set)    │
        └──────────────────────────────────────────────────────┘

        Grad-CAM: attention across {region} region.
        No focal enlargement pattern detected.

        Disease probability {conf_pct:.1f}% < threshold {threshold:.3f}.
        Cardiac silhouette and CTR appear within normal limits.

            RESEARCH MODEL — not for clinical use.
            Does not exclude early cardiac disease.
        """).strip()

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    colour    = '#c0392b' if prediction == 'Sick' else '#27ae60'
    fig.suptitle(
        f"Prediction: {prediction}  |  Confidence: {conf_pct:.1f}%  |  "
        f"Threshold: {threshold:.3f}  ({method})",
        fontsize=13, fontweight='bold', color=colour)

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    orig = np.clip(img_tensor.permute(1,2,0).numpy() * std + mean, 0, 1)
    axes[0].imshow(orig); axes[0].set_title('Input X-Ray'); axes[0].axis('off')

    overlay_cam(img_tensor, cam, f'Grad-CAM (focus: {region})', axes[1])

    axes[2].axis('off')
    axes[2].text(0.01, 0.99, reasoning, transform=axes[2].transAxes,
                 fontsize=7.5, verticalalignment='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='#fffde7',
                           alpha=0.92, edgecolor=colour, linewidth=1.5))

    plt.tight_layout()
    out_fig = '/content/explanation.png'
    plt.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n Explanation → {out_fig}")
    print("\n" + reasoning)

    return {'prediction': prediction, 'confidence': round(conf_pct, 2),
            'threshold': round(threshold, 3), 'method': method,
            'focus_region': region, 'reasoning': reasoning}


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    # First run creates ~1,500 augmented images — takes a few minutes.
    # Subsequent runs skip augmentation automatically.
    # To force a full rebuild from scratch:
    #   import shutil
    #   shutil.rmtree("/content/xray_data_aug", ignore_errors=True)
    #   shutil.rmtree("/content/xray_data",     ignore_errors=True)
    main()

    # After training, run inference on a new patient:
    # result = predict_and_explain(
    #     xray_path = "/content/drive/My Drive/Thesis_Data/new_patient.png",
    #     use_tta   = True,
    # )

In [ ]:
# Recreate variables lost after runtime restart
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model    = CardiacXRayModel().to(device)
ckpt     = torch.load('/content/cardiac_xray_model.pt', weights_only=False)
model.load_state_dict(ckpt['state_dict'])
model.eval()
test_ds  = XRayDataset("/content/xray_data_aug", split='test')
print(f" Model loaded  (val AUC={ckpt.get('val_auc', 0):.3f})")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from PIL import Image

y_prob_tta = []
for path in test_ds.paths:
    y_prob_tta.append(tta_predict(model, Image.open(path).convert('RGB')))

y_true_arr           = np.array(test_ds.labels)
fpr, tpr, thresholds = roc_curve(y_true_arr, y_prob_tta)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(thresholds, tpr,   label='Sensitivity (TPR)', color='#e74c3c', lw=2)
plt.plot(thresholds, 1-fpr, label='Specificity (TNR)', color='#2ecc71', lw=2)
plt.axhline(0.90, color='navy', ls='--', lw=1, label='90% target')
plt.xlabel('Threshold'); plt.title('Sensitivity vs Specificity')
plt.legend(fontsize=8); plt.gca().invert_xaxis()

plt.subplot(1, 2, 2)
fn = [int(round((1-s) * 29)) for s in tpr]
plt.plot(thresholds, fn, color='#c0392b', lw=2)
plt.axhline(3, color='navy', ls='--', lw=1, label='<=3 missed (90%)')
plt.xlabel('Threshold'); plt.ylabel('Missed sick patients (FN)')
plt.title('False Negatives vs Threshold')
plt.legend(fontsize=8); plt.gca().invert_xaxis()

plt.tight_layout()
plt.savefig('/content/threshold_analysis.png', dpi=150)
plt.show()

In [ ]:
def find_sensitivity_threshold(y_true, y_prob, min_sensitivity=0.90):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    valid = [(t, 1-f, s) for f, s, t in zip(fpr, tpr, thresholds)
             if s >= min_sensitivity]
    if not valid:
        print(f"Cannot reach {min_sensitivity:.0%} — using lowest threshold.")
        best = (float(thresholds[-1]), float(1-fpr[-1]), float(tpr[-1]))
    else:
        best = max(valid, key=lambda x: x[1])
    return float(best[0]), float(best[2]), float(best[1])

from sklearn.metrics import classification_report, confusion_matrix

print("Target | Threshold | Sensitivity | Specificity | Missed/29")
print("-" * 62)
for target in [0.80, 0.85, 0.90, 0.93, 0.95]:
    t, sens, spec = find_sensitivity_threshold(y_true_arr, y_prob_tta, target)
    missed = int(round((1 - sens) * 29))
    print(f"  {target:.0%}  |   {t:.3f}   |    {sens:.3f}    "
          f"|    {spec:.3f}    |  {missed}/29")

# Apply 90% sensitivity threshold
threshold, sensitivity, specificity = find_sensitivity_threshold(
    y_true_arr, y_prob_tta, min_sensitivity=0.90)

print(f"\nSelected threshold : {threshold:.3f}")
print(f"Sensitivity        : {sensitivity:.3f}  "
      f"({int(round(sensitivity*29))}/29 sick caught)")
print(f"Specificity        : {specificity:.3f}  "
      f"({int(round(specificity*29))}/29 healthy cleared)")

y_pred_new = (np.array(y_prob_tta) >= threshold).astype(int)
print("\n" + "="*55)
print(classification_report(y_true_arr, y_pred_new,
                            target_names=['Healthy', 'Sick']))
tn, fp, fn, tp = confusion_matrix(y_true_arr, y_pred_new).ravel()
print(f"Sick correctly caught  (TP) : {tp}/29")
print(f"Sick patients MISSED   (FN) : {fn}/29")
print(f"Healthy correctly clear(TN) : {tn}/29")
print(f"Healthy flagged as sick(FP) : {fp}/29")

# Save new threshold to checkpoint
ckpt['threshold']          = float(threshold)
ckpt['threshold_strategy'] = 'min_sensitivity_0.90'
torch.save(ckpt, '/content/cardiac_xray_model.pt')
print(f"\nNew threshold {threshold:.3f} saved to checkpoint.")

In [ ]:
# =============================================================================
# FINAL YEAR THESIS: CARDIAC DISEASE DETECTION — INDUSTRIAL STANDARD
# =============================================================================
#
# Architecture  : CheXNet DenseNet121 pretrained on 100k+ chest X-rays
# Dataset       : NIH ChestX-ray14 + 8x offline augmentation
# Preprocessing : CLAHE (clinical standard) + grayscale (1-channel)
# Loss          : Focal Loss (gamma=2, clinical standard)
# Training      : 5-fold cross validation, 3-phase per fold
# Inference     : 5-model ensemble x 5 TTA views = 25 votes per prediction
# Threshold     : 90% sensitivity target (Youden's J as fallback)
# Metrics       : AUC, AP, Sensitivity, Specificity, PPV, NPV, LR+, LR-, NNS
# =============================================================================

import os, glob, shutil, textwrap, math, random
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score,
                             precision_recall_curve, average_precision_score)
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
from google.colab import drive

# Install torchxrayvision if needed
try:
    import torchxrayvision as xrv
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "torchxrayvision"])
    import torchxrayvision as xrv


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Device: {device}")
if device.type == 'cpu':
    print("Go to Runtime > Change runtime type > T4 GPU!")

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
# 1. CLAHE PREPROCESSING
# =============================================================================
def apply_clahe(img_pil, clip_limit=2.0, grid_size=8):
    """
    Contrast Limited Adaptive Histogram Equalisation.
    Clinical standard preprocessing for chest X-ray AI.
    Enhances cardiac border and pulmonary vasculature without amplifying noise.
    Returns RGB PIL image (grayscale conversion happens in transforms).
    """
    arr  = np.array(img_pil.convert('L')).astype(np.float32)
    h, w = arr.shape
    th   = h // grid_size
    tw   = w // grid_size
    out  = np.zeros_like(arr)

    for i in range(grid_size):
        for j in range(grid_size):
            y0 = i * th; y1 = min(y0 + th, h)
            x0 = j * tw; x1 = min(x0 + tw, w)
            tile = arr[y0:y1, x0:x1]
            hist, bins = np.histogram(tile.flatten(), bins=256, range=(0, 255))
            clip_val   = clip_limit * tile.size / 256
            excess     = np.sum(np.maximum(hist - clip_val, 0))
            hist       = np.minimum(hist, clip_val)
            hist      += excess / 256
            cdf        = hist.cumsum()
            cdf        = (cdf - cdf.min()) / (cdf.max() - cdf.min() + 1e-8) * 255
            out[y0:y1, x0:x1] = np.interp(
                tile.flatten(), bins[:-1], cdf).reshape(tile.shape)

    return Image.fromarray(out.clip(0, 255).astype(np.uint8)).convert('RGB')


# =============================================================================
# 2. AUGMENTATION FUNCTIONS
# =============================================================================
def aug_rotate(img):
    return img.rotate(random.uniform(-15, 15), resample=Image.BILINEAR)

def aug_brightness(img):
    return ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))

def aug_contrast(img):
    return ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.4))

def aug_sharpness(img):
    return ImageEnhance.Sharpness(img).enhance(random.uniform(0.5, 2.0))

def aug_gamma(img):
    g   = random.uniform(0.7, 1.4)
    arr = np.array(img).astype(np.float32) / 255.0
    return Image.fromarray(
        (np.power(arr, g) * 255).clip(0, 255).astype(np.uint8))

def aug_flip(img):
    return img.transpose(Image.FLIP_LEFT_RIGHT)

def aug_zoom_crop(img):
    zoom  = random.uniform(1.05, 1.20)
    w, h  = img.size
    nw, nh = int(w * zoom), int(h * zoom)
    img_z = img.resize((nw, nh), Image.BILINEAR)
    return img_z.crop(((nw-w)//2, (nh-h)//2,
                        (nw-w)//2+w, (nh-h)//2+h))

def aug_gaussian_noise(img):
    std = random.uniform(3, 12)
    arr = np.array(img).astype(np.float32)
    return Image.fromarray(
        np.clip(arr + np.random.normal(0, std, arr.shape),
                0, 255).astype(np.uint8))

def _box_blur(arr, k):
    """
    Shape-safe separable box blur.
    pad_l=k//2, pad_r=k-pad_l ensures total padding=k so output shape=input shape.
    """
    h_in, w_in = arr.shape
    pad_l = k // 2;  pad_r = k - pad_l
    a = np.pad(arr, ((0, 0), (pad_l, pad_r)), mode='reflect')
    a = (np.cumsum(a, axis=1)[:, k:] - np.cumsum(a, axis=1)[:, :-k]) / k
    a = np.pad(a, ((pad_l, pad_r), (0, 0)), mode='reflect')
    a = (np.cumsum(a, axis=0)[k:, :] - np.cumsum(a, axis=0)[:-k, :]) / k
    return a[:h_in, :w_in]

def _bilinear_sample(arr, my, mx):
    h, w   = arr.shape
    x0     = np.floor(mx).astype(int).clip(0, w-2)
    y0     = np.floor(my).astype(int).clip(0, h-2)
    x1, y1 = x0+1, y0+1
    return ((x1-mx)*(y1-my)*arr[y0,x0] + (x1-mx)*(my-y0)*arr[y1,x0] +
            (mx-x0)*(y1-my)*arr[y0,x1] + (mx-x0)*(my-y0)*arr[y1,x1])

def aug_elastic(img):
    alpha = random.uniform(20, 60); sigma = random.uniform(5, 8)
    arr   = np.array(img).astype(np.float32)
    h, w  = arr.shape[:2]
    k     = int(sigma * 3) | 1
    dx    = _box_blur(np.random.randn(h, w).astype(np.float32) * alpha, k)
    dy    = _box_blur(np.random.randn(h, w).astype(np.float32) * alpha, k)
    gx, gy = np.meshgrid(np.arange(w), np.arange(h))
    mx    = np.clip(gx + dx, 0, w-1).astype(np.float32)
    my    = np.clip(gy + dy, 0, h-1).astype(np.float32)
    if arr.ndim == 3:
        out = np.stack([_bilinear_sample(arr[:,:,c], my, mx)
                        for c in range(arr.shape[2])], axis=2)
    else:
        out = _bilinear_sample(arr, my, mx)
    return Image.fromarray(out.clip(0, 255).astype(np.uint8))

AUGMENTATION_PIPELINE = [
    (aug_flip,           'flip'),
    (aug_rotate,         'rot'),
    (aug_brightness,     'bright'),
    (aug_contrast,       'contrast'),
    (aug_gamma,          'gamma'),
    (aug_sharpness,      'sharp'),
    (aug_zoom_crop,      'zoom'),
    (aug_gaussian_noise, 'noise'),
    (aug_elastic,        'elastic'),
]

def apply_offline_augmentation(src_dir, dst_dir, n_augments=7):
    os.makedirs(dst_dir, exist_ok=True)
    originals = sorted(glob.glob(f"{src_dir}/*.png"))
    if not originals: return 0
    total = 0
    for orig_path in originals:
        stem = os.path.splitext(os.path.basename(orig_path))[0]
        img  = Image.open(orig_path).convert('RGB')
        img.save(f"{dst_dir}/{stem}_orig.png"); total += 1
        for aug_fn, suffix in random.sample(AUGMENTATION_PIPELINE,
                                            min(n_augments,
                                                len(AUGMENTATION_PIPELINE))):
            try:
                aug_fn(img).save(f"{dst_dir}/{stem}_{suffix}.png"); total += 1
            except Exception as e:
                print(f"   ⚠️  {suffix} failed on {stem}: {e}")
    return total


# =============================================================================
# 3. DATA PIPELINE
# =============================================================================
def setup_data(force_reprocess=False):
    drive.mount('/content/drive')
    DRIVE     = "/content/drive/My Drive/Thesis_Data"
    LOCAL     = "/content/xray_data"
    LOCAL_AUG = "/content/xray_data_aug"
    TEMP      = "/content/temp_xray_raw"

    if os.path.exists(f"{LOCAL_AUG}/train/sick") and not force_reprocess:
        print("Data already on disk. Skipping setup.")
        return

    if force_reprocess:
        for d in [LOCAL, LOCAL_AUG]:
            if os.path.exists(d): shutil.rmtree(d)

    print("\n  Starting data pipeline...")
    if not os.path.exists(TEMP):
        print("   Unzipping (4.3 GB)...")
        shutil.unpack_archive(f"{DRIVE}/nih_xray.zip", TEMP)

    for split in ['train', 'val', 'test']:
        for label in ['sick', 'healthy']:
            os.makedirs(f"{LOCAL}/{split}/{label}", exist_ok=True)

    csvs = glob.glob(f"{TEMP}/**/*.csv", recursive=True)
    if not csvs: print("No CSV found."); return

    df         = pd.read_csv(csvs[0])
    sick_df    = df[df['Finding Labels'].str.contains('Cardiomegaly', na=False)]
    healthy_df = df[df['Finding Labels'] == 'No Finding']
    n          = min(len(sick_df), len(healthy_df), 500)
    sick_df    = sick_df.sample(n=n, random_state=SEED)
    healthy_df = healthy_df.sample(n=n, random_state=SEED)
    print(f"   -> {n} sick + {n} healthy selected.")

    all_pngs = glob.glob(f"{TEMP}/**/*.png", recursive=True)
    file_map  = {os.path.basename(f): f for f in all_pngs}

    def three_split(df_):
        tr, tmp = train_test_split(df_, test_size=0.30, random_state=SEED)
        vl, te  = train_test_split(tmp,  test_size=0.67, random_state=SEED)
        return tr, vl, te

    tr_s, vl_s, te_s = three_split(sick_df)
    tr_n, vl_n, te_n = three_split(healthy_df)

    totals = {}
    for df_, sp, lb in [(tr_s,'train','sick'), (vl_s,'val','sick'),
                        (te_s,'test','sick'),  (tr_n,'train','healthy'),
                        (vl_n,'val','healthy'),(te_n,'test','healthy')]:
        c = 0
        for img in df_['Image Index']:
            if img in file_map:
                shutil.copy(file_map[img], f"{LOCAL}/{sp}/{lb}/{img}"); c += 1
        totals[f"{sp}/{lb}"] = c

    print("Base split done!")
    for sp in ['train', 'val', 'test']:
        s = totals.get(f"{sp}/sick", 0); h = totals.get(f"{sp}/healthy", 0)
        print(f"   {sp:5s}: {s} sick + {h} healthy = {s+h}")

    for sp in ['val', 'test']:
        for lb in ['sick', 'healthy']:
            os.makedirs(f"{LOCAL_AUG}/{sp}/{lb}", exist_ok=True)
            for f in glob.glob(f"{LOCAL}/{sp}/{lb}/*.png"):
                shutil.copy(f, f"{LOCAL_AUG}/{sp}/{lb}/")

    print("\n Generating offline augmentations (train only)...")
    for lb in ['sick', 'healthy']:
        n_out = apply_offline_augmentation(
            f"{LOCAL}/train/{lb}", f"{LOCAL_AUG}/train/{lb}", n_augments=7)
        print(f"   train/{lb}: {totals.get(f'train/{lb}',0)} → {n_out} total")

    print(" Augmentation complete.")


# =============================================================================
# 4. TRANSFORMS
# =============================================================================
# torchxrayvision DenseNet121 requires:
#   - Single grayscale channel  (1, H, W)
#   - Pixel range approximately [-1, 1] after normalisation
#   - NOT standard ImageNet normalisation

TRAIN_TF = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.Grayscale(num_output_channels=1),   # → (1, H, W)
    transforms.ToTensor(),                          # → [0, 1]
    transforms.Normalize(mean=[0.5], std=[0.5])     # → [-1, 1]
])

EVAL_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

TTA_TFS = [
    transforms.Compose([
        transforms.Resize((256, 256)), transforms.CenterCrop(224),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(), transforms.Normalize([0.5], [0.5])]),
    transforms.Compose([
        transforms.Resize((256, 256)), transforms.RandomCrop(224),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(), transforms.Normalize([0.5], [0.5])]),
    transforms.Compose([
        transforms.Resize((256, 256)), transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(), transforms.Normalize([0.5], [0.5])]),
    transforms.Compose([
        transforms.Resize((240, 240)), transforms.CenterCrop(224),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(), transforms.Normalize([0.5], [0.5])]),
    transforms.Compose([
        transforms.Resize((256, 256)), transforms.RandomCrop(224),
        transforms.RandomRotation(5),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(), transforms.Normalize([0.5], [0.5])]),
]


# =============================================================================
# 5. DATASET
# =============================================================================
class XRayDataset(Dataset):
    def __init__(self, root_dir, split='train', use_clahe=True, indices=None):
        self.root      = os.path.join(root_dir, split)
        self.split     = split
        self.tf        = TRAIN_TF if split == 'train' else EVAL_TF
        self.use_clahe = use_clahe

        sick_imgs    = sorted(glob.glob(f"{self.root}/sick/*.png"))
        healthy_imgs = sorted(glob.glob(f"{self.root}/healthy/*.png"))
        all_paths    = sick_imgs + healthy_imgs
        all_labels   = [1]*len(sick_imgs) + [0]*len(healthy_imgs)

        if indices is not None:
            self.paths  = [all_paths[i]  for i in indices]
            self.labels = [all_labels[i] for i in indices]
        else:
            self.paths  = all_paths
            self.labels = all_labels

        print(f"[{split.upper():5s}] {self.labels.count(1)} sick + "
              f"{self.labels.count(0)} healthy = {len(self.paths)} images"
              f"{'  (CLAHE on)' if use_clahe else ''}")

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
            if self.use_clahe:
                img = apply_clahe(img)
            return self.tf(img), torch.tensor(self.labels[idx], dtype=torch.float)
        except Exception:
            return (torch.zeros(1, 224, 224),      # 1-channel placeholder
                    torch.tensor(float(self.labels[idx])))

    def get_sampler(self):
        n0 = self.labels.count(0); n1 = self.labels.count(1)
        w  = [1.0/n0 if l == 0 else 1.0/n1 for l in self.labels]
        return WeightedRandomSampler(w, num_samples=len(w), replacement=True)


# =============================================================================
# 6. MODEL — CheXNet DenseNet121 (domain-pretrained on NIH chest X-rays)
# =============================================================================
class CheXNetModel(nn.Module):
    """
    DenseNet121 pretrained on 112,120 chest X-rays (NIH ChestX-ray14).
    CheXNet architecture

    Input : (batch, 1, 224, 224)  — single-channel grayscale
    Output: (batch, 1)            — logit for sick/healthy
    """
    def __init__(self, dropout=0.4):
        super().__init__()
        # Load weights pretrained on the exact same NIH dataset we're using
        base          = xrv.models.DenseNet(weights="densenet121-res224-nih")
        self.features = base.features   # DenseNet feature extractor
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self._freeze_backbone()

        # Custom head — outputs binary classification logit
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(1024, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def _freeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = False

    def unfreeze_top_blocks(self, n_blocks=2):
        blocks = list(self.features.children())
        for block in blocks[-n_blocks:]:
            for p in block.parameters(): p.requires_grad = True
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"   Phase 2 — top {n_blocks} blocks unfrozen, {n:,} params.")

    def unfreeze_full(self):
        for p in self.features.parameters(): p.requires_grad = True
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"   Phase 3 — full network, {n:,} params.")

    def forward(self, x):
        # x: (batch, 1, 224, 224) — torchxrayvision DenseNet expects 1-channel
        feat = F.relu(self.features(x), inplace=True)
        feat = torch.flatten(self.pool(feat), 1)   # (batch, 1024)
        return self.head(feat)                      # (batch, 1)


# =============================================================================
# 7. FOCAL LOSS
# =============================================================================
class FocalLoss(nn.Module):
    """
    Focal Loss — Lin et al. 2017 (RetinaNet).
    FL = -alpha * (1-p)^gamma * log(p)
    gamma=2 standard for medical imaging.
    Down-weights easy examples so training focuses on hard borderline cases.
    """
    def __init__(self, gamma=2.0, alpha=0.75):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce  = F.binary_cross_entropy_with_logits(
            logits, targets, reduction='none')
        prob = torch.sigmoid(logits)
        p_t  = prob * targets + (1 - prob) * (1 - targets)
        a_t  = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()


# =============================================================================
# 8. MIXUP + CUTMIX
# =============================================================================
def mixup_batch(x, y, alpha=0.3):
    if alpha <= 0: return x, y
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x+(1-lam)*x[idx], lam*y+(1-lam)*y[idx]

def cutmix_batch(x, y, alpha=1.0):
    if alpha <= 0: return x, y
    lam  = np.random.beta(alpha, alpha)
    idx  = torch.randperm(x.size(0), device=x.device)
    _,_,H,W = x.shape
    ch,cw = int(H*math.sqrt(1-lam)), int(W*math.sqrt(1-lam))
    cx,cy = random.randint(0,W), random.randint(0,H)
    x1,x2 = max(cx-cw//2,0), min(cx+cw//2,W)
    y1,y2 = max(cy-ch//2,0), min(cy+ch//2,H)
    xm = x.clone(); xm[:,:,y1:y2,x1:x2] = x[idx,:,y1:y2,x1:x2]
    lam_a = 1-(x2-x1)*(y2-y1)/(H*W)
    return xm, lam_a*y+(1-lam_a)*y[idx]

def batch_aug(x, y, phase):
    if phase == 1: return mixup_batch(x, y, 0.3)
    if phase == 2: return (mixup_batch(x,y,0.4) if random.random()<0.5
                           else cutmix_batch(x,y,1.0))
    return x, y


# =============================================================================
# 9. SCHEDULER
# =============================================================================
def warmup_cosine(optimizer, warmup_ep, total_ep, base_lr, min_lr=1e-7):
    def fn(ep):
        if ep < warmup_ep:
            return max(ep/max(warmup_ep,1), 1e-6/base_lr)
        p = (ep-warmup_ep)/max(total_ep-warmup_ep,1)
        return min_lr/base_lr + 0.5*(1+math.cos(math.pi*p))*(1-min_lr/base_lr)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)


# =============================================================================
# 10. TRAINING HELPER
# =============================================================================
def run_epoch(model, loader, criterion, optimizer=None,
              use_aug=False, phase=1):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    all_true, all_prob = [], []

    with (torch.enable_grad() if is_train else torch.no_grad()):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if is_train and use_aug:
                imgs, labels = batch_aug(imgs, labels, phase)
            out  = model(imgs).squeeze(1)
            loss = criterion(out, labels)
            if is_train:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()
            prob = torch.sigmoid(out).detach().cpu().numpy()
            all_true.extend(labels.cpu().numpy().tolist())
            all_prob.extend(prob.tolist() if prob.ndim > 0 else [float(prob)])

    # Round MixUp/CutMix continuous labels to int for sklearn
    true_int = np.round(np.array(all_true)).astype(int)
    pred_int = (np.array(all_prob) >= 0.5).astype(int)
    f1 = f1_score(true_int, pred_int, average='macro', zero_division=0)
    try:    auc = roc_auc_score(true_int, all_prob)
    except: auc = 0.0
    return total_loss/len(loader), f1, auc, true_int.tolist(), all_prob


# =============================================================================
# 11. CHECKPOINT HELPERS (PyTorch 2.6 safe — pure Python scalars only)
# =============================================================================
def save_ckpt(path, model, val_auc, threshold=None, fold=None):
    obj = {'state_dict': model.state_dict(), 'val_auc': float(val_auc)}
    if threshold is not None: obj['threshold'] = float(threshold)
    if fold      is not None: obj['fold']      = int(fold)
    torch.save(obj, path)

def load_ckpt(path, model=None):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    if model is not None and 'state_dict' in ckpt:
        model.load_state_dict(ckpt['state_dict'])
    return ckpt


# =============================================================================
# 12. THRESHOLD SELECTION
# =============================================================================
def sensitivity_threshold(y_true, y_prob, min_sens=0.90):
    """
    Highest threshold achieving min_sens sensitivity.
    Maximises specificity among all thresholds meeting the sensitivity target.
    Clinical gold standard for screening tool threshold selection.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    valid = [(t, 1-f, s) for f,s,t in zip(fpr,tpr,thresholds) if s >= min_sens]
    if not valid:
        best = (float(thresholds[-1]), float(1-fpr[-1]), float(tpr[-1]))
    else:
        best = max(valid, key=lambda x: x[1])
    return float(best[0]), float(best[2]), float(best[1])

def find_youden(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j = tpr - fpr
    best = int(np.argmax(j))
    return float(thresholds[best]), float(tpr[best]), float(1-fpr[best])


# =============================================================================
# 13. TTA PREDICTION
# =============================================================================
def tta_predict(model, img_pil, use_clahe=True):
    """Average predictions over 5 augmented views. Reduces variance."""
    if use_clahe:
        img_pil = apply_clahe(img_pil)
    model.eval(); probs = []
    with torch.no_grad():
        for tf in TTA_TFS:
            t = tf(img_pil).unsqueeze(0).to(device)  # (1, 1, 224, 224)
            probs.append(torch.sigmoid(model(t)).item())
    return float(np.mean(probs))


# =============================================================================
# 14. TRAIN ONE FOLD
# =============================================================================
def train_fold(fold_idx, train_ds, val_ds,
               n_ep_p1=7, n_ep_p2=8, n_ep_p3=10, patience=6):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold_idx+1}/5")
    print(f"{'='*60}")

    train_loader = DataLoader(train_ds, batch_size=16,
                              sampler=train_ds.get_sampler(),
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False,
                              num_workers=2, pin_memory=True)

    model     = CheXNetModel(dropout=0.4).to(device)
    n_sick    = train_ds.labels.count(1)
    n_healthy = train_ds.labels.count(0)
    alpha     = n_healthy / max(n_sick + n_healthy, 1)
    criterion = FocalLoss(gamma=2.0, alpha=alpha)
    print(f"  Focal loss alpha={alpha:.3f}  "
          f"(sick={n_sick}, healthy={n_healthy})")

    TOTAL    = n_ep_p1 + n_ep_p2 + n_ep_p3
    CKPT     = f'/content/fold_{fold_idx}.pt'
    best_auc = 0.0; patience_ctr = 0; phase = 1
    history  = {'tr_loss':[], 'va_loss':[], 'tr_auc':[], 'va_auc':[]}

    optimizer = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=1e-3, weight_decay=1e-4)
    scheduler = warmup_cosine(optimizer, 2, n_ep_p1, 1e-3, 1e-6)

    print(f"\n  Phase 1: head only ({n_ep_p1} ep)...")

    for epoch in range(1, TOTAL+1):

        if epoch == n_ep_p1+1:
            phase = 2
            print(f"\n  Phase 2: top-2 blocks ({n_ep_p2} ep)...")
            model.unfreeze_top_blocks(2)
            optimizer = optim.AdamW([
                {'params':[p for n,p in model.named_parameters()
                           if 'head' in n and p.requires_grad], 'lr':1e-4},
                {'params':[p for n,p in model.named_parameters()
                           if 'head' not in n and p.requires_grad], 'lr':5e-6},
            ], weight_decay=1e-4)
            scheduler = warmup_cosine(optimizer, 1, n_ep_p2, 1e-4, 1e-7)

        if epoch == n_ep_p1+n_ep_p2+1:
            phase = 3
            print(f"\n  Phase 3: full network ({n_ep_p3} ep)...")
            model.unfreeze_full()
            optimizer = optim.AdamW([
                {'params':[p for n,p in model.named_parameters()
                           if 'head' in n],     'lr':5e-5},
                {'params':[p for n,p in model.named_parameters()
                           if 'head' not in n], 'lr':5e-7},
            ], weight_decay=1e-4)
            scheduler = warmup_cosine(optimizer, 2, n_ep_p3, 5e-5, 1e-8)

        tr_loss,tr_f1,tr_auc,*_ = run_epoch(
            model, train_loader, criterion, optimizer,
            use_aug=(phase<=2), phase=phase)
        va_loss,va_f1,va_auc,*_ = run_epoch(model, val_loader, criterion)
        scheduler.step()

        history['tr_loss'].append(tr_loss); history['va_loss'].append(va_loss)
        history['tr_auc'].append(tr_auc);   history['va_auc'].append(va_auc)

        print(f"  [P{phase}] Ep {epoch:2d}/{TOTAL} | "
              f"Tr {tr_loss:.4f}/{tr_auc:.3f} | "
              f"Va {va_loss:.4f}/{va_auc:.3f}/{va_f1:.3f}", end='')

        if va_auc > best_auc:
            best_auc = va_auc; patience_ctr = 0
            save_ckpt(CKPT, model, best_auc, fold=fold_idx)
            print("  💾")
        else:
            patience_ctr += 1; print()
            if patience_ctr >= patience:
                print(f"  ⏹  Early stop ep {epoch}.")
                break

    return CKPT, history


# =============================================================================
# 15. ENSEMBLE PREDICTION
# =============================================================================
def ensemble_predict_tta(ckpt_paths, img_pil, use_clahe=True):
    """
    Average TTA predictions from all 5 fold models.
    5 models × 5 TTA views = 25 votes per prediction.
    Uncorrelated errors between folds cancel out → lower variance.
    """
    all_probs = []
    for ckpt_path in ckpt_paths:
        m = CheXNetModel().to(device)
        load_ckpt(ckpt_path, m)
        all_probs.append(tta_predict(m, img_pil, use_clahe))
    return float(np.mean(all_probs))


# =============================================================================
# 16. MAIN — 5-FOLD CROSS VALIDATION + ENSEMBLE
# =============================================================================
def main():
    setup_data(force_reprocess=False)

    full_train = XRayDataset("/content/xray_data_aug", split='train',
                             use_clahe=True)
    test_ds    = XRayDataset("/content/xray_data_aug", split='test',
                             use_clahe=True)

    if len(full_train) == 0:
        print(" No data. Run setup_data(force_reprocess=True)."); return

    skf       = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    indices   = np.arange(len(full_train))
    label_arr = np.array(full_train.labels)

    fold_ckpts     = []
    fold_histories = []

    print(f"\n{'='*60}")
    print(f"  5-FOLD CROSS VALIDATION  ({len(full_train)} train images)")
    print(f"{'='*60}")

    for fold_idx, (tr_idx, va_idx) in enumerate(
            skf.split(indices, label_arr)):

        train_fold_ds = XRayDataset("/content/xray_data_aug", split='train',
                                    use_clahe=True, indices=tr_idx.tolist())
        val_fold_ds   = XRayDataset("/content/xray_data_aug", split='train',
                                    use_clahe=True, indices=va_idx.tolist())
        val_fold_ds.tf = EVAL_TF   # no augmentation on validation fold

        ckpt_path, history = train_fold(
            fold_idx, train_fold_ds, val_fold_ds,
            n_ep_p1=7, n_ep_p2=8, n_ep_p3=10, patience=6)

        fold_ckpts.append(ckpt_path)
        fold_histories.append(history)

    # Ensemble evaluation on held-out test set
    print(f"\n{'='*60}")
    print(f"  ENSEMBLE EVALUATION ON TEST SET")
    print(f"{'='*60}")
    print("Running ensemble TTA (5 models × 5 views = 25 votes)...")

    y_prob_ensemble = []
    for i, path in enumerate(test_ds.paths):
        img  = Image.open(path).convert('RGB')
        prob = ensemble_predict_tta(fold_ckpts, img, use_clahe=True)
        y_prob_ensemble.append(prob)
        if (i+1) % 10 == 0:
            print(f"  {i+1}/{len(test_ds.paths)} done...")

    y_true_arr = np.array(test_ds.labels)
    y_prob_arr = np.array(y_prob_ensemble)

    # Threshold: 90% sensitivity target
    threshold, sensitivity, specificity = sensitivity_threshold(
        y_true_arr, y_prob_arr, min_sens=0.90)
    y_pred = (y_prob_arr >= threshold).astype(int)

    n_sick_test = int(y_true_arr.sum())
    print(f"\nThreshold (90% sensitivity) : {threshold:.3f}")
    print(f"Sensitivity : {sensitivity:.3f}  "
          f"({int(round(sensitivity*n_sick_test))}/{n_sick_test} sick caught)")
    print(f"Specificity : {specificity:.3f}")

    print("\n" + "="*65)
    print(classification_report(y_true_arr, y_pred,
                                target_names=['Healthy', 'Sick']))

    auc = roc_auc_score(y_true_arr, y_prob_arr)
    ap  = average_precision_score(y_true_arr, y_prob_arr)
    cm  = confusion_matrix(y_true_arr, y_pred)
    tn,fp,fn,tp = cm.ravel()
    ppv   = tp/(tp+fp+1e-8);  npv = tn/(tn+fn+1e-8)
    sens  = tp/(tp+fn+1e-8);  spec = tn/(tn+fp+1e-8)
    lr_p  = sens/(1-spec+1e-8)
    lr_n  = (1-sens)/(spec+1e-8)
    nns   = 1/(sensitivity+1e-8)

    print(f"  ROC-AUC          : {auc:.4f}")
    print(f"  Avg Precision    : {ap:.4f}")
    print(f"  PPV              : {ppv:.4f}")
    print(f"  NPV              : {npv:.4f}")
    print(f"  LR+  (>10=good)  : {lr_p:.2f}")
    print(f"  LR-  (<0.1=good) : {lr_n:.3f}")
    print(f"  NNS              : {nns:.1f}")
    print(f"  TP/FP/FN/TN      : {tp}/{fp}/{fn}/{tn}")

    torch.save({'fold_ckpts': fold_ckpts,
                'threshold':  float(threshold),
                'test_auc':   float(auc),
                'test_ap':    float(ap)},
               '/content/ensemble_model.pt')
    print("\n Ensemble checkpoint → /content/ensemble_model.pt")

    _plot_results(fold_histories, y_true_arr, y_prob_arr,
                  y_pred, threshold, auc, ap)


# =============================================================================
# 17. RESULTS PLOT — 7-panel clinical dashboard
# =============================================================================
def _plot_results(histories, y_true, y_prob, y_pred, threshold, auc, ap):
    fig = plt.figure(figsize=(20, 12))
    gs  = fig.add_gridspec(2, 4, hspace=0.4, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])
    for i, h in enumerate(histories):
        ax1.plot(range(1, len(h['va_auc'])+1), h['va_auc'],
                 alpha=0.7, lw=1.5, label=f'Fold {i+1}')
    ax1.set_ylim(0, 1); ax1.set_title('Val AUC — all folds')
    ax1.set_xlabel('Epoch'); ax1.legend(fontsize=7)

    ax2 = fig.add_subplot(gs[0, 1])
    max_ep = max(len(h['tr_loss']) for h in histories)
    def pad(arr): return np.pad(arr, (0, max_ep-len(arr)),
                                constant_values=np.nan)
    tr_m = np.array([pad(h['tr_loss']) for h in histories])
    va_m = np.array([pad(h['va_loss']) for h in histories])
    ep = range(1, max_ep+1)
    ax2.plot(ep, np.nanmean(tr_m, 0), label='Train (avg)')
    ax2.plot(ep, np.nanmean(va_m, 0), label='Val (avg)')
    ax2.set_title('Avg loss across folds')
    ax2.set_xlabel('Epoch'); ax2.legend(fontsize=7)

    ax3 = fig.add_subplot(gs[0, 2])
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ax3.plot(fpr, tpr, color='#2a9d8f', lw=2, label=f'Ensemble AUC={auc:.3f}')
    ax3.plot([0,1],[0,1],'--', color='gray', lw=1)
    ax3.set_xlabel('1-Specificity'); ax3.set_ylabel('Sensitivity')
    ax3.set_title('ROC curve (ensemble, TTA)'); ax3.legend(fontsize=8)

    ax4 = fig.add_subplot(gs[0, 3])
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ax4.plot(rec, prec, color='#e76f51', lw=2, label=f'AP={ap:.3f}')
    ax4.set_xlabel('Recall'); ax4.set_ylabel('Precision')
    ax4.set_title('Precision-Recall (ensemble, TTA)'); ax4.legend(fontsize=8)

    ax5 = fig.add_subplot(gs[1, 0])
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax5,
                xticklabels=['Healthy','Sick'], yticklabels=['Healthy','Sick'])
    ax5.set_title('Confusion matrix (ensemble)')
    ax5.set_xlabel('Predicted'); ax5.set_ylabel('Actual')

    ax6 = fig.add_subplot(gs[1, 1])
    probs = np.array(y_prob)
    ax6.hist(probs[y_true==0], bins=20, alpha=0.6, label='Healthy',
             color='#2ecc71')
    ax6.hist(probs[y_true==1], bins=20, alpha=0.6, label='Sick',
             color='#e74c3c')
    ax6.axvline(threshold, color='navy', ls='--', lw=1.5,
                label=f'Threshold={threshold:.2f}')
    ax6.set_title('Confidence distribution'); ax6.legend(fontsize=7)

    ax7 = fig.add_subplot(gs[1, 2:])
    ax7.axis('off')
    tn,fp,fn,tp = cm.ravel()
    sens = tp/(tp+fn+1e-8); spec = tn/(tn+fp+1e-8)
    ppv  = tp/(tp+fp+1e-8); npv  = tn/(tn+fn+1e-8)
    f1v  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    lr_p = sens/(1-spec+1e-8); lr_n = (1-sens)/(spec+1e-8)
    rows = [
        ['Metric',        'Value',              'Clinical standard'],
        ['ROC-AUC',       f'{auc:.4f}',         '>0.85 = good'],
        ['Avg Precision', f'{ap:.4f}',           '>0.80 = good'],
        ['Sensitivity',   f'{sens:.4f}',         '>0.90 for screening'],
        ['Specificity',   f'{spec:.4f}',         '>0.70 preferred'],
        ['PPV',           f'{ppv:.4f}',          'Prob(sick|positive)'],
        ['NPV',           f'{npv:.4f}',          'Prob(healthy|negative)'],
        ['LR+',           f'{lr_p:.2f}',         '>10 = strong evidence'],
        ['LR-',           f'{lr_n:.3f}',         '<0.1 = strong rule-out'],
        ['Macro F1',      f'{f1v:.4f}',          'Balance prec/recall'],
        ['Threshold',     f'{threshold:.3f}',    '90% sensitivity target'],
        ['TP/FP/FN/TN',   f'{tp}/{fp}/{fn}/{tn}','Raw counts'],
        ['Ensemble',      '5 models x 5 TTA',   '25 votes per prediction'],
    ]
    tbl = ax7.table(cellText=rows[1:], colLabels=rows[0],
                    cellLoc='center', loc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1.2, 1.4)
    ax7.set_title('Clinical metrics — ensemble on test set', pad=12)

    plt.suptitle(
        'Cardiac Screening — CheXNet DenseNet121 | 5-Fold Ensemble | '
        'CLAHE + Grayscale | TTA',
        fontsize=13, fontweight='bold')
    plt.savefig('/content/results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(" Results plot → /content/results.png")


# =============================================================================
# 18. GRAD-CAM
# =============================================================================
class GradCAM:
    def __init__(self, model):
        self.model  = model
        self._acts  = self._grads = None
        target      = model.features.denseblock4
        self._h1    = target.register_forward_hook(
            lambda _,__,o: setattr(self,'_acts',o))
        self._h2    = target.register_full_backward_hook(
            lambda _,__,gi: setattr(self,'_grads',gi[0]))

    def remove(self): self._h1.remove(); self._h2.remove()

    def generate(self, img_tensor, class_idx=1):
        self.model.eval()
        x   = img_tensor.unsqueeze(0).to(device)
        out = self.model(x)
        self.model.zero_grad()
        (out[0,0] if class_idx==1 else -out[0,0]).backward()
        w   = self._grads.mean(dim=(2,3), keepdim=True)
        cam = F.relu((w*self._acts).sum(dim=1)).squeeze()
        cam = cam.detach().cpu().numpy()
        cam = (cam-cam.min())/(cam.max()-cam.min()+1e-8)
        return cam


def overlay_cam(img_t, cam, title, ax, alpha=0.45):
    """
    img_t: (1, H, W) single-channel tensor normalised to [-1, 1]
    Converts to displayable (H, W, 3) before blending with heatmap.
    """
    img_gray = img_t.squeeze(0).cpu().numpy()           # (H, W)
    img_disp = (img_gray * 0.5 + 0.5).clip(0, 1)       # undo [-1,1] → [0,1]
    img_rgb  = np.stack([img_disp, img_disp, img_disp], axis=2)  # (H,W,3)
    h_up     = np.array(Image.fromarray(
        (cam*255).astype(np.uint8)).resize(
        (img_rgb.shape[1], img_rgb.shape[0]), Image.BILINEAR)) / 255.0
    ax.imshow((1-alpha)*img_rgb + alpha*mpl_cm.jet(h_up)[:,:,:3])
    ax.set_title(title, fontsize=9, pad=4); ax.axis('off')

def dominant_region(cam):
    h, w = cam.shape
    q = {'upper-left': cam[:h//2,:w//2].mean(),
         'upper-right':cam[:h//2,w//2:].mean(),
         'lower-left': cam[h//2:,:w//2].mean(),
         'lower-right':cam[h//2:,w//2:].mean()}
    return max(q, key=q.get)


# =============================================================================
# 19. INFERENCE + GRAD-CAM EXPLANATION
# =============================================================================
def predict_and_explain(xray_path,
                        ensemble_ckpt='/content/ensemble_model.pt'):
    """
    Predict cardiac disease using the full 5-model ensemble with TTA.
    Shows Grad-CAM from fold-0 model and clinical reasoning text.

    Parameters
    ----------
    xray_path     : str  Path to chest X-Ray (.png / .jpg)
    ensemble_ckpt : str  Path to ensemble checkpoint from main()
    """
    meta       = load_ckpt(ensemble_ckpt)
    fold_ckpts = meta['fold_ckpts']
    threshold  = float(meta.get('threshold', 0.5))

    img_pil    = Image.open(xray_path).convert('RGB')
    confidence = ensemble_predict_tta(fold_ckpts, img_pil, use_clahe=True)
    prediction = 'Sick'    if confidence >= threshold else 'Healthy'
    pred_idx   = 1         if prediction == 'Sick'    else 0
    conf_pct   = confidence * 100

    # Grad-CAM from fold-0 model
    gcam_model = CheXNetModel().to(device)
    load_ckpt(fold_ckpts[0], gcam_model)
    img_clahe  = apply_clahe(img_pil)
    img_tensor = EVAL_TF(img_clahe)     # (1, 224, 224)
    gcam       = GradCAM(gcam_model)
    cam        = gcam.generate(img_tensor, class_idx=pred_idx)
    gcam.remove()
    region     = dominant_region(cam)

    colour    = '#c0392b' if prediction == 'Sick' else '#27ae60'
    reasoning = textwrap.dedent(f"""
    ┌──────────────────────────────────────────────────────────┐
    │  PREDICTION  : {prediction:<10}                              │
    │  Confidence  : {conf_pct:5.1f}%  (ensemble of 5 models)        │
    │  Method      : 5-fold ensemble x 5 TTA = 25 votes        │
    │  Threshold   : {threshold:.3f}  (90% sensitivity target)        │
    └──────────────────────────────────────────────────────────┘

    Preprocessing : CLAHE → Grayscale (clinical standard pipeline)
    Grad-CAM      : highest attention → {region} chest region.

    Key regions for cardiomegaly:
      Central  : cardiac silhouette / CTR
      Lateral  : cardiothoracic ratio borders
      Upper    : pulmonary vascular congestion

    {'Confidence '+ f'{conf_pct:.1f}%' + ' exceeds threshold '
        + f'{threshold:.3f}.' if prediction == 'Sick'
      else 'Confidence ' + f'{conf_pct:.1f}%'
        + ' below threshold ' + f'{threshold:.3f}.'}

     RESEARCH MODEL — not for clinical use.
        Confirm with echocardiography and cardiologist review.
    """).strip()

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(
        f"Prediction: {prediction}  |  Confidence: {conf_pct:.1f}%  |  "
        f"Threshold: {threshold:.3f}  (5-model ensemble, 25 votes)",
        fontsize=13, fontweight='bold', color=colour)

    # Panel 1: original X-ray (display as grayscale)
    img_gray = img_tensor.squeeze(0).cpu().numpy()
    img_disp = np.stack([(img_gray*0.5+0.5).clip(0,1)]*3, axis=2)
    axes[0].imshow(img_disp)
    axes[0].set_title('Input (CLAHE + grayscale)'); axes[0].axis('off')

    # Panel 2: Grad-CAM overlay
    overlay_cam(img_tensor, cam, f'Grad-CAM (focus: {region})', axes[1])

    # Panel 3: reasoning
    axes[2].axis('off')
    axes[2].text(0.01, 0.99, reasoning, transform=axes[2].transAxes,
                 fontsize=7.5, va='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='#fffde7',
                           alpha=0.92, edgecolor=colour, lw=1.5))

    plt.tight_layout()
    out_fig = '/content/explanation.png'
    plt.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n Explanation → {out_fig}")
    print("\n" + reasoning)

    return {'prediction':   prediction,
            'confidence':   round(conf_pct, 2),
            'threshold':    round(threshold, 3),
            'focus_region': region,
            'n_models':     len(fold_ckpts)}


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    # NOTE: 5-fold training takes ~3-4 hours on T4 GPU.
    # Each fold saves to /content/fold_N.pt
    # Final ensemble checkpoint → /content/ensemble_model.pt
    #
    # To force full data rebuild:
    #   import shutil
    #   shutil.rmtree("/content/xray_data",     ignore_errors=True)
    #   shutil.rmtree("/content/xray_data_aug", ignore_errors=True)
    main()

    # Inference after training:
    # result = predict_and_explain(
    #     xray_path = "/content/drive/My Drive/Thesis_Data/new_patient.png"
    # )

In [ ]:
from google.colab import files
for f in ['ensemble_model.pt','fold_0.pt','fold_1.pt',
          'fold_2.pt','fold_3.pt','fold_4.pt']:
    files.download(f'/content/{f}')

In [ ]:
from google.colab import files
files.download('/content/fold_3.pt')